Cell M7-1(markdown)|第 1 節:notebook 定位

### 07 | 法遵與公平性檢視(M7)

**目的**:<br/>
M0 至 M6 依序回答了「誰會流失、為什麼流失、該對誰做什麼挽留動作、怎麼驗證挽留動作有效」;本模組回答最後一個問題:「在考量法規與公平性的前提下,能不能直接使用 M5 設計的挽留名單與 M4 建立的風險模型」。<br/>

本模組所說的「M5 的挽留名單」與「現行名單規則」,指的是 notebook 05 定案的規則式名單(notebook 05 第 8 節名單總表、決策日誌 [M5-1] 至 [M5-3]),全模組都以這一套規則為檢視對象,規則內容如下:<br/>
- 風險軸:四個行為旗標(低活躍 = 近 12 個月交易筆數 ≤ 61、零動用 = 額度動用率 = 0、淺關係 = 往來產品數 ≤ 2、高聯繫 = 近 12 個月客服聯繫次數 ≥ 4),每位客戶命中幾個旗標就是幾分(0 到 4 分,稱為命中數);命中數 ≥ 2 的客戶進入挽留名單,共 2,476 人。這套「四旗標命中數」規則在 notebook 02 稱為 C3-b 規則。<br/>
- 價值軸:近 12 個月交易金額依人數切成三等份(低、中、高價值層,邊界 2,567 元與 4,476 元)。<br/>
- 四個策略區與動作:命中 ≥ 2 且高價值層 = 搶救區(426 人,由客戶經理人工聯繫);命中 ≥ 2 且低或中價值層 = 低成本區(2,050 人,自動化推播),低成本區中消費變化比 < 0.50 的 308 人是收到加碼版回饋(而非低成本區通用的回饋版本);命中 ≤ 1 的客戶依價值層分為深化區與監測區,不收到挽留動作。<br/>
這套規則只使用客戶的行為欄位與交易金額,沒有使用性別、年齡等人口屬性。<br/>

為達成前述目的,M7 具體檢查以下兩件事:<br/>
(1) **名單層的檢查(待驗證事項 #29)**:上述名單規則雖然沒有使用性別與年齡,但執行結果仍可能讓某一個性別或某一個年齡組的客戶「系統性地」得到較多或較少的挽留資源。「系統性」指的是「差異不是個別客戶的隨機差異,而是規則對整個族群一致地產生的差異」。若各族群得到的挽留資源有差異,本模組要區分差異的來源是下列哪一種:(a) 這個族群的客戶確實流失率較高,所以規則依行為把他們圈進來得較多,這是真實風險造成的差異,可以接受;不過客群的組成會隨時間改變,今天合理的差距有可能在日後擴大,所以須列為上線後的監測項,定期追蹤該族群的選取率、漏報率與實際流失率是否仍與其他族群相近;(b) 規則本身對這個族群不利,例如規則系統性地漏掉這個族群中真正會流失的客戶,這種情形在公平性檢視上稱為「差別性影響」(disparate impact,規則表面上沒有用「受保護特徵」,但執行結果對某一族群明顯不利),此時須修改規則。區分方法見第 1 節 1.5 與第 4 節。<br/>
(2) **模型層的檢查(待驗證事項 #27)**:M4 的模型 B 是一個邏輯斯迴歸模型,此模型把性別當成輸入變數之一(模型的輸入變數在機器學習中稱為「特徵」feature;「性別作為特徵」就是「性別是模型計算流失機率時使用的欄位之一」)。模型 B 顯示,在其他變數都相同的條件下,女性客戶的流失勝算是男性客戶的 2.12 倍(此為 M4 主切分 random_state 42 的估計值,五個切分皆顯著)。<br/>
這件事與本模組的關聯是:M5 把模型 B 列為名單圈選的升級選項(決策日誌 [M5-3],預設用規則圈選名單,符合切換條件時則改用模型 B 的機率分數排序圈名單);若日後真的改用模型 B 圈名單,每一位女性客戶的分數都會單純因為性別而被往上調,名單就會「依性別給予不同待遇」,這在公平性檢視上稱為「差別待遇」(disparate treatment,**直接依受保護特徵給予不同對待**)。<br/>
因此,本模組要回答:**性別能不能留在模型 B 裡當特徵;若不能,就要把性別從模型的輸入變數中移除,重新訓練一版不含性別特徵的模型 B,再比較「含性別版模型 B」與「不含性別版模型 B」在同樣名單人數下,名單中真的流失的客戶比例(精確率)相差多少,這個差距就是「移除性別特徵要付出的精確率代價」**。<br/>
以第 5 節的實測值為例:703 人名單中,含性別版的精確率(五個切分平均)是 55.74%、不含性別版是 55.54%,代價就是 0.20 個百分點。<br/>
本案 M5 預設的規則式名單本來就沒有用性別,所以「移除性別」只影響模型 B 這個升級選項,不影響現行名單。<br/>

細節說明:<br/>
勝算(odds)是「流失人數 ÷ 未流失人數」,勝算比(odds ratio)是兩組勝算相除;2.12 倍的實務意義是:兩位行為、信用額度、收入等條件完全相同的客戶,只差一位是女性、一位是男性,模型會給女性明顯較高的流失機率。<br/>
「五個切分皆顯著」指的是「M4 Cell M4-17 的穩定性檢查」:把 10,127 人以五個不同的隨機種子(random_state 0、1、7、42、99)各切一次訓練集與測試集、各訓練一次模型,五次算出的性別勝算比都在 2.12 至 2.97 之間、p 值都小於 0.001;這代表「性別的效果不是某一次抽樣碰巧出現的,而是穩定存在的(切五次的做法與理由見 notebook 04 第 7 節)」。


**支持的決策**:<br/>
(1) 問題定義書決策 1(挽留資源應優先投放在哪些客群)的約束條件:在法遵(法令遵循,compliance,指「業務作法符合法規與主管機關要求」)的前提下,上述名單規則能否照原樣上線使用。<br/>
(2) 問題定義書決策 2(高風險客戶的可辨識訊號是什麼)的一個子題:「性別能否作為辨識高風險客戶的訊號」。<br/>

本模組不重做 M5 的名單設計,只回答「在法遵前提下,現行名單規則能否照原樣使用」以及「若不能,應往哪個方向修改規則」。<br/>


**風險定位(本模組所有取捨的前提)**:<br/>
本案名單的用途是「行銷資源分配」,決定哪些客戶先收到挽留動作(推播、消費回饋、客戶經理聯繫);本案名單用途不是「授信決策」,不會決定銀行核給客戶多少信用額度、對客戶收取多少循環信用利率或分期利率。這兩種用途在法規上的風險等級不同。依據是:美國的平等信用機會法(Equal Credit Opportunity Act, ECOA)及其施行細則 Regulation B 明文禁止授信決策考量性別、婚姻狀態、年齡等特徵,違反者有法律責任;台灣的金融消費者保護法第 7 條要求金融服務業對金融消費者應本於公平合理原則,金管會 2024 年 6 月 20 日發布的「金融業運用人工智慧(AI)指引」則要求金融機構(銀行、保險、證券等受金管會監理的機構)對該機構使用的每一個 AI 系統自行評估風險等級,風險等級越高,機構內部的審查程序與主管機關的監理要求就越嚴格,高風險系統可能需要由機構以外、具 AI 專業的第三方審查或測試,這種「依風險高低決定審查嚴格程度」的做法稱為風險基礎方法(risk-based approach)。<br/>
**本模組依此判斷「本案名單屬於較低風險用途」:授信決策直接影響客戶能否取得金融服務與取得的條件,是監理程度最嚴格的用途;行銷資源分配影響的是「哪些客戶先得到優惠」,客戶沒有得到優惠並不會失去原有的服務,所以客戶權益受損的風險與主管機關裁罰的風險都比授信決策低。**<br/>
基於「本案名單屬於較低風險的行銷資源分配用途」這個判斷,本模組對名單與模型做公平性檢視時,檢視的項目定為兩項:<br/>
(a) 族群組成檢查(各性別、各年齡組進入名單的比例,以及名單在各族群內的準確程度,即第 4 節的檢查 A);<br/>
(b) 將「從模型 B 移除性別特徵的代價」量化(第 5 節的檢查 B)。<br/>
本模組不執行高風險用途才需要的兩種程序:一是「獨立第三方審查」,即把模型與名單交給機構以外、具 AI 專業的單位(例如會計師事務所、顧問公司或學術機構)重新檢驗;二是「多種公平性定義的全面比較」,即「同時以人口比例平等、機會平等、校準等多種公平性定義逐一計算並比較(本模組只選定一種主要判準,理由見第 1 節 1.6)」。不執行的理由是:這兩種程序的成本高,是監理機關對授信等高風險用途的要求,對行銷資源分配用途而言,投入與風險不相稱。<br/>

提醒:「本案名單屬於較低風險用途」是本模組的判斷,不是法規明文的分類,所以在此處寫明判斷依據,讓日後審核本模組的法遵或風險控管人員能檢查這個判斷是否成立。若這份挽留名單日後改作其他用途,例如「依名單決定是否調高客戶的信用額度(授信),或依名單對不同客戶收取不同的利率或年費(定價)」,用途的風險等級就會提高,屆時須重新評估應做哪些檢視項目,例如是否補做「獨立第三方審查與多種公平性定義的比較」,不能沿用本模組「只做兩項檢查」的結論。<br/>


**檢視的特徵與名單層級**:<br/>
(1) **檢視的特徵**:有三個,分兩種處理方式:<br/>
**性別(gender)**與**年齡(customer_age)**:性別在名單層與模型層都檢視,年齡只在名單層檢視。<br/>
理由:兩者是本資料集中最典型的受保護特徵(受保護特徵的定義見第 1 節 1.1;美國 ECOA 將性別與年齡都列為授信不得考量的特徵;台灣金管會的「金融服務業公平待客原則」中的「友善服務」原則要求對高齡等客群提供友善服務,可推論年齡是台灣金融監理持續關注的面向)。此外,M4 模型 B 的人口屬性中,性別的效果穩定且明顯(勝算比 2.12,五個切分皆顯著,notebook 04 第 6 節判讀 (3)、第 8 節結論 2),是模型層最需要處理的特徵。年齡則相反:年齡在模型 B 中的效果不穩定(五個切分中只有兩次 p < 0.05,notebook 04 第 7 節穩定性表),代表模型沒有穩定地依賴年齡,預期移除年齡的精確率代價接近 0(係數接近 0 的特徵對每一位客戶的預測流失機率幾乎沒有貢獻,移除後名單排序幾乎不變;對照性別的係數穩定且明顯,移除的代價也只有 703 人名單 0.2 個百分點;未實測),算出這個代價也不會改變任何決定,所以年齡只在名單層(檢查 A)檢視,在模型層(檢查 B)不另外訓練不含年齡的版本。<br/>
**婚姻狀態(marital_status)**:只在模型層(檢查 B)檢視,不在名單層(檢查 A)檢視。<br/>
理由:模型 B 的人口屬性中,除了性別以外,婚姻狀態的效果也穩定(單身客戶的流失勝算是已婚客戶的 1.93 倍,notebook 04 第 8 節結論 2;實務意義同性別:其他條件相同時,模型會給單身客戶較高的流失機率),而且婚姻狀態在美國 ECOA 下也是授信不得考量的受保護特徵。因此檢查 B 在「含性別版的模型 B」與「不含性別版的模型 B」之外,多訓練一版「同時不含性別與婚姻狀態」的模型 B,成本只有「同一段程式多跑一個版本」,就能算出若婚姻狀態也被視為受保護特徵時要付出的代價。<br/>
本模組不計算「名單在不同婚姻狀態客戶之間的組成是否均衡」,只列為第 9 節的方法限制。理由有三個:第一,台灣的個人資料保護法、金融消費者保護法與金管會的公平待客原則,都沒有把婚姻狀態列為行銷活動不得考量的特徵,美國 ECOA 對婚姻狀態的限制是針對授信決策,而本案名單屬於行銷用途,所以婚姻狀態在名單層的法遵優先順序低於性別與年齡;第二,本資料集的婚姻狀態欄位含有「Unknown(未知)」一類(notebook 04 Cell M4-13 特徵表),若把未知當成一個族群計算選取率,算出的數字沒有實務意義,須先決定未知值的處理方式才能做名單層的檢查;第三,本模組的檢視範圍在開工時定為性別與年齡(決策日誌 [M7-0]),婚姻狀態只在模型層順帶處理(因為模型層多檢查一個特徵的工作量最低:只要在檢查 B 的版本清單多加一個「不含婚姻狀態」的版本,同一段程式就會自動重訓練與重算;但名單層多檢查一個特徵,則要對三個名單層級各多做一組選取率、精確率、漏報率的表與判讀,且須先解決未知值的歸類問題)。日後若需要此計算,使用檢查 A 同一套程式,將分組欄位換成婚姻狀態、並先決定未知值的歸類即可。<br/>

(2) **名單層級**:挽留動作有三種資源強度不同的分配,公平性問題在每一種分配上都可能不同(例如挽留名單整體平衡,但其中成本最高的人工聯繫可能集中在某一族群),所以檢查 A 對三種分配各檢查一次:<br/>
(a) 挽留名單:命中 ≥ 2 的 2,476 人,是「會收到任何挽留動作的全部客戶」;<br/>
(b) 人工聯繫名單:搶救區 426 人,收到的是客戶經理逐一聯繫,每人成本最高,若某族群被系統性排除在外,損失也最大;<br/>
(c) 加碼回饋名單:低成本區內消費變化比 < 0.50 的 308 人,收到的是比基本推播更優惠的加碼版回饋,是基本動作之外的額外優惠。<br/>

除了這三層,本模組另外計算「聯繫 ≥ 5 次人工回訪」名單(230 人)的性別與年齡組成,作為對照:<br/>
這份名單是 M5 的服務補救動作(notebook 05 第 4 節 4.2 動作表、決策日誌 [M5-1] 附帶決策 (1):近 12 個月客服聯繫 ≥ 5 次的客戶,不分價值層一律由客戶經理人工回訪、了解聯繫原因),它的圈選條件只有一個行為欄位(客服聯繫次數),與性別、年齡完全無關。
計算的目的是回答:「一條完全只看行為的規則,圈出來的客戶在性別、年齡上會不會也與全體客戶不同?」若會,就表示行為欄位本身與性別、年齡有關聯,因此「規則沒有使用性別」不能等同於「名單與性別無關」,這就是第 1 節 1.3 所說的代理變數問題(**行為欄位間接帶入了性別資訊**)。這個對照名單不是挽留資源的分配對象,所以本模組不對它做「是否構成差別性影響」的判定,只用它來確認代理變數問題是否存在;結果見第 4 節判定 (7)。<br/>


**本模組的兩個產出**:<br/>
1. 本 notebook(`notebooks/07_compliance_fairness.ipynb`):(1) 檢查 A 與檢查 B 這兩項以數字計算的檢查(以下稱兩項量化檢查)的程式、輸出與判讀;(2) 不採用公平性套件(fairness toolkit,專門計算公平性指標的程式庫)的取捨說明。<br/>
2. `docs/07_compliance_fairness.md`:一頁式的法遵與公平性檢視報告(review memo)。讀者是法遵、風險控管與業務單位,此文件供其在幾分鐘內讀完、直接看到結論與依據,內容包含:兩項檢查的結論、本模組對名單規則的決定(M5 決定的是名單規則的內容;本模組決定的是「這套規則在法遵與公平性上能否照原樣使用」、「若日後圈選名單時決定升級為模型 B,能否在模型 B 內保留性別作為輸入特徵」、「挽留方案上線後要監測哪些族群指標」)、每一項決定對應的法規或指引(法規指個人資料保護法與金融消費者保護法;指引指金管會發布的「金融業運用人工智慧(AI)指引」與「金融服務業公平待客原則」,兩者是主管機關對業者的行政指導,沒有罰則,但金融檢查時會檢視業者是否遵循;以下合稱「法規與指引」),以及本模組的限制。M8 README 文件中的 Compliance & Fairness Considerations 一節由這份文件精簡而來。<br/>
另撰寫決策日誌 [M7-0] 至 [M7-2],待驗證事項清單更新到 v8。<br/>


**本模組與前六個模組的差別**:<br/>
(1) 相異之處:前六個模組討論「圈選的名單是否準確、設計的動作是否有效」;本模組討論「這份名單在法規與公平性上能否使用」。本模組的作法是:把法規與指引的要求翻譯成可以用資料計算的檢核項目(例如:把「避免歧視」翻譯成「在後來真的流失的客戶裡,女性沒有被名單圈到的比例,與男性沒有被名單圈到的比例,兩者是否相近」;若兩個比例相近,代表名單沒有特別漏掉某一個性別的流失客戶),計算之後說明每一項檢核結果對應哪一條法規或指引的要求,最後判斷是否必須為了符合法規而修改名單規則。<br/>
(2) 相同之處:本模組用來檢查的工具都已在之前的模組使用過:M2 的切片(依某個欄位分組後,比較各組的比例)、M3 的兩比例比較與 95% 信賴區間(判斷兩組比例的差距是否大到不能用抽樣波動解釋)、M4 的模型 B 重新擬合與「固定名單規模,比較精確率」(把模型 B 用同一組訓練資料重新訓練一次,再讓模型與規則各取同樣人數的名單,例如都取 178 人與 703 人,比較兩份名單中真的流失的客戶比例(精確率),M4 第 7 節的評估方式)。本模組新的部分是「討論的問題」與「對應的法規」,不是新的統計或建模技術。<br/>

**各節 Roadmap**:<br/>

| 節 | 目標 | 做法 | 實務效果 | 承接或定案的待驗證事項 |
|---|---|---|---|---|
| 1 | 說明本模組用到的概念 | 受保護特徵、差別待遇與差別性影響、代理變數、選取率與不利影響比率、族群切片評估、兩種公平性定義的衝突、較少歧視替代方案、風險基礎方法,每一個概念都寫明定義、為什麼需要、本案的設定 | 讀者具備基礎觀念,能理解本 notebook 每一節的判讀 | — |
| 2 | 對照業界做法 | 整理六個監理機關與大型公司的公開案例,以及台灣的「個人資料保護法」與金管會訂定的「金融服務業公平待客原則」(金融業對待客戶的十項原則,本案引用其中的商品或服務適合度、告知與揭露、友善服務三項);每一項逐一記錄「案例的要點、本案從該案例採用了哪些做法、本案刻意與該案例不同之處及理由」 | 說明本模組每一個檢查步驟都有對應的業界實務或法規依據,不是自創的做法 | — |
| 3 | 讀取資料、定義名單層級與年齡組 | 沿用 sql/05 [E0] 的客戶層級表;核對四個策略區與三個名單的人數 | 確保本模組檢查的對象與 M5、M6 是同一批客戶,而且四個旗標的定義、命中數、價值層的三等份切法、策略區的歸類規則都與 sql/04、sql/05 完全相同 | — |
| 4 | 檢查 A:圈選名單的族群組成與族群內品質 | (1) 對三個名單層級(挽留名單、人工聯繫名單、加碼回饋名單),分別依性別組與年齡組計算「選取率」(該族群中進入名單的比例)與「不利影響比率」(選取率最低族群除以最高族群);<br/>(2) 對三個名單層級,分別計算名單內各族群的「精確率」(名單內該族群中真的流失的比例)與「漏報率」(該族群真的會流失的客戶中沒有被圈進挽留名單的比例),並計算每個族群相對基準族群的精確率差距與漏報率差距各自的 95% 信賴區間。<br/>(基準族群:計算差距時當作比較基準的那一組:性別的基準族群取男性,沿用 M3 T3 與 M4 模型 B 的比較方向:M2 切片先發現女性流失率(17.36%)高於男性(14.62%),M3 T3 因此檢定「女性流失率是否高於男性流失率」、流失率差距以「女性減男性」計算,M4 模型 B 的性別欄位 gender_F 也以男性為基準組;本模組沿用同一方向,讓檢查 A 的差距與檢查 B 的勝算比都能與 M3、M4 的數字直接對照;<br/>年齡的基準族群取人數最多的 46–55 歲組,理由是基準組會出現在每一個差距的計算裡,基準組人數越多、它本身的抽樣誤差越小,所有差距的信賴區間就越窄、越好判讀<br/>(若選一個人數少的組當基準族群,每一個差距的信賴區間都會因為基準組本身的抽樣誤差而變寬。因為差距的標準誤同時包含兩組的抽樣誤差,基準組固定出現在每一個差距裡,它的誤差會傳到所有差距上);Cell M7-5 依「取人數最多的組」這條規則由程式自動選出並印出結果,表 A-1b 列有四個年齡組的人數可供核對哪一組人數最多);<br/>(3) 人工聯繫與加碼回饋這兩種動作只在部分客戶範圍內分配(人工聯繫只給高價值層、加碼回饋只給低成本區),所以漏報率除了以全體客戶為分母算一次,另以「該動作的候選範圍」為分母再算一次;原因是以全體為分母時,不在候選範圍內的流失客戶也會被算成「被漏掉」,會把各族群在價值層、策略區的人數比例差異混進漏報率。因此,只有以候選範圍為分母,才能回答「在同一個候選範圍內,各族群後來真的流失的客戶,有多少比例沒有被規則選到,這個比例是否相近」<br/>(人工聯繫的候選範圍是高價值層 3,375 人,規則只從其中選出命中 ≥ 2 的 426 人;加碼回饋的候選範圍是低成本區 2,050 人,規則只從其中選出消費變化比 < 0.50 的 308 人) | 回答「三個名單是否對某一性別或年齡組系統性地不利」;若各族群得到的資源有差異,區分差異是「該族群確實流失率較高」造成,還是「規則漏掉該族群的流失客戶」所造成的 | #29 |
| 5 | 檢查 B:「從模型 B 移除性別特徵」的代價 | 以同一組資料訓練三個版本的模型 B:含性別版(M4 原樣)、不含性別版、同時不含性別與婚姻狀態版;在五個隨機切分下,比較三版模型在同樣名單人數(178 人與 703 人)下的精確率、各版名單中的女性占比,以及不含性別版名單與含性別版名單有多少客戶相同(重疊率) | 回答「模型 B 能不能保留性別作為輸入變數」與「若不保留,名單精確率會下降多少」 | #27 |
| 6 | 「不採用公平性套件」的取捨判斷 | 說明 Fairlearn、AIF360、Aequitas 等公平性套件的用途、優缺點與本案不使用的理由 | 讀者與審核者(法遵或風險控管人員)能確認本案自行計算指標的做法有明確依據,並知道在什麼條件下應改用套件 | — |
| 7–10 | 結論、法遵對應摘要、方法限制、待驗證事項清單 v8 | 兩項檢查的結論與對名單規則的決定;四點法遵對應的摘要(完整版在 `docs/07_compliance_fairness.md`);方法限制;待驗證事項清單 v8 與 M8 開工檢核 | M8 可直接承接待驗證清單 | 待驗證事項 #27、#29 結案 |


**本模組承接的待驗證事項**(notebook 06 Cell M6-14 清單 v7):<br/>
(1) 待驗證事項 #27(M4 交辦):模型 B 中女性的勝算比為 2.12(控制其他變數後,五個切分皆顯著),代表只要模型 B 把性別當成輸入變數,它對每一位女性客戶算出的流失機率都會比條件相同的男性客戶高,也就是「以性別為特徵的模型會系統性地把女性列為較高風險」;M7 應比較「含性別」與「不含性別」兩版模型的精確率,量化移除性別的代價,並從法遵角度說明模型 B 若用於圈選名單,能否保留性別作為依據。補充說明兩件容易混淆的事:第一,M3 T3 的結論是「女性流失率 17.36%、男性 14.62%,差距 2.74 個百分點,統計上顯著但實務上不值得為此設計性別導向的挽留方案」,那是「單獨看性別一個變數」的結論;M4 模型 B 的 2.12 倍是「控制行為、額度、收入等其他變數之後」性別的獨立效果,兩者回答的是不同問題,並不矛盾。第二,本模組不是要「用性別圈名單」,而是相反:M5 把含性別的模型 B 列為升級選項,M7 要回答「這個選項裡的性別特徵能不能留」。<br/>
(2) 待驗證事項 #29(M5 交辦):M7 檢視四個策略區的性別與年齡組成。M5 的分群矩陣雖然沒有直接用性別、年齡分群,但用來分群的行為變數可能與性別、年齡有關聯,例如:若男性客戶高頻率聯繫客服的比例本來就高於女性,那麼「高聯繫」這個旗標就會較常命中男性,名單就會間接受性別影響(本模組第 4 節實測確認了這個例子:聯繫 ≥ 5 次的客戶占男性的 2.81%、占女性的 1.79%)。因此要檢查「各族群進入名單的比例(選取率)是否明顯不同」;若明顯不同,須說明名單是否構成差別待遇或差別性影響。<br/>
(3) notebook 06 對 M7 開工的提醒:M6 的實驗分組名單在性別、年齡上已做過平衡檢查,實驗組(257 人)中女性占 54.1%、對照組(260 人)中女性占 56.9%,兩組女性占比差距的標準化平均差(SMD,兩組平均值的差除以合併標準差)為 −0.057,絕對值小於 0.1 的慣例門檻,代表兩組的性別組成相近。因此,若本模組發現某個策略區內的女性比例偏高,這是「母體整體組成」的名單公平性問題(本模組要處理的事項),而不是 M6 關心的「同一母體內,實驗組與對照組是否可比」的問題,M6 的實驗結論不受本模組結果影響。<br/>


**本模組不處理的事項(說明本模組的工作範圍到哪裡為止)**:<br/>
(1) 不使用公平性套件,理由見第 6 節;<br/>
(2) 名單層的族群組成只檢視性別與年齡,不檢視婚姻狀態與其他特徵;婚姻狀態只在模型層(檢查 B)以「多訓練一版不含婚姻狀態的模型」處理,理由見上文「檢視的特徵與名單層級」;<br/>
(3) 不做後處理(post-processing,指模型訓練完成後,對不同族群設定不同的名單門檻,把各族群的選取率拉到相同),理由見第 1 節 1.6;<br/>
(4) 不對法條做全文論述;`docs/07_compliance_fairness.md` 對每一條法規或指引只寫出條號(或原則名稱)與該條的核心要求,並說明其對應本案的哪一項檢查或決定,不逐條解釋條文內容;<br/>
(5) 不重新設計 M5 定案的名單規則;若本模組檢查發現規則有問題,本模組只寫明「應修改的方向」,「是否修改」交由業務單位(問題定義書的決策者,挽留專案負責人)決定,本專案在待驗證事項清單中記錄這個待決事項。<br/>

**資料來源**:<br/>
`clean.customers`;客戶層級表(四個旗標、命中數、價值層、策略區、額外回饋旗標)沿用 sql/05_experiment_population.sql [E0] 的 `m6_base` 中的定義,本 notebook 以同一段 CTE 執行並在 Cell M7-4 印出全文;模型 B 的特徵處理沿用 notebook 04 Cell M4-13。<br/>

**判讀格式**:沿用本案通用的四段格式:實測 / 判定 / 對決策的意涵 / 待辦或限制。<br/>

**執行原則**:commit 前一律 Restart & Run All;模型切分沿用 M4 的五個 random_state(0、1、7、42、99),使檢查 B 的「含性別版」數字與 notebook 04 第 7 節完全相同,可直接對照。<br/>

Cell M7-2(markdown)|第 1 節:本模組使用的概念

## 第 1 節|本模組使用的概念

本節逐一說明本模組用到的公平性概念。每一個概念都依「定義 → 為什麼需要這個概念 → 本案的設定」三段撰寫,是第 4 節(檢查 A)與第 5 節(檢查 B)判讀時的共同依據。M2 至 M6 已經定義過的名詞(精確率、召回率、信賴區間、勝算比等)在 1.10 以一兩句話重述。<br/>

### 1.1 受保護特徵

**定義**:<br/>
受保護特徵(protected attribute)是法規明文列出、認定「不得作為差別對待依據」的個人屬性,常見的有性別、年齡、種族、宗教、國籍、婚姻狀態、身心障礙;每個國家、每種業務領域列出的清單都不同。機器學習文獻另有「敏感特徵」(sensitive attribute)一詞,範圍比受保護特徵更廣,除了法規列出的屬性,也包含法規沒有列出、但社會上認為應該謹慎處理的屬性;本模組檢視的性別、年齡、婚姻狀態都是法規明列的屬性,所以全模組統一使用「受保護特徵」這個詞。<br/>
舉例:(1) 美國的平等信用機會法(Equal Credit Opportunity Act, ECOA)在授信領域列出的受保護特徵包含性別、年齡、婚姻狀態、種族、宗教、國籍、是否領取社會救助;(2) 台灣的個人資料保護法第 6 條則把病歷、醫療、基因、性生活、健康檢查、犯罪前科六類資料列為「特種個人資料」,原則上禁止蒐集與利用,但這六類是「不得蒐集」的敏感資料,**與本節所說的「不得作為差別對待依據」的受保護特徵是兩個不同的概念**,本案資料集也沒有這六類欄位。<br/>
**為什麼需要這個概念**:<br/>
分析者若不先列出哪些特徵受保護,就無法檢查「名單或模型有沒有依這些特徵差別對待客戶」;任何檢查都必須先有明確的檢查對象。<br/>
**本案的設定**:<br/>
本資料集的 21 個欄位中,屬於典型受保護特徵的是「性別(gender)、年齡(customer_age)、婚姻狀態(marital_status)」三欄。本模組的名單層檢查(檢查 A)以「性別與年齡」為對象;婚姻狀態只在「模型層」檢查(檢查 B)以「多訓練一版不含婚姻狀態的模型」處理,名單層不檢查婚姻狀態,理由已在 Cell M7-1「檢視的特徵與名單層級」說明,同時列為第 9 節的方法限制。
收入(income_category)、教育程度(education_level)、扶養人數(dependent_count)三欄不列為檢查對象,理由有兩個:第一,各國的反歧視法規(就業、授信、保險)普遍都不把這三項列為受保護特徵,其中授信是管制最嚴格的領域,連授信法規都不列入的屬性,在風險較低的行銷用途更沒有理由列入;第二,公平性檢查的對象是「規則不應該據以差別對待」的特徵;而收入、教育程度、扶養人數描述的是客戶的經濟能力與家庭狀況,挽留動作依這些資訊差別設計(例如依收入判斷某個回饋方案是否適合該客戶)是法規允許、業務上也應該做的事,所以名單依這三欄產生的差異不是歧視,不需要列入檢查。<br/>


### 1.2 差別待遇與差別性影響

**定義**:<br/>
(1) 差別待遇(disparate treatment)指:「直接依受保護特徵給予不同的對待」。例如名單規則寫成「女性客戶不列入優惠名單」,或模型把「性別當成輸入特徵、使女性客戶的風險分數整體偏高」。後者之所以是差別待遇,原因是:邏輯斯迴歸會對每一個輸入特徵估計一個係數,性別特徵進了模型,模型就會估計「其他條件相同時,女性相對男性的流失勝算比」;M4 模型 B 估出的勝算比是 2.12,代表模型對每一位女性客戶算出的流失機率都比條件相同的男性客戶高。這是把性別特徵放進模型之後必然的結果,不是模型出錯;只要名單用這個分數排序,每一位女性客戶就會單純因為性別被排在較前面,這就是直接依受保護特徵給予不同的對待。<br/>
(2) 差別性影響(disparate impact)(又稱「間接歧視」)指:「規則表面上沒有使用受保護特徵,但執行結果對某一個族群明顯不利」。例如名單規則只用交易筆數作為條件來圈選,但因為某一個年齡組的交易筆數普遍較低,這一組客戶被大量排除在優惠之外,就是差別性影響。<br/>

**為什麼需要區分這兩個概念**:<br/>
因為兩者的檢查方法與處置方式不同。要檢查差別待遇,只要檢視規則或模型的特徵清單中有沒有受保護特徵即可確認;**要檢查差別性影響,必須實際算出各族群進入名單的比例與名單在各族群內的品質,看「執行結果」才知道**。美國公平授信(fair lending)的審查慣例是兩者都審查;其中差別性影響是「即使沒有歧視的意圖也可能構成違規」,除非機構能提出業務必要性(business necessity)的理由,而且證明沒有「效果相當、但對受保護族群影響較小」的替代做法(見 1.7)。<br/>

**本案的設定**:M5 的規則式名單只使用四個行為旗標與交易金額,沒有使用性別與年齡,所以「名單層沒有差別待遇的問題」,**要查的是差別性影響,這就是檢查 A 的工作**。M4 的**模型 B 把性別當成輸入特徵,若日後直接用模型 B 的分數圈選名單,就會構成差別待遇(見上方定義(1))**,所以檢查 B 要回答「模型 B 不用性別會怎樣」。(M4 模型 B 的 27 個特徵包含 gender_F 這一欄,notebook 04 Cell M4-13 特徵表)。<br/>



### 1.3 代理變數,以及為什麼「無知即公平」不能完整處理公平性問題

**定義**:<br/>
代理變數(proxy variable)是「本身不是受保護特徵,但與受保護特徵高度相關,因此會間接帶入該特徵資訊」的變數。例如郵遞區號可能是種族的代理變數,購買某一類商品的紀錄可能是性別的代理變數。「無知即公平」(fairness through unawareness)指「只要不把受保護特徵放進模型或規則,公平性議題就算處理完畢」的想法。<br/>

**為什麼「無知即公平」不能完整處理公平性問題**:<br/>
若其他變數是受保護特徵的代理變數,把受保護特徵移除之後,模型仍然會透過代理變數重現同樣的差別;而且因為受保護特徵已經不在模型裡,分析者反而更難察覺到並檢查這個差別。一項針對從業者的訪談研究(Deng 等人,2022,來源見第 2 節)發現,多數從業者遇到性別變數時直接刪除此特徵已處理公平性議題,但沒有進一步檢查「其他特徵是否為性別的代理變數」,這是實務上最常見的錯誤。<br/>
另一個方向的問題是:紐約州金融服務署(New York State Department of Financial Services, NYDFS)2021 年的 Apple Card 調查報告指出,收入、資產、信用紀錄這類「看起來中性」的財務變數,本身可能承載歷史上累積的性別與種族差異(報告稱為 legacy bias,既有偏差),所以「只用這些中性變數」不代表結果就不會有偏差。<br/>

**本案的設定**:<br/>
本模組不以單純「把性別特徵移除」當作處理方式,而是做三件事:(1) 先量化「移除性別要付出的代價」(檢查 B);(2) 檢查「只用行為變數圈出的名單」,這份名單的性別與年齡的實際組成是什麼(檢查 A),這一步就是在查「四個行為旗標有沒有在當性別或年齡的代理變數」;(3) 把「行為變數可能承載既有差異」寫在第 9 節的方法限制,不宣稱「檢查通過等於沒有偏差」。<br/>


### 1.4 選取率、不利影響比率與五分之四法則

**定義**:<br/>
**選取率(selection rate)**= 某一個族群中「被列入名單(也就是會收到某個挽留動作)」的人數 ÷ 該族群的總人數,回答「在同一個族群裡,每 100 人有幾人收到這個動作」。<br/>

**不利影響比率(adverse impact ratio, AIR)**用來回答一個問題:「這份名單中,被選取比例最低的族群,比起被選取比例最高的族群,差了多少?」它是判斷名單有沒有差別性影響的第一道篩選指標。<br/>
計算方式是:以選取率最高的族群為分母,每一個族群各算一個 AIR = 該族群的選取率 ÷ 選取率最高族群的選取率。選取率最高的族群拿自己除以自己,AIR 一定等於 1.000(它是這份名單中被選取比例最高的族群,不可能是被選取比例偏低的一方,所以不需要判讀它的 AIR);其他族群的選取率都比它低,AIR 都介於 0 到 1 之間,越接近 1 代表該族群被選取的比例越接近最高的族群。因為每一個族群各有一個 AIR,判讀時要看的是「最低的那一個」,原因是五分之四法則(見下)問的是「有沒有任何一個族群的選取率低於最高族群的 80%」,只要最低的那個族群沒有低於 0.8,其他族群就更不會低於 0.8;所以 Cell M7-1 Roadmap 所寫的「選取率最低族群除以最高族群」,就是指這個最低的 AIR。<br/>
以本案的挽留名單為例:依性別,男性的選取率 25.73% 最高(AIR 1.000),女性 23.31% ÷ 25.73% = 0.906,判讀時看 0.906;依年齡組,26–35 歲組的選取率 28.40% 最高(AIR 1.000),46–55 歲組 23.14% ÷ 28.40% = 0.815 最低,判讀時看 0.815(表 A-2、A-3)。<br/>
補充說明:AIR 低於 0.8 就要提醒「該族群選取率可能偏低,要觀察是否有差別性影響」。「選取率最高」是指「這個族群被圈入挽留名單的比例最高」。<br/>

**五分之四法則(four-fifths rule)**是美國就業與授信審查使用的經驗法則:在一份名單中,若某一個族群的選取率低於選取率最高族群的 80%(也就是該族群的 AIR 低於 0.8),就視為「這份名單可能對該族群(選取率較低的那個族群)有不利影響,需要進一步解釋原因」。<br/>

**為什麼需要這個概念**:<br/>
這些指標是最直觀的第一道檢查,把「名單有沒有差別性影響」變成一個「可以比較的數字」;而且計算只需要三個步驟:把客戶依族群分組(例如分成女性、男性兩組)、數出每一組的總人數與進入名單的人數、兩者相除;任何人拿到名單都能重算與核對。<br/>

**本案的設定**:<br/>
對「三個名單層級(挽留名單、人工聯繫名單、加碼回饋名單)」各算一次「性別組」與「年齡組」的「選取率」與「AIR」。<br/>
AIR 的 0.8 在本案只作為 **「需要在判讀中解釋」的提醒線**,不作為法遵門檻,理由有兩個:<br/>
(1) 五分之四法則源自美國就業與授信的監理脈絡,台灣的法規與指引沒有這個數字。<br/>
(2) 本案的名單規則是依行為旗標圈選,而行為旗標是流失風險的訊號,所以一個族群若真的流失得比較多,命中旗標的人就比較多,進入名單的比例(選取率)自然較高;相反地,真的流失得比較少的族群,選取率自然較低。<br/>
本案的不同性別、不同年齡組的真實流失率本來就不同(M3 T3:女性 17.36%、男性 14.62%),所以某一個族群的選取率較低、AIR 低於 0.8,有可能只是「這個族群實際狀況本來就比較不會流失」,不是規則對它不利。若要分辨到底是「這個族群實際狀況本來就比較不會流失」還是「規則真的對它不利」哪一種情況,必須再看這個族群「真的會流失的客戶」有沒有被名單漏掉(1.5 的漏報率):**若沒有被漏掉,選取率低只是反映真實風險低;若被漏掉的比例明顯較高,才是規則對它不利**。<br/>


### 1.5 族群切片評估:精確率與漏報率

**定義**:<br/>
族群切片評估(sliced evaluation)指:同一個評估指標,不只以全體客戶(本案 10,127 人)算一個總值,還要「依族群分組之後,每一組各算一次該指標的值,再比較各組之間該指標值的差距」;本案依性別分成 2 組、依年齡分成 4 組。<br/>

本案對下列兩個指標做族群切片:<br/>
(1)精確率(precision)= 名單內該族群中實際流失的人數 ÷ 名單內該族群的人數,回答「圈進名單的這群人裡,有多少是真的會流失的」;<br/>
(2)漏報率 = 該族群所有流失客戶中沒有進入名單的人數 ÷ 該族群所有流失客戶的人數,也就是偽陰性率(false negative rate, FNR),回答「這個族群真的會流失的客戶裡,有多少被名單漏掉了」。<br/>

**為什麼需要這個概念**:<br/>
只看全體的精確率與漏報率,會看不到「某一個族群的精確率特別低、或漏報率特別高」的情況,因為「人數多的族群」會導致「全體的數字」更接近「人數多族群的數字」。<br/>
Google 的 Fairness Indicators 工具(來源見第 2 節)就是以這個理由設計的,此工具的做法是「把偽陽性率與偽陰性率依族群切片、與基準族群比較、並附上信賴區間」。<br/>
對本模組的判斷而言,**漏報率是最關鍵的指標**,原因是:本模組要判斷的是「名單規則有沒有讓某一個族群系統性地得不到應有的挽留資源」,而被名單漏掉的流失客戶,就是「應該收到挽留動作卻沒有收到」的客戶;若某一個族群的漏報率明顯高於其他族群,就表示這個族群真的會流失的客戶被名單漏掉的比例較高,也就是這個族群得到的挽留資源系統性地比其他族群少,這正是「規則對該族群不利」的具體意思。
**精確率則回答另一個問題**:「名單在這個族群身上是不是誤圈了較多不會流失的人」,精確率越高,誤圈越少,反之亦然。誤圈多是「成本問題的考量」,不代表對該族群不利。<br/>
精確率不用來判斷名單對族群是否不利,但本模組仍然要算,原因有三個:(1) 解釋選取率的差異:當兩個族群的漏報率相近、選取率卻不同時,差異就是來自「誤圈人數不同」,精確率就是用來證明誤圈人數的數字(見下方規則 (a));(2) 作為名單規則的修改依據:某一個族群的精確率明顯較低,代表四個旗標在這個族群身上的區分力較弱,是日後修改規則時要優先檢視的族群;(3) 成本:誤圈的客戶也會收到挽留動作,精確率低代表挽留預算花在不會流失的客戶身上的比例高。<br/>

**本案的設定**:<br/>
1.對三個名單層級依性別組與年齡組各算「精確率」與「漏報率」,族群之間的差距附 95% 信賴區間(沿用 M3 的兩比例比較方法)。<br/>

2.漏報率的分母(母體)有兩種算法,本模組兩種都算、並列呈現,各自回答不同的問題:<br/>

| 名單層級 | 以全體客戶為母體(表 A-4、A-5) | 以該動作的候選範圍為母體(表 A-6、A-7) | 
|---|---|---|
| 挽留名單(2,476 人) | 算。候選範圍就是全體客戶,所以只算這一種 | 不需要另算 | 
| 人工聯繫(426 人) | 算。回答「全體流失客戶中,有多少比例沒有收到人工聯繫」,是資源分配的總覽 | 算。母體為高價值層 3,375 人,回答「同樣在候選範圍內的流失客戶,各族群被人工聯繫覆蓋的比例是否相近」 | 
| 加碼回饋(308 人) | 算。意義同上 | 算。母體為低成本區 2,050 人,意義同上 | 

判斷「是否構成差別性影響」只用候選範圍那一欄,因為不在候選範圍內的流失客戶本來就不會收到這個動作,不是被規則漏掉;以全體為母體的數字仍然列出,原因是它是第一版的算法,並列可以讓讀者看到兩種分母得到的結論差多少(理由見 Cell M7-1 Roadmap 第 4 節與第 4 節判讀的修正一)。<br/>

3.本模組要判斷的核心問題只有一個:名單規則有沒有讓某一個族群系統性地得不到應有的挽留資源,也就是 1.2 所說的差別性影響;判斷的標準是「這個族群真的會流失的客戶,被名單漏掉的比例是否明顯高於其他族群」。下列三條判讀規則都以這個標準為準,以漏報率為主、選取率與精確率為輔。判讀規則在看到數字之前就先寫定,原因是:若看到結果之後才決定門檻,分析者很容易不自覺地挑一個讓結果好看的門檻。<br/>

(a) **若各族群的漏報率相近**(每一個族群相對基準族群的漏報率差距,其 95% 信賴區間都包含 0) → 統計上看不出名單對各族群流失客戶的覆蓋程度有差異,**判定「不構成差別性影響」**。在這種情況下,**若各族群的選取率仍然不同,差異來自誤圈人數不同,屬於成本問題**,原因如下:一個族群進入名單的人,可以拆成「真的會流失、被圈到的人」加上「不會流失、被誤圈的人」;漏報率相近代表第一種人的比例在各族群相近;所以選取率(進名單人數 ÷ 族群人數)若仍然不同,只能是第二種人(誤圈)的多寡不同,也就是精確率不同;誤圈的人多,代表挽留預算有較多花在不會流失的客戶身上,這是成本問題。但沒有任何一個族群被少給資源。<br/>
(b) **若某一個族群的選取率較高、但名單內該族群的精確率明顯較低** → 代表**規則在這個族群身上誤圈較多。這一點不構成「對該族群不利」**,原因是本模組判斷「不利」的標準是「應該收到挽留動作的客戶有沒有被漏掉」,而這個族群的情況是收到挽留動作的人比較多(包含許多誤圈的人),沒有「收到挽留動作的人較少」的問題。<br/>
「該族群的精確率明顯較低」,代表四個旗標在這個族群身上的區分力較弱,因此**列為日後修改名單規則時應優先檢視的對象**。<br/>
(c)**若某一個族群的漏報率明顯較高**(該族群相對基準族群的漏報率差距,其 95% 信賴區間不包含 0 且在 0 以上) → 代表「規則系統性地漏掉這個族群真的會流失的客戶,這個族群得到的挽留資源系統性地比其他族群少」,這就是**對該族群不利,也就是差別性影響**,須修改名單規則。<br/>

補充說明:「相近」與「明顯較高」在本模組的操作定義:<br/>
兩個族群漏報率的差距(例如女性漏報率減男性漏報率)會附一個 95% 信賴區間,這個區間是「考慮抽樣波動之後,真實差距可能落在的範圍」。若區間包含 0(例如本案挽留名單的兩性漏報率差距 +1.09 個百分點,區間 −3.31 至 5.49),代表真實差距有可能是正、是零、也有可能是負,統計上無法排除「兩者沒有差距」的可能,所以只能說「看不出差異」,本節以「相近」指這種情況;若區間不包含 0 而且整個區間都在 0 以上(例如兩性漏報率差距 +10.08 個百分點,區間 6.99 至 13.17),代表即使考慮抽樣波動,女性族群的漏報率仍確定會高於男性族群(基準族群),本節以「明顯較高」指這種情況。<br/>



### 1.6 兩種公平性定義,以及「這兩種無法同時滿足」的原因

**定義**:<br/>
公平性沒有單一的定義,最常被拿來比較的兩種定義是:<br/>
(1)**人口比例平等(demographic parity)**,要求**各族群的選取率相同**(選取率:某族群「全體客戶」,被圈進名單的人數比例)。<br/>
(2)**機會平等(equal opportunity)**(本案的主要判準),要求**各族群的漏報率相同**(漏報率:某族群「真正會流失的客戶」,沒有被圈進名單的人數比例)。<br/>

**為什麼兩者不能同時滿足**:<br/>
當各族群的真實流失率(統計上稱為基準率,base rate)不同時,**在不做差別待遇的前提下,這兩個公平性定義無法同時成立**。以本案的數字說明:<br/>
女性的流失率是 17.36%、男性是 14.62%。若名單對兩性的「漏報率相同(機會平等成立)」,「女性因為流失客戶的比例較高,**進入名單的比例必然也較高**,選取率就不會相同」;相反地,若在名單人數固定的前提下,強制讓兩性的選取率相同,名單就必須少圈一些女性、多圈一些男性;多圈進來的男性有兩種可能:若他們是會流失的客戶,男性被漏掉的流失客戶就變少、男性漏報率下降,而被移出名單的女性中有真的會流失的客戶、女性漏報率上升,兩性的漏報率不再相同,機會平等就被破壞;所以要在選取率相同的同時維持漏報率相同,多圈進來的男性只能是不會流失的客戶(誤圈)。無論哪一種,被移出名單的女性流失客戶都會使女性的漏報率上升。<br/>
這個衝突在公平性文獻中是已知的結果。由上述推理可知,在基準率不同時,要讓兩種定義同時成立,唯一的方法是對基準率較低的族群(本案為男性)另外設定一個較寬的圈選門檻,刻意多圈不會流失的男性,把男性的選取率拉到與女性相同;但「依性別設定不同的圈選門檻」本身就是 1.2 所說的差別待遇。因此,只要不做差別待遇,兩種公平性定義就不可能同時成立,分析者必須選定其中一種公平性定義作為主要判準並說明理由,不能宣稱兩種都滿足。<br/>

補充說明:一個族群的選取率 = 被圈到的流失客戶 + 被圈到的非流失客戶,除以族群人數。若多圈進來的男性全部是會流失的,男性被漏掉的流失客戶就變少,男性的漏報率下降;同時被移出名單的女性中有流失客戶,女性的漏報率上升;兩性的漏報率就不再相同,機會平等被破壞。所以若要「選取率相同」與「漏報率相同」同時成立,多圈進來的男性就只能是不會流失的客戶,這是數學上的必然。<br/>


**本案的設定**:<br/>
**本案以機會平等(各族群漏報率相近)為主要判準**,「各族群的選取率差距」與「各族群的 AIR」則一併報告並解釋。<br/>
理由是:挽留資源的目的是找到會流失的客戶並留住他們。從公司的角度看,應該被挽留卻被名單漏掉的客戶,就是本來可以留住卻流失的收入;從公平性法規的角度看(法規保護的對象是客戶),某一個族群的客戶若被名單系統性地漏掉,就是這個族群系統性地得不到本來會收到的挽留優惠。兩個角度指向同一個結論:名單最不應該發生的事是「漏掉真的會流失的客戶」,所以本案以漏報率是否相近(機會平等)為主要判準。若為了讓各族群的選取率相同而刻意少圈女性流失客戶,公司會損失本來可以留住的客戶,女性客戶也會系統性地得不到本來應有的挽留動作,對公司與對女性客戶都不利。<br/>
這也是本模組不做「後處理」的原因。後處理(post-processing)指「模型訓練完成後,對不同族群設定不同的名單門檻、把各族群的選取率拉到相同」;這種做法等於「明文依不同性別而設定不同的規則」,後處理這個動作本身就是 1.2 所說的差別待遇,而且在挽留情境下會製造前述的反效果:系統性地讓特定族群得不到本來應有的挽留動作。<br/>


### 1.7 較少歧視替代方案與將「移除性別特徵的代價」量化

**定義**:<br/>
較少歧視替代方案(less discriminatory alternative, LDA)是美國公平授信審查的概念:若一個規則或模型造成差別性影響,機構必須證明這個規則或模型有業務必要性,而且「沒有一個預測效果相當、但對受保護族群影響較小的替代方案」;審查時會要求機構實際尋找過替代方案,並留下每一個候選方案的紀錄,包括:候選方案的特徵集(例如「含性別的 27 個特徵」與「不含性別的 26 個特徵」是兩個候選方案)、公平性指標(例如各候選方案圈出的名單中女性占比、各族群的漏報率)、效果指標(例如各候選方案在 703 人名單的精確率)、採用或否決的理由(例如「不含性別版的精確率只低 0.2 個百分點,採用」)。<br/>

**為什麼需要這個概念**:<br/>
只要計算出「不使用性別特徵的模型,名單精確率會下降多少」,決策者就能把法遵風險與效果損失放在一起權衡,例如本案算出 703 人名單的精確率下降 0.2 個百分點,相當於每 703 位名單客戶少留住約 1 位,決策者可以判斷「少留住 1 位客戶」與「避免依性別差別對待的法遵風險」哪一個比較重要。<br/>

**本案的設定**:<br/>
1.檢查 B 是「較少歧視替代方案」的最小版本(完整的版本會對許多候選特徵集逐一比較(例如逐一移除 27 個特徵中的每一個),本案只比較與受保護特徵直接相關的三個特徵集,因為本案要回答的只有「性別(與婚姻狀態)能不能留在模型裡」這一個問題)。<br/>
2.以同一組資料訓練三個版本的模型 B,分別是「含性別版(M4 模型 B 原樣,27 個特徵)」、「不含性別版(26 個特徵)」、「同時不含性別與婚姻狀態版(23 個特徵:婚姻狀態有四個類別,轉成虛擬變數後以 Married 為基準組,占 3 個特徵欄位,所以移除婚姻狀態會減少 3 個特徵)」,三版模型的其他設定則完全相同。<br/>
3.三版模型在固定名單規模下,**比較精確率(M4 第 7 節的評估方式,見 1.10 定義)**,並**記錄「各版名單的女性占比」與「名單之間的重疊率」**。<br/>
各版名單的女性占比:各版模型取 703 人名單時,名單中女性客戶的比例。<br/>
名單之間的重疊率:不含性別版的 703 人名單與含性別版的 703 人名單中,「相同客戶的人數 ÷ 703」;重疊率越高,代表移除性別之後名單實際上被換掉的客戶越少,表示**性別對名單排序的影響越小**。<br/>

4.檢查 B 的判讀規則同樣在看到數字之前先寫定(判讀規則決定「結論」與「代價」):<br/>
**本模組對此的判讀規則:無論移除性別特徵之後,模型的精確率下降多少,本模組的決策都是「若日後用模型 B 圈選名單,不以性別作為輸入特徵」**:<br/>
原因有兩個。第一,法規與指引:金管會「金融業運用人工智慧(AI)指引」的公平性原則要求金融機構避免 AI 系統因性別、年齡等因素對客戶產生偏見與歧視,美國 ECOA 更明文禁止授信決策考量性別;以性別作為圈選名單的依據,就是 1.2 所說的直接依受保護特徵給予不同待遇。第二,可說明性:同一份指引的透明性與可解釋性原則要求金融機構能向客戶與主管機關說明決策邏輯;若模型含性別,當客戶、業務單位或主管機關問「這位客戶為什麼進入挽留名單」時,理由會包含「因為她是女性」,而這正是第一點所說法規要求避免的理由,所以不是可以提出的合理說明。<br/>


如何衡量代價:使用「精確率」作為指標。在固定名單規模(178 人與 703 人)下,不含性別版模型名單的精確率,比含性別版模型名單的精確率低多少個百分點,取五個切分的平均。<br/>
判斷「下降多少算有實質差異」的基準(M4 算出的一個數字):178 人名單中,名單內每多一位真的會流失的客戶,精確率就上升 1 ÷ 178 ≈ 0.6 個百分點;反之,名單內每少一位真的會流失的客戶,精確率就下降 0.6 個百分點。所以 1 個百分點以內的精確率差距,相當於 178 人名單中不到 2 位客戶的差別,與不同切分之間的抽樣波動差不多大,因此「1 個百分點以內的精確率差距」不能視為真正的差異。三個區間如下:<br/>
(a) 「不含性別版比含性別版的精確率」下降小於 1 個百分點 → 結論寫「性別對名單排序的貢獻很小,移除性別幾乎不需付出精確率代價」。<br/>
(b) 「不含性別版比含性別版的精確率」下降達到 3 個百分點以上 → 結論仍然是「不以性別作為輸入特徵」(如前述的判讀規則原因),但必須如實寫出下降的百分點數(這個範圍的下降相當於 178 人名單中 5 位以上客戶的差別,703 人名單中 21 位以上),並說明這個精確率損失是用「降低法遵風險」與「名單對客戶、業務單位、主管機關可以說明」換來的,讓決策者知道代價是什麼。(計算:178 人名單中每一位客戶影響精確率約 0.56 個百分點(1 ÷ 178),3 個百分點 ÷ 0.56 ≈ 5.3,所以 3 個百分點以上相當於 5 位以上客戶的差別;703 人名單中每一位影響 0.14 個百分點,3 個百分點相當於 21 位以上)<br/>
(c) 「不含性別版比含性別版的精確率」下降介於 1 至 3 個百分點之間 → 結論仍然是「不以性別作為輸入特徵」(原因同前述的判讀規則),並如實載明下降的百分點數;與 (b) 的差別只在代價的大小:這個範圍的下降相當於 178 人名單中 2 至 5 位客戶的差別,須在結論中載明,但不需要像 (b) 那樣特別向決策者強調代價。<br/>

三個區間的決策相同(如前述的判讀規則原因),事先分段的目的是「避免看到數字之後再決定說明的措辭」。<br/>

無論結果落在哪一個區間,M5 目前預設的規則式名單本來就不含性別,不受檢查 B 的結果影響;但若日後依決策日誌 [M5-3] 的切換條件升級為模型 B 圈選名單,就必須採用不含性別版,而且切換條件中的成本節省比例也要依不含性別版重算(第 5 節意涵 (2))。<br/>


### 1.8 信賴區間在本模組的角色

notebook 03 第 2 節 2.2 已定義 95% 信賴區間(confidence interval);本模組沿用 95% 這個水準,與 M3 以來所有檢定一致(95% 對應 p 值 0.05,是統計與業界最常用的設定)。<br/>
本模組用信賴區間判斷「某一個族群與基準族群之間,精確率的差距或漏報率的差距,是不是抽樣波動就能造成的」。比較一律是兩兩比較:性別只有兩組,所以是女性對男性一次;年齡有四組,所以是其他三組各自對 46–55 歲組比較,共三次比較;婚姻狀態不在名單層檢查,沒有信賴區間。<br/>
差距的信賴區間寬窄取決於兩組的人數:人數越多,抽樣波動越小,區間越窄,差距的估計越可信。本案兩性各有四、五千人,所以挽留名單中兩性漏報率差距的信賴區間很窄(差距 +1.09 個百分點,區間 −3.31 至 5.49,寬度不到 9 個百分點);但年齡組中人數最少的 26–35 歲組(919 人)與加碼回饋名單(308 人)交叉之後,加碼回饋名單內的 26–35 歲客戶只有 16 人,這 16 人算出的精確率與 46–55 歲組的精確率差距(+12.09 個百分點),信賴區間是 −5.83 至 30.01,寬達 36 個百分點,實務意義是:樣本太少,算出來的差距可能完全是抽樣波動,不能拿來下結論。<br/>
判讀時,若差距的信賴區間跨過 0(例如上述 −3.31 至 5.49,區間同時包含負值與正值),代表真實差距可能是負、零或正,統計上無法排除「兩組沒有差距」,所以只能說「看不出兩組的漏報率有差異」,不能說「兩組漏報率沒有差異」。<br/>
若差距的信賴區間沒跨過 0,代表「統計顯著」,在 95% 信心水準下可認定兩組確實有差異存在。<br/>
「95% 信心水準」的意思是:用同樣的方法反覆抽樣並計算區間,長期會有 95% 的區間包含真實差距、5% 不包含;所以「確實有差異」這個認定帶有 5% 的誤判機率,這是分析者事先接受的錯誤機率,不是計算上的誤差。<br/>

本模組另外沿用決策日誌 [M2-10] 的小樣本慣例:某一個族群在某一個名單層級內的人數少於 100 人時(例如加碼回饋名單內的 26–35 歲組 16 人、56–73 歲組 56 人),不對這一格單獨下結論,只描述數字,並且列為監測項:名單每季重算時,重算這一格的人數、精確率與漏報率,等該格人數累積到 100 人以上後再判讀(第 10 節 #37)。<br/>


### 1.9 風險基礎方法

**定義**:<br/>
風險基礎方法(risk-based approach)指:金融機構使用的每一個 AI 或分析系統(本案就是挽留名單規則與模型 B),依該分析系統的決策對客戶權益的影響程度分級,若該系統的決策對客戶影響越大,上線前要做的審查程序(內部審查、外部第三方測試、上線後的持續監測)越嚴格。舉例:一個決定「客戶能不能核卡、額度多少、利率多少」的授信模型(系統),決策結果直接影響客戶能否取得金融服務,屬於高風險,上線前可能需要機構外的第三方測試。<br/>
另一個決定「哪些客戶先收到優惠推播」的行銷名單,客戶沒有收到推播並不會失去原有的服務,屬於較低風險,由機構內部依既定程序審查即可。<br/>
金管會 2024 年 6 月 20 日發布的「金融業運用人工智慧(AI)指引」就是以這個方法落實六項核心原則(建立治理及問責機制、重視公平性及以人為本的價值觀、保護隱私及客戶權益、確保系統穩健性與安全性、落實透明性與可解釋性、促進永續發展),由金融機構自行評估每一個 AI 系統的風險等級,高風險系統可能需要由機構以外、具 AI 專業的第三方進行審查或測試。<br/>

**為什麼需要這個概念**:<br/>
審查的嚴格程度要與風險相稱。若對每一個行銷分析都要求做與授信模型相同的審查(例如委託外部第三方測試、比較多種公平性定義),需要的人力、費用與時間會讓行銷分析無法在業務時程內完成,而且低風險的決策即使做了這些審查,客戶權益也不會因此多得到保障;所以監理機關不要求這樣做,機構也沒有必要這樣做。<br/>

**本案的設定**:<br/>
本案的挽留名單屬於行銷資源分配用途,風險等級低於授信與定價用途,因此本模組只做兩項檢查:檢查 A 檢查「名單在性別與年齡上的組成與品質」(各族群的選取率、AIR、精確率、漏報率),檢查 B 量化「從模型 B 移除性別特徵要付出的精確率代價」;不做獨立第三方審查與多種公平性定義的全面比較(完整的判斷依據見 Cell M7-1「風險定位」)。docs/07_compliance_fairness.md 第 1 節會寫明這個判斷,並說明「若名單用途改變,須重新評估要做哪些檢查」。<br/>


### 1.10 M2 至 M6 已定義的名詞

| 名詞 | 定義 | 出處 |
|---|---|---|
| 精確率(precision) | 名單內真的會流失的人數 ÷ 名單人數 | notebook 02 C3-b、notebook 04 第 1 節 1.6 |
| 召回率(recall) | 名單圈住的流失客戶人數 ÷ 全體流失客戶人數;漏報率 = 1 − 召回率 | 同上 |
| 偽陰性(false negative) | 又稱「漏報率」(本模組一致用語),實際流失但沒有進入名單的客戶,也就是 M4 混淆矩陣中的「漏抓」格 | notebook 04 第 1 節 1.7 |
| 95% 信賴區間(confidence interval) | 考慮抽樣波動之後,真實的「兩組的指標的比例差距」的合理範圍;信賴區間不包含 0 時,等同於 p 值小於 0.05(95% 為業界常用設定,1.8 已說明) | notebook 03 第 2 節 2.2 |
| 勝算比(odds ratio) | 勝算比 = 被比較族群的勝算 ÷ 基準族群的勝算(例如女性 ÷ 男性);邏輯斯迴歸的係數取 e 的次方,就是該變數的勝算比 | notebook 03 T2、notebook 04 第 1 節 1.3 |
| 固定名單規模比精確率 | 讓模型 B 取「預測流失機率」最高的 N 位客戶,再比較模型名單與規則名單各自的精確率;兩份名單的人數相同,比較才公平 | notebook 04 第 7 節、決策日誌 [M4-2] |
| 五個切分的穩定性檢查 | 以五個不同的 random_state(0、1、7、42、99)各切一次訓練集與測試集、各訓練一次模型,結論(例如「模型 B 的精確率高於規則 5 個百分點以上」)在五次都成立才採信,避免把單一次切分的抽樣波動當成結論 | notebook 04 第 7 節、Cell M4-17 |
| 標準化平均差(standardized mean difference, SMD) | 實驗組與對照組的平均值之差 ÷ 合併標準差;絕對值小於 0.1 視為兩組在該變數(例如女性占比、年齡)上的分布相近,也就是 M6 實驗所稱的分組平衡 | notebook 06 第 1 節 1.8 |
| 命中數、策略區、額外回饋旗標 | 命中數 = 四個行為旗標(0 或 1)相加;策略區 = 依「命中數 ≥ 2 或 ≤ 1」與「是否為高價值層」分成搶救區、低成本區、深化區、監測區四區;額外回饋旗標 = 低成本區內消費變化比 < 0.50 的客戶 | notebook 05 第 2、5 節、sql/05 [E0] |

補充說明:<br/>
模型 B 的「預測流失機率」是:模型 B 對每位客戶輸出的數值,本案只用來排序,不宣稱為流失機率(決策日誌 [M4-2])。<br/>


### 1.11 概念對應到本案的設定:總表

| 概念 | 本案的設定 | 依據 |
|---|---|---|
| 受保護特徵 | 性別(名單層與模型層都檢視)、年齡(只在名單層檢視)、婚姻狀態(只在模型層檢視) | 1.1、Cell M7-1 |
| 檢查對象 | 挽留名單(命中 ≥ 2,2,476 人)、人工聯繫名單(搶救區 426 人,候選範圍為高價值層)、加碼回饋名單(308 人,候選範圍為低成本區);另以「聯繫 ≥ 5 次人工回訪」名單(230 人)作為代理變數問題的對照 | notebook 05 第 8 節、Cell M7-1 |
| 差別待遇的檢查 | 規則式名單沒有使用性別、年齡 → 無差別待遇;模型 B 含性別特徵 → 由檢查 B 處理差別待遇的檢查 | 1.2 |
| 差別性影響的檢查 | 選取率、AIR(提醒線 0.8)、族群內精確率與漏報率、「各族群相對基準族群的精確率差距與漏報率差距,各自的 95% 信賴區間」 | 1.4、1.5 |
| 主要公平性判準 | 機會平等(各族群的漏報率相近);將「各族群選取率的差距」一併報告並解釋 | 1.6 |
| 將「移除特徵的代價」量化 | 模型 B 三個版本在 178 人與 703 人名單、五個切分下的精確率差距;各版名單的女性占比;名單重疊率(定義見 1.7) | 1.7 |
| 年齡分組 | 固定年齡段為 26–35、36–45、46–55、56–73 歲 | Cell M7-4 |
| 小樣本慣例 | 某族群在某名單層級內少於 100 人時,不單獨下結論 | 1.8、決策日誌 [M2-10] |
| 風險定位 | 行銷資源分配用途,不是授信用途;檢視項目為兩項:檢查 A(名單在性別與年齡上的選取率、AIR、精確率、漏報率)、檢查 B(從模型 B 移除性別特徵的精確率代價) | 1.9、Cell M7-1 |

Cell M7-3(markdown)|第 2 節:業界參考案例、台灣法規與指引,與本案的對應

## 第 2 節|業界怎麼做:六個參考案例、台灣的法規與指引,與本案的對應

**本節結論**:本模組的每一個檢查步驟(檢查 A 的選取率、不利影響比率、族群切片評估;檢查 B 的移除性別特徵代價的量化;風險定位;不採用公平性套件)都有監理機關或大型公司的公開做法作為依據,不是本模組自創的方法;本模組與這些做法刻意不同的地方,都是因為本案的風險等級較低(行銷資源分配,不是授信)或資料條件不同(公開資料集,沒有真實客戶的同意紀錄),而且每一處不同都寫明理由。<br/>

下表為參考來源,共六個業界案例及台灣的法規與指引(最後一列):<br/>

| 案例 | 案例的要點 | 本案採用的做法 | 本案刻意不同之處與理由 |
|---|---|---|---|
| **紐約州金融服務署(New York State Department of Financial Services, NYDFS)Apple Card 性別歧視調查報告(2021 年 3 月)** | 2019 年有多位消費者投訴 Apple Card 核給女性的信用額度低於條件相近的男性,NYDFS 因此展開調查。NYDFS 針對約 40 萬筆信用卡申請紀錄,以迴歸分析同時檢視差別待遇與差別性影響(兩者的定義見 1.2),結論是「信用特徵(收入、負債、信用紀錄等)相當的女性申請人與男性申請人,申請結果大致相同」,沒有發現違法的歧視。報告另外指出兩點:第一,收入、資產等財務變數本身可能承載歷史上累積的性別與種族差異,所以「模型只用財務變數、沒有用性別」不代表結果沒有偏差;第二,發卡機構無法向消費者說明額度是怎麼決定的,這種不透明才是消費者信任危機的主因 | (1) 檢查邏輯相同:NYDFS 的做法是「控制信用特徵之後比較兩性的結果」;M4 模型 B 的「控制其他變數後女性流失勝算比 2.12」就是同一種做法,檢查 B 則接著回答「這個性別效果能不能用在名單分配上」;<br/>(2) 「行為變數可能承載既有差異」這一點寫在第 9 節方法限制;<br/>(3) 「公司能向客戶說明每一位客戶為什麼進入名單」作為本案選用邏輯斯迴歸(M4)與規則式名單(M5)的法遵理由,寫在 `docs/07_compliance_fairness.md` 第 4 節 | NYDFS 檢視的是授信決策(信用額度的核給),本案檢視的是行銷資源分配,兩者的風險等級不同(見 1.9);所以本案不做「逐一複核每一位客戶的結果」這種個案層級的審查,只做族群層級的名單組成檢查與名單品質檢查 |
| **美國公平授信(fair lending)的模型偏差測試流程** | 美國金融機構為了符合 ECOA 等公平授信法規,對授信模型做偏差測試時,業界通行的標準步驟有四個:(1) 區分差別待遇與差別性影響;(2) 以不利影響比率(AIR)與五分之四法則做第一道篩選(定義見 1.4);(3) 代理變數分析,也就是把某一個特徵從模型移除之後,觀察各族群之間的選取率或錯誤率(模型預測錯誤的比例;本案使用的漏報率就是其中一種,定義見 1.5)差距是否明顯縮小,縮小就代表那個特徵在當「受保護特徵的代理變數」;(4) 較少歧視替代方案搜尋(定義見 1.7),也就是尋找預測效果相當、但對受保護族群影響較小的替代模型,並留下每一個候選方案的紀錄 | 檢查 A 的選取率與 AIR 就是第一道篩選;檢查 B 的三個模型版本就是較少歧視替代方案搜尋的最小版本:每一版的特徵集記錄在 Cell M7-7 的 VERSIONS 設定,每一版的精確率與名單的女性占比記錄在表 B-1 與表 B-2 | 兩處不同:第一,AIR 的 0.8 只作為提醒線、不作為門檻,理由見 1.4(台灣法規沒有這個數字;本案各族群的真實流失率本來就不同);第二,不做完整的較少歧視替代方案搜尋(完整的搜尋會逐一移除 27 個特徵中的每一個再比較),只測試與受保護特徵直接相關的三個版本,因為本案要回答的只有待驗證事項 #27「性別能不能留在模型 B 裡」,不是建立一個授信模型 |
| **Google Fairness Indicators(TensorFlow 團隊,2019 年 12 月)** | 這是 Google 公開的模型評估工具,做法是把分類模型的評估指標(特別是偽陽性率與偽陰性率)依族群切片各算一次、與基準族群比較、並附上信賴區間。工具的說明文件寫明:「只看全體的指標,會看不到某一個族群的錯誤率(例如漏報率)特別高」,以及「各族群的指標相近,不等於這個模型或名單就是公平的,指標本身不能達成公平」(意思是:指標只能量出「依分析者選定的族群與公平性定義,名單有沒有差距」,量不出沒有被選進來檢查的特徵、也改不了資料裡既有的差異;指標是檢查工具,不是處置) | 檢查 A 的族群切片評估採用同一種做法:精確率與漏報率依性別組、年齡組切片各算一次,與基準族群比較,附 95% 信賴區間(見 1.5、1.8);「指標相近不等於公平」可見第 9 節方法限制 | 不使用這個工具,因為它是 TensorFlow 生態系的套件,而本案的模型是 statsmodels 的邏輯斯迴歸,無法直接套用;本案改用 pandas 與 statsmodels 的兩比例比較方法自行計算,族群切片評估使用的指標只有精確率與漏報率兩個,每一步計算都寫在 Cell M7-5,審核者(法遵或風險控管人員)可以直接重算 |
| **金管會「金融業運用人工智慧(AI)指引」(2024 年 6 月 20 日發布)** | 這份指引是金管會對金融機構使用 AI 的行政指導,列出六項核心原則(全文見 1.9)。其中「重視公平性及以人為本的價值觀」原則要求金融機構「宜評估公平性,注意偏見之產生並儘可能避免歧視」;「落實透明性與可解釋性」原則要求「應能清楚說明 AI 系統如何運作及其預測或決策的邏輯」;指引以風險為基礎落實各項原則,由金融機構自行評估每一個 AI 系統的風險等級 | 指引要求金融機構自行評估每一個 AI 系統的風險等級,再依等級決定審查深度;本案據此把名單定為較低風險(行銷資源分配,Cell M7-1「風險定位」),檢視項目定為兩項:名單在性別組與年齡組的選取率、精確率、漏報率(檢查 A),以及模型 B 移除性別特徵後的精確率代價(檢查 B);公平性原則對應檢查 A 與檢查 B;可解釋性原則對應 M4 選用邏輯斯迴歸(決策日誌 [M4-0]):邏輯斯迴歸對每一個特徵估計一個係數,可以說出某一位客戶的預測流失機率高是因為哪幾個特徵;M5 選用規則式名單(決策日誌 [M5-0]):可以直接說出該客戶命中了哪幾個行為旗標。兩者都符合「能清楚說明預測或決策的邏輯」的要求;`docs/07_compliance_fairness.md`第 1 節先列出這份指引的六項原則,再逐項寫出本案的對應 | 這份指引是行政指導,沒有量化門檻;本案使用的門檻中,AIR 提醒線 0.8 與精確率差距 1 個百分點是本模組自訂並寫明依據的,小樣本慣例 100 人沿用決策日誌 [M2-10];三者都不是法規或指引的數字,本模組不宣稱它們來自法規或指引 |
| **英國金融行為監理總署(Financial Conduct Authority, FCA)禁止保險業「忠誠度懲罰」(2022 年 1 月生效)** | 忠誠度懲罰(price walking)指保險公司對續保的忠誠客戶逐年加價、對新客戶以低價搶客;FCA 認定這種「依客戶轉換保險公司的可能性差別定價」的做法損害忠誠客戶,要求續保價格不得高於提供給同等條件新客戶的價格,FCA 估計這項規定十年內可為消費者省下 42 億英鎊 | 這是監理機關直接管制「挽留手段」的案例,證明挽留手段本身會被監理機關檢視公平性。本案的加碼回饋(只給 308 人)與人工聯繫(只給搶救區 426 人),本質上也是依流失風險對不同客戶差別給予優惠;`docs/07_compliance_fairness.md` 第 5 節因此寫明這類差別給予優惠的可接受條件:依客戶的行為而不是身分給予、規則可以向客戶說明、未得到優惠的客戶不會因此失去原有的服務或被加價、加碼回饋只對實際達到消費門檻的客戶支付 | FCA 管制的是「讓一群客戶的待遇變差」(忠誠客戶被加價),本案的動作是「讓一群客戶的待遇變好、其他客戶不會失去原有的服務或被加價」(高風險客戶得到優惠),兩者的方向不同;本案引用這個案例的目的,是說明監理機關確實會檢視「挽留手段的公平性」,不是指本案的挽留動作違反了這項規定。 |
| **公平性工具使用現況的從業者研究(Deng 等人,2022;訪談 11 位、問卷 31 位從業者)** | 這是一項針對機器學習從業者的訪談與問卷研究。研究發現:多數從業者遇到性別變數時直接刪除,沒有進一步檢查其他特徵是否為代理變數;公平性工具的名詞不直觀,難以向非工程人員說明;是否做公平性測試幾乎完全取決於利害關係人有沒有要求(92% 的受訪者如此回答);只有 16% 的受訪者所屬組織有明確的公平性流程 | 本模組不採用「把性別特徵刪掉就算處理完畢」的做法(1.3);以「對業務與法遵單位可以說明」作為所有輸出表的設計標準,每一張表的每一欄都在判讀 Cell 開頭定義 | 不引入公平性套件(理由見第 6 節);本模組在開工時就把公平性檢視列為 M7 的工作項目(Cell M7-1、決策日誌 [M7-0]),不是等法遵或業務單位要求才做 |
| **台灣的個人資料保護法與金管會「金融服務業公平待客原則」** | 個人資料保護法第 19 條:非公務機關(例如銀行)蒐集個人資料須有特定目的,並符合法定情形;第 20 條第 1 項:原則上只能在蒐集的特定目的範圍內利用個人資料;第 20 條第 2 項與第 3 項:利用個人資料進行行銷時,當事人表示拒絕後即應停止,而且首次行銷時應提供當事人免費的拒絕方式。金管會「金融服務業公平待客原則」共十項,與本案相關的是「商品或服務適合度原則」、「告知與揭露原則」、「友善服務原則」三項 | `docs/07_compliance_fairness.md` 第 2 節「個資與目的拘束」(目的拘束,purpose limitation,指個人資料只能在蒐集時所定的特定目的範圍內利用)依個人資料保護法的這三個條項撰寫:既有客戶的挽留行銷屬於信用卡業務蒐集目的範圍內的利用;推播必須提供拒絕機制;M6 對照組的「不得接觸」名單與分組名單檔案必須有存取權限控制(原因:這兩份檔案標示了哪些客戶被刻意不接觸,業務單位若看到並接觸了這些客戶,M6 的實驗就失效;做法:檔案只開放給實驗負責人與資料團隊,業務單位只收到要接觸的名單);交付給業務單位的名單不含 is_churned 欄位(這是資料集的事後流失標籤;真實業務中名單產生當下流失還沒有發生,業務單位執行挽留動作也不需要這個欄位,交付只需客戶識別碼與動作類型)與其他不必要的欄位(資料最小化,data minimisation,指「只蒐集與交付達成目的所必需的欄位」)。加碼回饋「只對實際刷卡達到門檻的客戶支付」符合商品或服務適合度原則 | 本案使用的是公開資料集,沒有真實客戶的同意紀錄與蒐集目的文件,所以法遵的部分只能寫成「若為真實業務,名單上線前須確認的事項」,本模組不宣稱已完成法遵審查 |

**來源**:<br/>
1.NYDFS 新聞稿與調查報告:https://www.dfs.ny.gov/reports_and_publications/press_releases/pr202103231 ;https://s3.documentcloud.org/documents/20521283/rpt_202103_apple_card_investigation.pdf <br/>
2.公平授信模型偏差測試流程(廠商整理的說明文章,流程描述與業界通行做法一致):https://www.openlayer.com/blog/post/credit-model-bias-testing-fair-lending-compliance <br/>
3.Google Fairness Indicators:https://blog.tensorflow.org/2019/12/fairness-indicators-fair-ML-systems.html <br/>
4.金管會「金融業運用人工智慧(AI)指引」摘要(理律法律事務所):https://www.leeandli.com/TW/Newsletters/7287.htm <br/>
5.FCA 新聞稿:https://www.fca.org.uk/news/press-releases/fca-confirms-measures-protect-customers-loyalty-penalty-home-motor-insurance-markets <br/>
6.Deng et al., 2022, Exploring How Machine Learning Practitioners (Try To) Use Fairness Toolkits:https://arxiv.org/abs/2205.06922 <br/>

**這張表怎麼使用**:每一列的第三欄與第四欄,就是本模組的「方法選擇紀錄」:本模組採用了什麼做法、為什麼採用、哪裡刻意與業界做法不同。法遵或風險控管人員審核本模組時,可以直接用這張表核對「每一個檢查步驟的依據是什麼」。<br/>

In [1]:
# Cell M7-4(code)|第 3 節:環境設定、讀取客戶層級表、定義年齡組與三個名單層級

import sys
import pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.proportion import confint_proportions_2indep
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from src.db import get_engine

warnings.filterwarnings("ignore")
engine = get_engine()

# 本模組共用常數(每一個常數的用途與依據寫在行尾)
BASELINE_CHURN_PCT = 16.07        # 全體客戶的基準流失率(%),M1 起沿用;下方以 assert 核對讀入的資料是否相同
RANDOM_STATE = 42                 # 檢查 B 的主切分(與 M4 相同的 random_state),使含性別版的數字可與 notebook 04 直接對照
TEST_SIZE = 0.30                  # 檢查 B 的測試集比例(30%),與 M4 相同
SEEDS = [0, 1, 7, 42, 99]         # 檢查 B 的五個隨機切分(random_state),與 M4 Cell M4-17 的穩定性檢查相同,使含性別版的五組數字可逐一對照 notebook 04 第 7 節
AIR_REFERENCE = 0.8               # 不利影響比率(AIR)的提醒線,來自五分之四法則;本案只作「需要在判讀中解釋」的提醒,不作為法遵門檻(理由見第 1 節 1.4)
MIN_GROUP_N = 100                 # 小樣本慣例:某一個族群在某一個名單層級內少於 100 人時,不對這一格單獨下結論(決策日誌 [M2-10])
AGE_BINS = [25, 35, 45, 55, 73]   # 年齡組邊界。pd.cut 預設左開右閉(right=True):區間 (25, 35] 代表「大於 25、小於等於 35」,小括號表示不含該端點、中括號表示包含;年齡是整數,所以四個區間分別對應 26–35、36–45、46–55、56–73 歲。首尾取 25 與 73,是因為資料集的客戶年齡最小 26、最大 73,下方 assert 會確認沒有人落在區間之外
AGE_LABELS = ["26-35", "36-45", "46-55", "56-73"]


# 客戶層級表的查詢說明:
# (1) 以下 SQL 指令的 flags、scored、zoned、m6_base 四個 CTE,沿用 sql/05_experiment_population.sql [E0] 的 m6_base 定義(四個行為旗標、命中數、價值層、策略區、額外回饋旗標),再以 clientnum 與 clean.customers 全表 JOIN,取回模型 B 需要、但 m6_base 沒有的「人口屬性欄位(education_level、marital_status、income_category、dependent_count)」。
# (2) 四個旗標的門檻、NTILE 的排序鍵(total_trans_amt, clientnum)、策略區的歸類規則都與 sql/04、sql/05 完全相同,所以四個策略區的人數必須等於 426 / 2,050 / 2,949 / 4,702;下方以 assert 核對,不相等就中止執行。
# (3) 本模組不另開 SQL 檔;若要以純 SQL 重現檢查 A 的名單組成,對 m6_base 以 gender(或年齡組)與 zone 分組計數即可;「不利影響比率、信賴區間、檢查 B 的模型重新訓練」在 Python 端完成。
# (4) 最後一行 ORDER BY clientnum 是為了固定列的順序,原因:檢查 B 的 train_test_split 依列的位置切分,若列的順序不固定,同一個 random_state 在不同環境會切出不同的訓練集與測試集(決策日誌 [M4-1] 的問題紀錄)。
M7_SQL = """
WITH flags AS (
    SELECT clientnum, is_churned,
           (total_trans_ct <= 61)::int           AS low_activity,
           (avg_utilization_ratio = 0)::int       AS zero_util,
           (total_relationship_count <= 2)::int   AS shallow_rel,
           (contacts_count_12_mon >= 4)::int      AS high_contact,
           total_trans_amt, total_amt_chng_q4_q1
    FROM clean.customers
),
scored AS (
    SELECT *,
           low_activity + zero_util + shallow_rel + high_contact AS hit_count,
           NTILE(3) OVER (ORDER BY total_trans_amt, clientnum)   AS value_tier
    FROM flags
),
zoned AS (
    SELECT *,
           CASE WHEN hit_count >= 3 THEN '3+' ELSE hit_count::text END AS risk_tier,
           CASE WHEN hit_count >= 2 AND value_tier = 3 THEN '1_rescue'
                WHEN hit_count >= 2                    THEN '2_low_cost'
                WHEN value_tier = 3                    THEN '3_deepen'
                ELSE                                        '4_monitor' END AS zone
    FROM scored
),
m6_base AS (
    SELECT clientnum, value_tier, risk_tier, zone, hit_count,
           low_activity, zero_util, shallow_rel, high_contact,
           (zone = '2_low_cost' AND total_amt_chng_q4_q1 < 0.50)::int AS extra_reward
    FROM zoned
)
SELECT c.*,
       b.value_tier, b.risk_tier, b.zone, b.hit_count,
       b.low_activity, b.zero_util, b.shallow_rel, b.high_contact, b.extra_reward
FROM clean.customers AS c
JOIN m6_base AS b USING (clientnum)
ORDER BY c.clientnum
"""
print(M7_SQL)   # 印出 SQL 查詢指令全文,讀者不必另開 SQL 檔即可核對查詢的定義

df = pd.read_sql(M7_SQL, engine)
df["is_churned"] = df["is_churned"].astype(int)

# 核對:總人數(10,127)、基準流失率(16.07%)、四個策略區人數必須與 M5、M6 完全相同,否則後續所有的族群比較都失去對照基礎
assert len(df) == 10_127
assert round(df["is_churned"].mean() * 100, 2) == BASELINE_CHURN_PCT
zone_n = df["zone"].value_counts().sort_index()
assert zone_n.tolist() == [426, 2050, 2949, 4702], f"策略區人數與 M5 不符:{zone_n.tolist()}"

# 年齡組:用固定年段切分(pd.cut),不用人數等份切分(SQL 的 NTILE 或 pandas 的 pd.qcut;本案的價值層就是用 NTILE 切成人數相等的三層)。理由與決策日誌 [M2-11] 相同:年段是業務單位與法規使用的語言(例如「55 歲以上客戶」),分位編號(第一分位、第二分位)對法遵讀者沒有意義。
# 四個年齡組在全體層級的人數為 919 / 3,742 / 4,135 / 1,331,沒有小樣本問題;年齡組與人數少的名單(例如 308 人的加碼回饋名單)交叉之後的小格子,在 Cell M7-5 另以 small_sample 欄位標示。
df["age_band"] = pd.cut(df["customer_age"], bins=AGE_BINS, labels=AGE_LABELS)
assert df["age_band"].notna().all(), "有客戶年齡落在年齡組邊界之外"

# 三個名單層級與一個對照名單,全部沿用 notebook 05 第 8 節名單總表的定義;每一欄都是 0 或 1, 1 代表該客戶收到這個動作
df["list_retention"] = (df["hit_count"] >= 2).astype(int)              # 挽留名單:命中數 ≥ 2 的客戶(2,476 人)= 搶救區 426 人 + 低成本區 2,050 人
df["list_manual"]    = (df["zone"] == "1_rescue").astype(int)          # 人工聯繫名單:搶救區(426 人),由客戶經理逐一聯繫,是每人成本最高的挽留動作
df["list_reward"]    = df["extra_reward"].astype(int)                  # 加碼回饋名單:低成本區內消費變化比 < 0.50 的客戶(308 人),收到加碼版回饋
df["list_contact5"]  = (df["contacts_count_12_mon"] >= 5).astype(int)  # 對照名單:近 12 個月客服聯繫 ≥ 5 次、由客戶經理人工回訪的客戶(230 人);「圈選條件只有一個行為欄位」,用來檢視「只用一個行為條件圈出的名單,性別與年齡組成是否也與全體不同」(代理變數問題的對照,見 Cell M7-1)
assert df["list_retention"].sum() == 2476 and df["list_manual"].sum() == 426 and df["list_reward"].sum() == 308

LIST_LEVELS = {"list_retention": "挽留名單(命中 ≥2)",
               "list_manual":    "人工聯繫(搶救區)",
               "list_reward":    "加碼回饋(低成本區內消費變化比 < 0.50)",
               "list_contact5":  "對照:聯繫 ≥5 次人工回訪"}

print(f"載入 {len(df):,} 列 × {df.shape[1]} 欄 | 基準流失率 {df['is_churned'].mean():.2%}")
print("策略區人數:", zone_n.to_dict())
print("名單層級人數:", {LIST_LEVELS[k]: int(df[k].sum()) for k in LIST_LEVELS})
print("性別人數:", df["gender"].value_counts().to_dict())
print("年齡組人數:", df["age_band"].value_counts().sort_index().to_dict())


WITH flags AS (
    SELECT clientnum, is_churned,
           (total_trans_ct <= 61)::int           AS low_activity,
           (avg_utilization_ratio = 0)::int       AS zero_util,
           (total_relationship_count <= 2)::int   AS shallow_rel,
           (contacts_count_12_mon >= 4)::int      AS high_contact,
           total_trans_amt, total_amt_chng_q4_q1
    FROM clean.customers
),
scored AS (
    SELECT *,
           low_activity + zero_util + shallow_rel + high_contact AS hit_count,
           NTILE(3) OVER (ORDER BY total_trans_amt, clientnum)   AS value_tier
    FROM flags
),
zoned AS (
    SELECT *,
           CASE WHEN hit_count >= 3 THEN '3+' ELSE hit_count::text END AS risk_tier,
           CASE WHEN hit_count >= 2 AND value_tier = 3 THEN '1_rescue'
                WHEN hit_count >= 2                    THEN '2_low_cost'
                WHEN value_tier = 3                    THEN '3_deepen'
                ELSE                                        '4_monitor' END AS zon

In [2]:
# Cell M7-5(code)|第 4 節:檢查 A:名單的「族群組成(各族群的選取率、不利影響比率)」與「族群內品質(各族群在名單內的精確率與漏報率,以及每一個族群相對基準族群的精確率差距、漏報率差距各自的 95% 信賴區間)」

# ---------- 共用函式 ----------
def zone_composition(d: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """表 A-1a / A-1b:每一個策略區內各族群的占比(列百分比,每一列加總為 100),最後一列 all 是全體客戶的族群占比,供對照「某一個策略區的族群比例是否偏離全體」:把該區那一列的族群占比與最後一列 all 相減,差距越大,代表該區的客戶組成與全體越不同(例如搶救區女性 46.9%、全體 52.9%,差 6.0 個百分點,代表女性進入搶救區的比例低於她們在全體客戶的占比)。"""
    ct = pd.crosstab(d["zone"], d[group_col])
    ct.loc["all"] = ct.sum()                                   # 加一列「全體客戶的計數」
    pct = (100 * ct.div(ct.sum(axis=1), axis=0)).round(1)      # # ct 每一格是「某策略區內某族群的人數」;這一行把每一格除以該列(該策略區)的總人數再乘 100,得到「該策略區內各族群的占比」,每一列加總為 100
    pct.insert(0, "n", ct.sum(axis=1))
    return pct


def selection_table(d: pd.DataFrame, group_col: str, list_col: str) -> pd.DataFrame:
    """表 A-2 / A-3:某一個名單層級依族群計算的「選取率」與「不利影響比率」。
    n_group            = 該族群的總人數;
    selected           = 該族群中進入這份名單的人數;
    churners           = 該族群的流失人數;
    selection_rate_pct = selected ÷ n_group × 100,即選取率(該族群中每 100 人有幾人收到這個動作);
    AIR                = 該族群的選取率 ÷ 選取率最高族群的選取率(選取率最高的族群本身為 1.000)(AIR = 不利影響比率);
    churn_pct_group    = churners ÷ n_group × 100,即「該族群的整體流失率」,供對照「選取率的高低是否與該族群的真實流失率同向」。"""
    g = d.groupby(group_col, observed=True).agg(n_group=("clientnum", "size"),
                                                selected=(list_col, "sum"),
                                                churners=("is_churned", "sum"))
    g["selection_rate_pct"] = (100 * g["selected"] / g["n_group"]).round(2)
    g["AIR"] = (g["selection_rate_pct"] / g["selection_rate_pct"].max()).round(3)
    g["churn_pct_group"] = (100 * g["churners"] / g["n_group"]).round(2)
    g = g.reset_index().rename(columns={group_col: "group"})   # 族群欄統一命名為 group:性別分組時這一欄叫 gender、年齡分組時叫 age_band,改成同名之後,下方 flag 那一行才能用 pd.concat 把兩張表疊成一張,一次篩出所有 AIR 低於 0.8 的組合
    g.insert(0, "dim", group_col)                              # dim 標示這一列是依性別(gender)還是依年齡組(age_band)分組
    g.insert(1, "list_level", LIST_LEVELS[list_col])
    return g


def list_quality(d: pd.DataFrame, group_col: str, list_col: str, ref_group, eligible=None, pop_label="全體") -> pd.DataFrame:
    """表 A-4 至 A-7:名單內各族群的精確率與漏報率,以及「每一個族群相對基準族群的差距」與「差距的 95% 信賴區間」。
    n_pop            = 母體中該族群的人數。母體預設為全體客戶;若傳入 eligible,母體就是該動作的候選範圍(見下方說明);
    in_list          = 名單內該族群的人數;
    churners_in_list = 名單內該族群的流失人數;
    churners_total   = 母體中該族群的全部流失人數;missed = churners_total − churners_in_list;
    sel_in_pop_pct   = in_list ÷ n_pop × 100,即母體內的選取率;
    precision_pct    = churners_in_list ÷ in_list × 100,即「精確率(圈進名單的人裡有多少真的流失)」;
    fnr_pct          = missed ÷ churners_total × 100,即「漏報率(真的會流失的人裡有多少沒有進名單;偽陰性率)」;
    diff_prec_pp、diff_fnr_pp = 該族群的數值 − 基準族群的數值(精確率數值、漏報率數值),單位為百分點;
    ci_prec_*、ci_fnr_* = 這個精確率差距或漏報率差距的 95% 信賴區間(兩個獨立比例的差,常態近似,與 M3 的兩比例方法一致)(信賴區間以常態近似計算,也就是在樣本數夠大時,用常態分布近似兩個比例差距的抽樣分布,再算出 95% 區間,這是標準的 Wald 法,與 M3 的兩比例方法一致;人數少於 100 人的格子近似較差,所以另以 small_sample 標示);
    small_sample     = True 代表名單內該族群不足 MIN_GROUP_N(100)人,依決策日誌 [M2-10] 不單獨下結論。
    eligible 參數     : 布林遮罩,用來「把母體限制在該動作的候選範圍」。人工聯繫只在高價值層(3,375 人)內決定誰收到、加碼回饋只在低成本區(2,050 人)內決定誰收到,所以「機會平等」的正確問法是「在同一個候選範圍內,各族群真的會流失的客戶,有多少比例沒有被規則選到,這個比例是否相近」;
    以候選範圍為母體的根本理由是問題定義:漏報率要回答「規則有資格選、卻沒有選到的流失客戶有多少」,不在候選範圍內的客戶規則本來就不會選,把他們算成「被漏掉」,等於把規則沒有做的決定算成規則的錯,算出來的不是原本要查的比率。所以即使各族群在候選範圍內的人數比例完全相同,以全體為母體仍然是錯的算法;在本案,這個錯誤的具體後果是把「各族群在高價值層、低成本區的人數比例不同」混進漏報率,放大了兩性的差距(推算見第 4 節判讀修正一)。

    基準族群           :性別取男性(沿用 M3 T3 與 M4 模型 B「女性相對男性」的比較方向);年齡取人數最多的一組(由下方 AGE_REF 自動選出)。"""
    if eligible is not None:
        d = d[eligible]
    rows = []
    for grp, sub in d.groupby(group_col, observed=True):
        in_list = sub[sub[list_col] == 1]
        rows.append({"group": grp, "n_group": len(sub), "in_list": len(in_list),
                     "churners_in_list": int(in_list["is_churned"].sum()),
                     "churners_total": int(sub["is_churned"].sum())})
    q = pd.DataFrame(rows).set_index("group").rename(columns={"n_group": "n_pop"})
    q["missed"] = q["churners_total"] - q["churners_in_list"]
    q["sel_in_pop_pct"] = (100 * q["in_list"] / q["n_pop"]).round(2)
    q["precision_pct"] = (100 * q["churners_in_list"] / q["in_list"]).round(2)
    q["fnr_pct"] = (100 * q["missed"] / q["churners_total"]).round(2)

    ref = q.loc[ref_group]
    for metric, num, den in [("prec", "churners_in_list", "in_list"), ("fnr", "missed", "churners_total")]:
        diffs, lows, highs = [], [], []
        for grp, r in q.iterrows():
            if grp == ref_group:
                diffs.append(0.0)
                lows.append(np.nan)
                highs.append(np.nan)
                continue
            # confint_proportions_2indep(count1, nobs1, count2, nobs2):這個函式的四個參數依序是(第一組的成功數、第一組的樣本數、第二組的成功數、第二組的樣本數),回傳「第一組比例 − 第二組比例」的信賴區間;程式把該族群放第一組、基準族群放第二組,所以區間是該族群減基準族群
            # 此處,「比例 1 = 該族群」、「比例 2 = 基準族群」,所以區間是「該族群 − 基準族群」的數值差距信賴區間 
            # nobs1 = 第一個獨立樣本的總觀測次數/樣本總數 (Number of Observations)
            lo, hi = confint_proportions_2indep(r[num], r[den], ref[num], ref[den], method="wald")
            diffs.append(100 * (r[num] / r[den] - ref[num] / ref[den]))
            lows.append(100 * lo)
            highs.append(100 * hi)
        q[f"diff_{metric}_pp"] = np.round(diffs, 2)
        q[f"ci_{metric}_low"] = np.round(lows, 2)
        q[f"ci_{metric}_high"] = np.round(highs, 2)
    q["small_sample"] = q["in_list"] < MIN_GROUP_N      # True = 名單內該族群不足 100 人,依決策日誌 [M2-10] 不單獨下結論
    q.insert(0, "population", pop_label)
    q.insert(0, "list_level", LIST_LEVELS[list_col])
    return q.reset_index()


# ---------- 表 A-1a / A-1b:四個策略區 × 性別、年齡組的組成 ----------
print("表 A-1a:策略區 × 性別(列百分比;最後一列 all 為全體)")
display(zone_composition(df, "gender"))
print("表 A-1b:策略區 × 年齡組(列百分比)")
display(zone_composition(df, "age_band"))

# ---------- 表 A-2 / A-3:三個名單層級與對照名單,依性別、年齡組的選取率與不利影響比率(AIR) ----------
sel_gender = pd.concat([selection_table(df, "gender", k) for k in LIST_LEVELS], ignore_index=True)
sel_age    = pd.concat([selection_table(df, "age_band", k) for k in LIST_LEVELS], ignore_index=True)
print("表 A-2:各名單層級依性別的選取率與不利影響比率(AIR;選取率最高的族群 = 1.000;提醒線 0.8:AIR 低於 0.8 的族群需在判讀中解釋)") 
display(sel_gender)
print("表 A-3:各名單層級依年齡組的選取率與不利影響比率")
display(sel_age)

# 把 AIR 低於提醒線 0.8 的「名單層級 × 族群」組合一次列出,供判讀引用;被列出的組合不代表「直接被判定為有問題」,只代表「需要搭配表 A-4 至 A-7 的族群內品質」一起解釋這些組合是否可能有問題
flag = pd.concat([sel_gender, sel_age], ignore_index=True)
flag = flag[flag["AIR"] < AIR_REFERENCE][["dim", "list_level", "group", "selection_rate_pct", "AIR", "churn_pct_group"]]
print(f"AIR 低於 {AIR_REFERENCE} 的組合(需要在判讀中解釋):")
display(flag if len(flag) else pd.DataFrame({"note": ["無任何族群低於提醒線"]}))

# ---------- 表 A-4 / A-5:三個名單層級內的族群品質(精確率、漏報率),母體 = 全體客戶 ----------
AGE_REF = df["age_band"].value_counts().idxmax()    # 年齡的基準族群:四個年齡組中人數最多的一組(實際資料為 46-55 歲組,4,135 人)。「取人數最多的一組」這條規則是分析者決定的(理由見第 1 節 1.8:基準組人數越多,差距的信賴區間越窄);程式只負責依這條規則找出那一組,並在下一行印出,供讀者核對
print(f"年齡的基準族群:{AGE_REF}(人數最多的一組);性別的基準族群:M")
qual_gender = pd.concat([list_quality(df, "gender", k, "M") for k in ["list_retention", "list_manual", "list_reward"]], ignore_index=True)
qual_age    = pd.concat([list_quality(df, "age_band", k, AGE_REF) for k in ["list_retention", "list_manual", "list_reward"]], ignore_index=True)
print("表 A-4:名單內各性別的精確率、漏報率,與相對基準族群(男性)的差距及 95% 信賴區間(百分點)")
display(qual_gender)
print(f"表 A-5:名單內各年齡組的精確率、漏報率,與相對基準族群({AGE_REF} 歲組)的差距及 95% 信賴區間(百分點)")
display(qual_age)

# ---------- 表 A-6 / A-7:以「該動作的候選範圍」為母體,重算人工聯繫與加碼回饋的族群內品質 ----------
# 人工聯繫的候選範圍是高價值層(value_tier = 3,3,375 人):規則只從高價值層中選出命中數 ≥ 2 的 426 人(搶救區)收到人工聯繫;命中數 ≤ 1 的 2,949 人(深化區)不會收到
# 加碼回饋的候選範圍是低成本區(2,050 人):規則只從低成本區中選出消費變化比 < 0.50 的 308 人收到加碼回饋。
# 以全體客戶為母體時(表 A-4 / A-5)的錯誤效果:不在候選範圍內的流失客戶全部被算成「被漏掉」,人工聯繫的漏報率被墊高到 84% 至 95%;各族群不在候選範圍的比例不同(女性流失客戶只有 11.4% 在高價值層、男性 28.8%),被墊高的幅度也不同,兩性差距從層內的 4.73 個百分點被放大成 10.08 個百分點,直接判讀會把組成差異誤判成規則對女性不利。
# 表 A-4 / A-5 仍然保留,原因:挽留名單的候選範圍就是全體客戶,它的漏報率只在那兩張表;精確率不受母體影響,三個名單的精確率都在那兩張表;第一版的數字並列,讀者可以看到修正前後的差別(第 4 節判讀修正一)。
# 本表把母體限制在候選範圍內,回答的才是「在同一個候選範圍內,各族群真的會流失的客戶被規則選到的比例是否相近」,這是第 1 節 1.6 機會平等的正確比較範圍。
ELIGIBLE = {"list_manual": (df["value_tier"] == 3, "高價值層(value_tier = 3)"),
            "list_reward": (df["zone"] == "2_low_cost", "低成本區(zone = 2_low_cost)")}
qual_gender_pop = pd.concat([list_quality(df, "gender", k, "M", eligible=m, pop_label=lab) for k, (m, lab) in ELIGIBLE.items()], ignore_index=True)  # pop_label=lab 以關鍵字參數傳入,會覆蓋函式的預設值「全體」;輸出表 A-6 的 population 欄會顯示 lab 的內容,可據此核對
qual_age_pop    = pd.concat([list_quality(df, "age_band", k, AGE_REF, eligible=m, pop_label=lab) for k, (m, lab) in ELIGIBLE.items()], ignore_index=True)
print("表 A-6:人工聯繫(母體 = 高價值層)與加碼回饋(母體 = 低成本區)依性別的母體內選取率、精確率、漏報率")
display(qual_gender_pop)
print("表 A-7:人工聯繫(母體 = 高價值層)與加碼回饋(母體 = 低成本區)依年齡組的母體內選取率、精確率、漏報率") 
display(qual_age_pop)

表 A-1a:策略區 × 性別(列百分比;最後一列 all 為全體)


gender,n,F,M
zone,,,
1_rescue,426,46.9,53.1
2_low_cost,2050,51.2,48.8
3_deepen,2949,59.7,40.3
4_monitor,4702,49.9,50.1
all,10127,52.9,47.1


表 A-1b:策略區 × 年齡組(列百分比)


age_band,n,26-35,36-45,46-55,56-73
zone,,,,,
1_rescue,426,10.6,38.7,40.6,10.1
2_low_cost,2050,10.5,35.8,38.2,15.4
3_deepen,2949,7.2,38.0,43.6,11.2
4_monitor,4702,9.5,36.6,40.2,13.6
all,10127,9.1,37.0,40.8,13.1


表 A-2:各名單層級依性別的選取率與不利影響比率(AIR;選取率最高的族群 = 1.000;提醒線 0.8:AIR 低於 0.8 的族群需在判讀中解釋)


,dim,list_level,group,n_group,selected,churners,selection_rate_pct,AIR,churn_pct_group
0,gender,挽留名單(命中 ≥2),F,5358,1249,930,23.31,0.906,17.36
1,gender,挽留名單(命中 ≥2),M,4769,1227,697,25.73,1.000,14.62
2,gender,人工聯繫(搶救區),F,5358,200,930,3.73,0.787,17.36
3,gender,人工聯繫(搶救區),M,4769,226,697,4.74,1.000,14.62
4,gender,加碼回饋(低成本區內消費變化比 < 0.50),F,5358,158,930,2.95,0.937,17.36
5,gender,加碼回饋(低成本區內消費變化比 < 0.50),M,4769,150,697,3.15,1.000,14.62
6,gender,對照:聯繫 ≥5 次人工回訪,F,5358,96,930,1.79,0.637,17.36
7,gender,對照:聯繫 ≥5 次人工回訪,M,4769,134,697,2.81,1.000,14.62


... (8 outputs left out)

表 A-6:人工聯繫(母體 = 高價值層)與加碼回饋(母體 = 低成本區)依性別的母體內選取率、精確率、漏報率


,group,list_level,population,n_pop,in_list,churners_in_list,churners_total,missed,sel_in_pop_pct,precision_pct,fnr_pct,diff_prec_pp,ci_prec_low,ci_prec_high,diff_fnr_pp,ci_fnr_low,ci_fnr_high,small_sample
0,F,人工聯繫(搶救區),高價值層(value_tier = 3),1962,200,53,106,53,10.19,26.50,50.00,-22.17,-31.11,-13.24,4.73,-7.02,16.47,False
1,M,人工聯繫(搶救區),高價值層(value_tier = 3),1413,226,110,201,91,15.99,48.67,45.27,0.00,NaN,NaN,0.00,NaN,NaN,False
2,F,加碼回饋(低成本區內消費變化比 < 0.50),低成本區(zone = 2_low_cost),1049,158,124,612,488,15.06,78.48,79.74,12.48,2.55,22.41,4.74,-0.58,10.06,False
3,M,加碼回饋(低成本區內消費變化比 < 0.50),低成本區(zone = 2_low_cost),1001,150,99,396,297,14.99,66.00,75.00,0.00,NaN,NaN,0.00,NaN,NaN,False


表 A-7:人工聯繫(母體 = 高價值層)與加碼回饋(母體 = 低成本區)依年齡組的母體內選取率、精確率、漏報率


,group,list_level,population,n_pop,in_list,churners_in_list,churners_total,missed,sel_in_pop_pct,precision_pct,fnr_pct,diff_prec_pp,ci_prec_low,ci_prec_high,diff_fnr_pp,ci_fnr_low,ci_fnr_high,small_sample
0,26-35,人工聯繫(搶救區),高價值層(value_tier = 3),256,45,19,43,24,17.58,42.22,55.81,6.38,-9.72,22.49,6.22,-11.05,23.49,True
1,36-45,人工聯繫(搶救區),高價值層(value_tier = 3),1285,165,63,106,43,12.84,38.18,40.57,2.34,-7.95,12.64,-9.03,-21.89,3.84,False
2,46-55,人工聯繫(搶救區),高價值層(value_tier = 3),1460,173,62,123,61,11.85,35.84,49.59,0.00,NaN,NaN,0.00,NaN,NaN,False
3,56-73,人工聯繫(搶救區),高價值層(value_tier = 3),374,43,19,35,16,11.50,44.19,45.71,8.35,-8.13,24.82,-3.88,-22.60,14.84,True
4,26-35,加碼回饋(低成本區內消費變化比 < 0.50),低成本區(zone = 2_low_cost),216,16,14,71,57,7.41,87.50,80.28,12.09,-5.83,30.01,2.13,-7.93,12.20,True
5,36-45,加碼回饋(低成本區內消費變化比 < 0.50),低成本區(zone = 2_low_cost),734,114,92,382,290,15.53,80.70,75.92,5.29,-5.24,15.82,-2.23,-8.06,3.60,False
6,46-55,加碼回饋(低成本區內消費變化比 < 0.50),低成本區(zone = 2_low_cost),784,122,92,421,329,15.56,75.41,78.15,0.00,NaN,NaN,0.00,NaN,NaN,False
7,56-73,加碼回饋(低成本區內消費變化比 < 0.50),低成本區(zone = 2_low_cost),316,56,25,134,109,17.72,44.64,81.34,-30.77,-45.86,-15.67,3.20,-4.49,10.88,True


Cell M7-6(markdown)|第 4 節判讀:檢查 A

**第 4 節判讀|名單的族群組成與名單在各族群內的品質(待驗證事項 #29)**

**本節結論**:<br/>
三個名單層級(挽留名單、人工聯繫名單、加碼回饋名單)在性別與年齡上都不構成差別性影響,現行名單規則可以照原樣使用,不需要修改。<br/>
依據是:<br/>
在各動作的候選範圍內,每一個族群真的會流失的客戶被名單漏掉的比例(漏報率)與基準族群相近(漏報率差距的 95% 信賴區間都包含 0,符合 1.5 判讀規則 (a));各族群選取率的差異,來自「同樣命中旗標的客戶中,各族群實際流失的比例不同」,也就是規則在某些族群身上誤圈較多,這是成本問題,不是哪一個族群被少給挽留資源。<br/>
本節另外有兩個發現,後面的章節會用到:<br/>
第一,對照名單證明行為欄位本身與性別、年齡有關聯(代理變數問題確實存在);<br/>
第二,名單的精確率在五個「名單層級 × 族群」的格子明顯低於基準族群(挽留名單內的男性、26–35 歲組與 56–73 歲組,人工聯繫名單內的女性,加碼回饋名單內的男性;挑選條件見意涵 (3)),這五格是日後修改名單規則時應優先檢視的對象。<br/>

以下先說明每一張表的每一欄代表什麼,再寫事前預期、實作後的修正紀錄,最後依本案通用的四段格式(實測 / 判定 / 對決策的意涵 / 待辦或限制)逐項判讀。<br/>

**各表欄位的意義**:<br/>
1.表 A-1a / A-1b:<br/>
每一列是一個策略區,數字是「該策略區內各族群的占比」(列百分比,每一列加總為 100);最後一列 all 是全體客戶的族群占比(女性占全體客戶的 52.9%;四個年齡組 26–35、36–45、46–55、56–73 歲占全體客戶的比例依序為 9.1%、37.0%、40.8%、13.1%),用來對照「某一個策略區的族群比例是否偏離全體」。<br/>

2.表 A-2 / A-3:<br/>
每一列是「一個名單層級 × 一個族群」。<br/>
(1) n_group 是該族群的總人數;(2) selected 是該族群中進入這份名單的人數;(3) churners 是該族群的流失人數;(4) selection_rate_pct 是「該族群的選取率」 = selected ÷ n_group × 100,回答「這個族群裡每 100 人有幾人收到這個動作」;(5) AIR 是「該族群的選取率 ÷ 選取率最高族群的選取率」(選取率最高的族群為 1.000,定義見 1.4),AIR 低於提醒線 0.8 的「名單層級 × 族群」組合另外列在表 A-3 之後;(6) churn_pct_group 是「該族群的整體流失率」= churners ÷ n_group × 100,用來與同一列的 selection_rate_pct 並排對照,判斷「該族群選取率的高低,是否與該族群的真實流失率同方向」。同方向指:選取率較高的族群,流失率也較高(或選取率較低的族群,流失率也較低);因為名單是依流失風險圈選的,同方向代表選取率的差異有可能只是反映真實風險的差異。方向相反則代表選取率的差異不是真實風險造成的,規則在選取率較高的那個族群身上誤圈較多,必須再看表 A-4 至 A-7 的精確率與漏報率才能判定。本案兩種情況都有:挽留名單的年齡組是方向相反的例子,流失率最低的 26–35 歲組(13.28%)選取率反而最高(28.40%);人工聯繫名單在高價值層內是同方向的例子,女性的選取率(10.19%)低於男性(15.99%),高價值層內女性的流失率(5.4%)也低於男性(14.2%)。<br/>

3.表 A-4 / A-5(母體 = 全體客戶):<br/>
每一列是「一個名單層級 × 一個族群」。<br/>
(1) n_pop 是該族群的總人數;(2) in_list 是名單內該族群的人數;(3) churners_in_list 是名單內該族群中實際流失的人數;(4) churners_total 是「該族群全部的流失人數」;(5) missed 是該族群沒有進入名單的流失人數;(6) precision_pct 是「該族群在名單內的精確率」 = churners_in_list ÷ in_list × 100(**圈進名單的該族群客戶裡,有多少比例真的流失**);(7) fnr_pct 是「該族群的漏報率」 = missed ÷ churners_total × 100(**該族群真的會流失的客戶裡,有多少比例沒有進名單**);(8) **diff_prec_pp 是「該族群的精確率減去基準族群的精確率」**,**diff_fnr_pp 是「該族群的漏報率減去基準族群的漏報率」**,單位都是百分點;(9) **ci_prec_low / ci_prec_high 是「精確率差距的 95% 信賴區間」**,**ci_fnr_low / ci_fnr_high 是「漏報率差距的 95% 信賴區間」**;(10) small_sample 為 True 代表「名單內該族群不足 100 人」,依決策日誌 [M2-10] 的小樣本慣例不單獨下結論。(11) 基準族群:性別為「男性」(沿用 M3 T3 與 M4 模型 B「女性相對男性」的比較方向),年齡為「人數最多的 46–55 歲組(4,135 人)」。<br/>

4.表 A-6 / A-7(母體 = 該動作的候選範圍):<br/>
欄位與表 A-4 / A-5 相同,但(1) **n_pop 與 churners_total 只計算「候選範圍內的客戶」**;(2) sel_in_pop_pct 是「候選範圍內該族群的選取率」。
「人工聯繫的候選範圍」是「高價值層(3,375 人)」,「加碼回饋的候選範圍」是「低成本區(2,050 人)」。<br/>
**判斷「是否構成差別性影響」要用這兩張表,不是用表 A-4 / A-5 判斷**,原因:以全體客戶為母體時(表 A-4),人工聯繫對每一個族群的漏報率都在 84% 至 95% 之間,因為人工聯繫本來就只給高價值層中命中 ≥ 2 的 426 人,低價值層與中價值層的流失客戶「本來就不在這個動作的候選範圍內」,不是被規則漏掉;把母體限制在「高價值層」,算出的漏報率才能回答「同樣是**高價值層的流失客戶**,高價值層中的各族群被人工聯繫覆蓋的比例是否相近」(覆蓋的比例以下稱 **「覆蓋率」,覆蓋率 = 1 − 漏報率**,即「該族群真的會流失的客戶中,被名單圈到的比例」),這才是第 1 節 1.6 所述 **機會平等的正確比較範圍**(兩種母體各自回答什麼問題,見 1.5 本案的設定第 2 點的對照表)。<br/>


**事前預期與實測的對照**:<br/>
本模組在執行 Cell M7-5 之前先寫下預期,目的是避免看到數字之後才事後找理由。事前預期有三點:(1) 依 M4 模型 B「其他條件相同時女性流失勝算比 2.12」與 M3 T3「女性流失率 17.36% 高於男性 14.62%」,預期「女性在挽留名單的選取率會略高於男性」;(2) M2 依年齡分組後各組的流失率相近(notebook 02 Cell 31:各年齡箱的流失率介於 13.72% 至 17.37%),預期「四個年齡組在挽留名單的選取率也相近」;(3) 對照名單(聯繫 ≥ 5 次人工回訪)只用一個條件圈選:近 12 個月客服聯繫次數 ≥ 5 次。本模組執行前的假設是「客戶聯繫客服的次數與性別、年齡無關」;若這個假設成立,從全體客戶中挑出聯繫 ≥ 5 次的人,挑到的女性比例與各年齡組比例,就應該與全體客戶相近(女性 52.9%;四個年齡組 9.1%、37.0%、40.8%、13.1%)。所以預期「對照名單的性別與年齡組成接近全體」。<br/>
實測與預期相反的有三處:(1) 女性在挽留名單的選取率(23.31%)反而低於男性(25.73%);(2) 年齡組的選取率與流失率方向相反,流失率最低的 26–35 歲組(13.28%)在挽留名單的選取率反而最高(28.40%);(3) 對照名單(聯繫 ≥ 5 次人工回訪)本身就有明顯的性別差異:女性客戶中聯繫 ≥ 5 次的比例是 1.79%(96 ÷ 5,358 人),男性客戶中是 2.81%(134 ÷ 4,769 人),男性高頻率聯繫客服的比例明顯較高(表 A-2)。<br/>
這三處是本節判讀的主軸。<br/>


**實作後的修正紀錄**(兩項;做法同決策日誌 [M6-2]、[M6-3] 的「實作後修正」):<br/>
**修正一:人工聯繫與加碼回饋的漏報率母體,由「全體客戶」改為「該動作的候選範圍」(新增表 A-6、A-7)。**<br/>
1.未修正前的結果:第一版的 Cell M7-5 只有表 A-4、A-5,三個名單層級的漏報率都以「該族群全部的流失客戶」為分母。第一次執行後,人工聯繫那一列出現了本模組最像「差別性影響」的數字:女性的選取率 3.73%、男性 4.74%(AIR 0.787,低於提醒線 0.8);女性流失客戶的漏報率 94.30%、男性 84.22%,兩性漏報率的差距 10.08 個百分點,95% 信賴區間 6.99 至 13.17,不包含 0。若直接依 1.5 的判讀規則 (c) 判讀,結論會是「規則系統性地漏掉女性流失客戶,須修改名單規則」。<br/>

2.查核思路:先確認「這個分母是否正確」。<br/>
先查分母是處理比率指標異常時的實務慣例:漏報率、選取率、流失率都是比率,比率的意義完全由分母決定,分母若包含了不應該包含的人,後面再精細的統計(信賴區間、檢定)都只是在精確地回答一個錯誤的問題,所以要先核對分子與分母是誰,再解釋數字。<br/>
人工聯繫只給搶救區(高價值層中命中 ≥ 2 的客戶);**一位低價值層的女性流失客戶沒有收到人工聯繫,是因為她不在這個動作的候選範圍內,不是被規則漏掉**;以全體流失客戶為分母,等於把「各族群在高價值層的人數比例不同」混進漏報率。接著從已有的表 A-1a 推算:高價值層 = 搶救區 426 人 + 深化區 2,949 人 = 3,375 人,其中女性約 58%(搶救區女性 46.9% × 426 人 + 深化區女性 59.7% × 2,949 人,約 1,960 人),高於全體的 52.9%;但搶救區內女性只占 46.9%,代表高價值層的女性客戶較少命中 ≥ 2 個旗標。<br/>
查到這裡只能判斷「以全體為母體算出的兩性漏報率差距,被族群組成的效應放大了」。用數字說明:以全體為母體時,女性的漏報率 = 沒有收到人工聯繫的女性流失客戶 ÷ 全部女性流失客戶(930 人);這 930 人中只有 106 人在高價值層,其餘 824 人不在高價值層,她們一定沒有收到人工聯繫,全部被算進分子;男性 697 位流失客戶中有 201 人在高價值層、496 人不在。<br/>
換成比例,女性流失客戶只有 11.4%(106 ÷ 930)在高價值層,男性有 28.8%(201 ÷ 697),所以女性的漏報率中,「本來就不在候選範圍、必然沒有收到」的那一塊比男性大得多,兩性的漏報率差距因此被放大。但這個推算仍然無法判斷「高價值層內兩性的漏報率是否還有差距」,因為以全體算出的 94.30% 與 84.22% 是兩塊混在一起的(層外必然沒收到的人,加上層內真的被規則漏掉的人),**不把層外那一塊拿掉,就看不出層內那一塊有沒有差距**;<br/>
要看層內,只能用高價值層內的 106 位女性與 201 位男性流失客戶重新計算,所以第二版新增以候選範圍為母體的表 A-6、A-7。<br/>

3.以「候選範圍為母體的表」重算的結果:高價值層內,女性流失客戶的漏報率 50.00%、男性 45.27%,兩性漏報率的差距 +4.73 個百分點,95% 信賴區間 −7.02 至 16.47,包含 0;高價值層內的流失率女性 5.4%(106 ÷ 1,962)、男性 14.2%(201 ÷ 1,413),女性較少命中旗標與女性確實較少流失是同一個方向。<br/>
**結論由「人工聯繫名單可能系統性地漏掉女性流失客戶,須修改規則」變成「人工聯繫名單對女性不構成差別性影響」**。改變的原因是:以正確的母體(高價值層)重算後,兩性漏報率差距 +4.73 個百分點的 95% 信賴區間 −7.02 至 16.47 包含 0,沒有達到 1.5 判讀規則 (c)「信賴區間不包含 0 且在 0 以上」的門檻,而是落在規則 (a)「看不出差異」;第一版算出的差距 10.08 個百分點(信賴區間 6.99 至 13.17)之所以達到門檻,是分母錯誤造成的。<br/>

這個修正的意義是:**公平性指標的分母必須是「該動作候選範圍內的客戶」**,分母定義錯誤會「把族群組成的差異誤判成規則的偏差」;這也是 1.5 本案的設定第 2 點具體實例:判斷「是否構成差別性影響」只用候選範圍那一欄,因為不在候選範圍內的流失客戶本來就不會收到這個動作,不是被規則漏掉。<br/>

**修正二:「對照名單的族群組成應接近全體」的事前假設被實測否定。**<br/>
第一版把「聯繫 ≥ 5 次人工回訪」當作「依客服紀錄觸發、與人口屬性無關」的對照,預期它的性別組成接近全體;實測為女性 1.79%、男性 2.81%(AIR 0.637),男性高頻率聯繫客服的比例明顯較高。這不是程式錯誤,而是假設錯誤:行為欄位本身就與性別有關聯。處置是改寫對照名單的用途說明(Cell M7-1、M7-4):它不是「與性別無關的基準」,而是「只用一個行為條件圈出的名單,族群組成也會與全體不同」的證據,支持 1.3「把性別刪掉不等於名單與性別無關」的論點,並成為第 5 節判定 (3)(不含性別版模型的名單女性占比仍然高於規則名單)的對照。<br/>
這兩項修正都沒有改變檢查的方法與判讀規則,改變的是「分母的定義」與「對照名單的用途」;第一版的數字保留在表 A-4、A-5,與修正後的表 A-6、A-7 並列,讀者可以自行比對兩種母體定義下結論的差別。<br/>



**實測**:<br/>
1.策略區的族群組成(表 A-1a、A-1b):女性在搶救區占 46.9%、低成本區 51.2%、深化區 59.7%、監測區 49.9%(全體 52.9%)。高價值層(搶救區 + 深化區,3,375 人)的女性占比約 58%,高於全體;但高價值層中「命中 ≥ 2 個旗標」的搶救區,女性只占 46.9%,代表高價值層的女性客戶命中較少旗標。年齡組在四個策略區的占比與全體占比的最大差距是 3.0 個百分點(56–73 歲組在搶救區占 10.1%,在全體占 13.1%),四區中每一個年齡組的占比與全體占比的差距,最大的是搶救區的 56–73 歲組(10.1% 對全體 13.1%,差 3.0 個百分點),其次是深化區的 46–55 歲組(43.6% 對全體 40.8%,差 2.8 個百分點),其餘十四格的差距都在 2.5 個百分點以內(表 A-1b)。<br/>

2.各族群的選取率與 AIR(表 A-2、A-3):<br/>
(1) 依性別:<br/>
挽留名單的選取率女性 23.31%、男性 25.73%,女性的 AIR 0.906;<br/>
人工聯繫名單的選取率女性 3.73%、男性 4.74%,女性的 AIR 0.787;<br/>
加碼回饋名單的選取率女性 2.95%、男性 3.15%,女性的 AIR 0.937;<br/>
對照名單(聯繫 ≥ 5 次)的選取率女性 1.79%、男性 2.81%,女性的 AIR 0.637。<br/>
(2)依年齡組:<br/>
挽留名單的選取率 26–35 歲組 28.40%(最高)、36–45 歲組 24.02%、46–55 歲組 23.14%(最低,AIR 0.815)、56–73 歲組 26.97%,而四組的流失率依序是 13.28%、16.19%、16.64%、15.85%,**流失率最低的 26–35 歲組選取率反而最高**;<br/>
人工聯繫名單的選取率依序為 26–35 歲組 4.90%(最高,AIR 1.000)、36–45 歲組 4.41%、46–55 歲組 4.18%、56–73 歲組 3.23%(最低,AIR 0.659),但人工聯繫名單內的 56–73 歲組只有 43 人;<br/>
加碼回饋名單的選取率依序為 26–35 歲組 1.74%(最低,AIR 0.413)、36–45 歲組 3.05%、46–55 歲組 2.95%、56–73 歲組 4.21%(最高,AIR 1.000),但加碼回饋名單內的 26–35 歲組只有 16 人。<br/>

(3)AIR 低於提醒線 0.8 的「名單層級 × 族群」組合共八個:<br/>
**人工聯繫 × 女性**、**人工聯繫 × 56–73 歲組**、**加碼回饋 × 26–35 歲組**、**加碼回饋 × 36–45 歲組**、**加碼回饋 × 46–55 歲組**,以及對照名單的三個組合(**女性**、**36–45 歲組**、**46–55 歲組**)。<br/>
八個組合中,名單內不足 100 人的有五個(人工聯繫 × 56–73 歲組 43 人、加碼回饋 × 26–35 歲組 16 人、對照名單 × 女性 96 人、對照名單 × 36–45 歲組 79 人、對照名單 × 46–55 歲組 83 人),依小樣本慣例不單獨下結論;
**人數足夠的三個組合(人工聯繫 × 女性、加碼回饋 × 36–45 歲組、加碼回饋 × 46–55 歲組)在下方判定 (4)、(6) 逐一解釋。**
**對照名單的組合在判定 (7) 說明。**<br/>

3.名單在各族群內的「精確率」與「漏報率」:<br/>
前提說明:<br/>
(a) 挽留名單的候選範圍就是全體客戶,所以挽留名單的精確率與漏報率都以正確的母體計算,是本模組判定的依據;<br/>
(b) 人工聯繫與加碼回饋以全體客戶為母體算出的漏報率,母體不正確(原因見修正一),本模組不用來判定,正確的漏報率在第 4 點;但精確率只看名單內的客戶,不受母體影響,所以這兩個名單的精確率可以使用,本點引用其中一格(加碼回饋名單內的 56–73 歲組);<br/>
(c) 以下信賴區間的說明:「不包含 0,差異確實存在」代表「在 95% 信心水準下可以認定該族群的指標確實高於或低於基準族群」;「包含 0,看不出差異」代表「真實差距可能是正、零或負,統計上無法排除沒有差距,依 1.8 只能說看不出差異,不能說沒有差異」。<br/>

母體 = 全體客戶(表 A-4、A-5):<br/>
挽留名單內,女性的精確率 53.24%、男性 41.24%,兩性精確率的差距 +12.00 個百分點,95% 信賴區間 8.10 至 15.91,不包含 0 (差異確實存在:女性的精確率確實高於男性)。<br/>
女性的漏報率 28.49%、男性 27.40%,兩性漏報率的差距 +1.09 個百分點,信賴區間 −3.31 至 5.49,包含 0。(看不出差異,定義見前提說明 (c))<br/>
依年齡組:<br/>
挽留名單內 26–35 歲組的精確率 34.48%,比基準族群 46–55 歲組的 50.47% 低 15.99 個百分點(精確率差距的信賴區間 −22.57 至 −9.41,不包含 0)(差異確實存在:26–35 歲組的精確率確實低於基準族群)。<br/>
56–73 歲組精確率 42.62%,比基準族群低 7.85 個百分點(信賴區間 −13.87 至 −1.83,不包含 0)(差異確實存在:56–73 歲組的精確率確實低於基準族群)。<br/>
36–45 歲組精確率 49.50%,比基準族群低 0.97 個百分點(信賴區間 −5.52 至 3.58,包含 0)(看不出差異,定義見前提說明 (c))。<br/>
四組的漏報率依序為 26.23%、26.57%、29.80%、27.49%,三組相對 46–55 歲組的漏報率差距,信賴區間都包含 0。(看不出差異,定義見前提說明 (c))。<br/>
加碼回饋名單內,56–73 歲組的精確率 44.64%,比基準族群 46–55 歲組的 75.41% 低 30.77 個百分點,信賴區間 −45.86 至 −15.67(不包含 0),但加碼回饋名單內的 56–73 歲組只有 56 人,不足 100 人,依小樣本慣例本模組對這一格只描述數字、不單獨下結論(表 A-5 的 small_sample 欄為 True 就是標示這件事)。<br/>

4.**名單在各族群內的「精確率」與「漏報率」:**<br/>
**母體 = 候選範圍(表 A-6、A-7):**<br/>
依性別:<br/>
高價值層 3,375 人中,女性 1,962 人(58.1%)、男性 1,413 人;<br/>
高價值層內的流失率女性 5.4%(106 人)、男性 14.2%(201 人),方向與全體相反(全體流失率女性 17.36% 高於男性 14.62%,高價值層內女性流失率低於男性流失率)。<br/>
人工聯繫在高價值層內的選取率女性 10.19%、男性 15.99%;對高價值層內流失客戶的漏報率女性 50.00%、男性 45.27%,兩性漏報率的差距 +4.73 個百分點,信賴區間 −7.02 至 16.47,包含 0。<br/>
人工聯繫名單內(也就是搶救區 426 人中)的精確率女性 26.50%、男性 48.67%,女性相對男性的精確率差距 −22.17 個百分點,信賴區間 −31.11 至 −13.24(不包含 0,差異確實存在:女性的精確率確實低於男性;精確率只看名單內的客戶,所以與表 A-4 的數字相同)。<br/>

低成本區 2,050 人中,女性 1,049 人、男性 1,001 人;<br/>
加碼回饋在低成本區內的選取率女性 15.06%、男性 14.99%;對低成本區內流失客戶的漏報率女性 79.74%、男性 75.00%,兩性漏報率的差距 +4.74 個百分點,信賴區間 −0.58 至 10.06,包含 0 但下限接近 0。(看不出差異,但整個區間幾乎都在 0 以上,代表資料偏向「女性的漏報率較高」的方向,只是證據還不夠強;再累積一些資料就有可能越過 1.5 判讀規則 (c) 的門檻,這是判定 (5) 把這一格列為優先監測項的原因)<br/>
加碼回饋名單內(也就是 308 人中)的精確率女性 78.48%、男性 66.00%,女性相對男性的精確率差距 +12.48 個百分點,信賴區間 2.55 至 22.41(不包含 0,差異確實存在:女性的精確率確實高於男性)。<br/>

依年齡組:<br/>
人工聯繫(候選範圍 = 高價值層):高價值層內四組的人數依序為 256、1,285、1,460、374 人,四組在高價值層內的選取率依序為 17.58%、12.84%、11.85%、11.50%;四組的漏報率依序為 55.81%、40.57%、49.59%、45.71%,其他三組相對 46–55 歲組的漏報率差距,信賴區間都包含 0(看不出差異;最大的差距是 36–45 歲組的 −9.03 個百分點,信賴區間 −21.89 至 3.84)。<br/>
加碼回饋(候選範圍 = 低成本區):低成本區內四組的人數依序為 216、734、784、316 人,四組在低成本區內的選取率依序為 7.41%、15.53%、15.56%、17.72%;四組的漏報率依序為 80.28%、75.92%、78.15%、81.34%,其他三組相對 46–55 歲組的漏報率差距,信賴區間都包含 0(看不出差異)。<br/>

表 A-6、A-7 列出候選範圍內各族群的選取率,但沒有另外計算 AIR,原因是:AIR 只比較各族群被選取的比例,不看流失結果,它的用途是第一道篩選,已在表 A-2 / A-3 以全體為母體做過並逐一解釋;進入候選範圍之後,要回答的問題是「同樣有資格收到這個動作的流失客戶,各族群被漏掉的比例是否相近」,這要看漏報率,AIR 回答不了。以人工聯繫為例,高價值層內女性的選取率 10.19%、男性 15.99%,若換算 AIR 是 0.637,單看會以為對女性不利;但兩性的漏報率(50.00% 對 45.27%)看不出差異,而且高價值層內女性的流失率(5.4%)確實低於男性(14.2%),女性被選得少是因為她們確實流失得少。候選範圍內的選取率因此只用來與流失率對照方向,不用來判定。<br/>

白話說明:AIR 只比較「各族群被選進名單的比例」,它完全不看這些人後來有沒有流失。它的用途是第一道篩選:快速掃一遍,看哪些「名單 × 族群」的選取比例差距大到需要解釋。這一步本模組在表 A-2、A-3 已經做完,結果是八個組合低於 0.8,然後逐一解釋。進入候選範圍之後,要回答的問題變了,變成「同樣有資格收到這個動作的客戶裡,真的會流失的人有沒有被名單漏掉,各族群漏掉的比例一不一樣」。這個問題要看流失結果,所以指標是漏報率,不是選取率。這時候再算一次 AIR,得到的仍然只是「選取比例差多少」,它不能告訴我們「差距是因為該族群本來流失就少(合理),還是規則漏掉了該族群的流失客戶(不合理)」;而這正是 1.4 本案的設定寫的:AIR 低不等於規則有問題,必須搭配漏報率判讀。<br/>
選取率仍然列出來,但只用來看方向,不用來判定。把候選範圍內各族群的選取率與該族群在候選範圍內的流失率並排看,確認「選得少的族群是不是也流失得少」(同方向),用來支持「選取率差異來自真實風險」這個解釋;它是輔助說明,不是判定差別性影響的依據。判定的依據只有漏報率差距的信賴區間(1.5 判讀規則 (a)、(c))。<br/>

26–35 歲組與 56–73 歲組在人工聯繫名單內各只有 45 人與 43 人、在加碼回饋名單內各只有 16 人與 56 人,都不足 100 人,這四格的漏報率差距本模組只描述、不作為判定依據。<br/>


**判定**(判定結果與判定依據):<br/>

Cell 7-2 1.5 的三種判讀規則:(a) 若各族群的漏報率相近;(b) 若某一個族群的選取率較高、但名單內該族群的精確率明顯較低;(c) 若某一個族群的漏報率明顯較高。<br/>

(1) 差別待遇:無。<br/>
依據:規則式名單沒有直接使用性別與年齡作為圈選條件(規則式名單依四個行為旗標圈選)。(1.2 差別待遇的定義:直接依受保護特徵給予不同的對待)。<br/>

(2) 挽留名單 × 性別:不構成「對女性的差別性影響」,落在 1.5 判讀規則 (a)。<br/>
依據:
兩性的漏報率相近(女性 28.49%、男性 27.40%,差距的信賴區間 −3.31 至 5.49 包含 0),依 1.5 的操作定義,這代表「看不出規則對兩性流失客戶的覆蓋程度有差異」;<br/>
女性的選取率較低(23.31% 對 25.73%,AIR 0.906 高於提醒線),對應的是名單內女性的精確率明顯較高(53.24% 對 41.24%,差距的信賴區間不包含 0)。兩者合起來代表:**規則在男性身上誤圈較多,名單內的男性有 58.8% 不會流失,女性只有 46.8% 不會流失**。<br/>這與模型 B「其他條件相同時女性流失勝算比 2.12」互相印證:**同樣命中旗標的客戶中,女性確實流失較多**。<br/>
所以**依行為旗標圈出的名單在女性身上較精確**。<br/>
對公司而言,這是「名單在男性身上較不精確的**成本問題**」;對男性客戶而言,不是公平性問題,理由有兩個:第一,規則沒有使用性別,所以沒有任何一位男性是「因為性別」被排除在名單外;第二,男性流失客戶確實有被名單漏掉的(漏報率 27.40%,697 位男性流失客戶中有 191 位沒有進名單),但這個比例與女性的 28.49% 看不出差異(差距的信賴區間包含 0),所以沒有證據顯示規則漏掉男性流失客戶的比例高於女性。<br/>

(3) 挽留名單 × 年齡:不構成差別性影響,落在判讀規則 (a);<br/>
其中 26–35 歲組與 56–73 歲組同時符合規則 (b),列為日後修改名單規則時應優先檢視的對象(因為:這兩組的選取率不低,但名單內真的流失的比例(精確率)明顯較低,代表誤圈較多,也就是四個旗標在這兩個年齡組身上區分「會流失」與「不會流失」的能力較弱)<br/>
依據:四組的漏報率相近(26.23%、26.57%、29.80%、27.49%,三組相對 46–55 歲組的漏報率差距,信賴區間都包含 0:看不出任何一個年齡組被名單漏掉的比例高於基準族群,定義見實測第 3 點 (c))。<br/>
AIR 最低的 46–55 歲組 0.815 仍高於提醒線;<br/>
26–35 歲組與 56–73 歲組的選取率不低(28.40%、26.97%),但名單內的精確率明顯較低(34.48%、42.62%,相對 46–55 歲組的 50.47%,差距的信賴區間都不包含 0),代表四個旗標在這兩個年齡組身上誤圈較多,對這兩組而言有成本問題(把挽留動作投放在不會流失的人身上)。<br/>
可能的業務解釋如下,但這是未經資料驗證的推測,本模組沒有計算各年齡組的旗標命中率。推斷的前提是:四個旗標是為了偵測「既有客戶正在疏遠」而設計的訊號,但同一個訊號在不同生命階段的客戶身上,代表的意思可能不同。年輕客戶:「淺關係」旗標是往來產品數 ≤ 2,但剛開始與銀行往來的年輕客戶本來就還沒有累積存款、貸款、投資等其他產品,產品少代表「關係還在起點」,不是「關係在退縮」;「零動用」旗標是額度動用率 = 0,年輕客戶的信用額度較低、多數把信用卡當付款工具而沒有循環信用的需求,動用率為 0 是用卡習慣,不是減少使用。年長客戶:「低活躍」旗標是近 12 個月交易筆數 ≤ 61,退休或接近退休的客戶,通勤、外食、採購等日常刷卡場景減少,交易筆數本來就會下降,這是生活型態改變,不是轉向其他銀行。所以這三個旗標在這兩個年齡組會命中較多不會流失的客戶,精確率就較低。這個推測可以用「各年齡組中每一個旗標的命中率,以及命中後的流失率」驗證,交由 M8 列入後續工作。<br/>

(4) 人工聯繫 × 性別(女性的 AIR 0.787,低於提醒線):不構成差別性影響,落在判讀規則 (a)。<br/>
依據:以全體客戶為母體時,女性流失客戶被人工聯繫覆蓋的比例明顯較低(女性漏報率 94.30%、男性 84.22%),但這混入了「女性在高價值層的人數比例較高(58.1%),卻較少命中旗標」的組成效應;<br/>
以高價值層為母體重算之後,**兩性流失客戶被人工聯繫覆蓋的比例看不出差異**(女性漏報率 50.00%、男性 45.27%,差距的信賴區間 −7.02 至 16.47 包含 0;這個區間寬達 23 個百分點,因為高價值層內的流失客戶只有女性 106 人、男性 201 人,所以依 1.8 只能說「看不出差異」,不能說「沒有差異」)。(1.8:判讀時,若差距的信賴區間跨過 0(例如上述 −3.31 至 5.49,區間同時包含負值與正值),代表真實差距可能是負、零或正,統計上無法排除「兩組沒有差距」,所以只能說「看不出兩組的漏報率有差異」,不能說「兩組漏報率沒有差異」。)<br/>
女性 AIR 低於提醒線的原因是:高價值層的女性較少命中 ≥ 2 個旗標(高價值層內女性的選取率 10.19%、男性 15.99%),而這些女性客戶也確實流失較少(高價值層內女性的流失率 5.4%、男性 14.2%),選取率的差異與真實風險同方向。<br/>
另一個不受母體定義影響的觀察是:搶救區內女性的精確率只有 26.50%、男性 48.67%(兩性精確率差距的信賴區間不包含 0,差異確實存在:女性的精確率確實低於男性),代表在高價值層,「命中 ≥ 2 個旗標」對女性的區分力明顯較弱(精確率低,表示被「命中 ≥ 2 個旗標」這個條件圈進搶救區的女性中,大部分不會流失;搶救區 200 位女性中約 147 位不會流失);人工聯繫是成本最高的動作,依 1.5 判讀規則 (b),誤圈較多是成本問題而不是對該族群不利(這些女性客戶得到的是較多、不是較少的挽留資源),所以這一點列為日後修改名單規則時應優先檢視的對象(意涵 (3)),不列為公平性問題。<br/>
附帶一個關於資料本身的觀察:全體女性的流失率高於男性(17.36% 對 14.62%),但高價值層內女性的流失率遠低於男性(5.4% 對 14.2%),代表「性別與流失的關係在不同價值層的方向是相反」(高價值層內女性的流失率 5.4% 低於男性 14.2%;低、中價值層內女性的流失率 24.3%(824 ÷ 3,396)高於男性 14.8%(496 ÷ 3,356),由表 A-4 與表 A-6 的人數相減算出);本模組不深究這個現象的成因(這屬於 M2 切片範圍之外的事後發現,依決策日誌 [M2-1] 只記錄、不下結論),但這個現象說明「以全體客戶算出的族群差異」不能直接套用到「某一個名單層級」上,這正是本節要以候選範圍為母體重算的原因。<br/>

(5) 加碼回饋 × 性別:不構成差別性影響,落在判讀規則 (a),但「列為優先監測項」。<br/>
依據:女性的 AIR 0.937,高於提醒線;低成本區內兩性的選取率幾乎相同(女性 15.06%、男性 14.99%);加碼回饋名單內女性的精確率較高(78.48% 對 66.00%,差距的信賴區間 2.55 至 22.41 不包含 0,差異確實存在);低成本區內流失客戶的漏報率女性 79.74%、男性 75.00%,女性相對男性的漏報率差距 +4.74 個百分點,信賴區間 −0.58 至 10.06,包含 0,所以依規則 (a) 判定看不出差異、不構成差別性影響。<br/>
列為優先監測項的原因:這個信賴區間的下限 −0.58 只差 0.58 個百分點就不包含 0,整個區間幾乎都在 0 以上,代表資料偏向「女性流失客戶被加碼回饋漏掉的比例較高」的方向,只是證據還不夠強;在本模組的所有判定中,這一格是最接近規則 (c) 門檻、最可能在資料累積後改變結論的一格,監測資源應優先放在結論最可能翻轉的地方。所以第 10 節 #37 把「低成本區內女性流失客戶的加碼回饋覆蓋率(覆蓋率 = 1 − 漏報率)」列為每季優先重算的數字。<br/>

若日後重算的結果越過門檻(兩性漏報率差距的信賴區間不包含 0 且在 0 以上),代表加碼回饋的觸發條件「消費變化比 < 0.50」系統性地漏掉女性流失客戶,屬於規則 (c) 的差別性影響,必須修改規則。<br/>
修改的方向有兩種:<br/>
第一種是對所有客戶一致地調整觸發條件,不看性別。例如把門檻由 0.50 放寬到 0.60,把消費掉了四成至五成、原本沒被圈到的流失客戶圈進來,代價是名單變大、精確率下降;或保留 0.50、另外加入一條用不同欄位衡量消費下降的條件(例如近 12 個月交易筆數的降幅),兩條任一成立就觸發,用來補撈那些「金額沒有掉到一半、但使用頻率明顯下降的流失客戶」。兩種做法都要先用資料試算新條件圈到的人數、精確率與兩性漏報率,再由業務單位決定;這不涉及依性別差別對待。<br/>
第二種是對兩性分別設定不同的觸發門檻,這等於依性別設定不同的規則,是 1.2 所說的差別待遇,必須先經法遵單位評估能否採用。<br/>
本模組建議先嘗試第一種,第二種不在本模組決定。<br/>

(6) 加碼回饋 × 年齡:不構成差別性影響,本模組不改規則;56–73 歲組的精確率列為上線後的監測項。這個結論的推論分三步:<br/>
第一,判定差別性影響看漏報率。低成本區內四組的漏報率依序為 80.28%、75.92%、78.15%、81.34%,其他三組相對基準族群 46–55 歲組的漏報率差距,信賴區間都包含 0(看不出任何一組被漏掉的比例高於基準族群),不符合規則 (c),所以不構成差別性影響,沒有修改規則的依據。<br/>
第二,56–73 歲組真正特別的地方是精確率低:它的選取率 4.21% 是四組中最高,但加碼回饋名單內的精確率 44.64% 是四組中最低,比基準族群的 75.41% 低 30.77 個百分點,信賴區間 −45.86 至 −15.67 不包含 0。這屬於規則 (b)「誤圈較多」的成本問題,不是對該族群不利;而且加碼回饋名單內的 56–73 歲組只有 56 人,不足 100 人,依決策日誌 [M2-10] 的小樣本慣例,本模組連「誤圈較多」這個結論都不單獨下。<br/>
第三,因為不能下結論,就不能據以修改規則,所以本模組把「加碼回饋名單內 56–73 歲組的精確率」列為監測項(第 10 節 #37),等這一格的人數累積到 100 人以上再判讀。

這個方向有一個可能的業務解釋,但同樣是未經資料驗證的推測。加碼回饋的觸發條件是消費變化比 < 0.50,也就是觀察期後段一季的消費金額不到前段一季的一半(欄位 total_amt_chng_q4_q1 的定義依欄位名稱推斷、尚未驗證,見第 10 節 #32;這裡的「季」是觀察期內的前後兩段,不是日曆上的年初與年底)。對工作年齡的客戶,一年之內消費金額掉了一半以上,通常代表把主要消費轉到其他銀行的卡,是流失前兆;對年長客戶,同樣幅度的下降更常來自退休、健康、行動範圍縮小、改由家人持卡消費等生活階段的改變,這些改變會讓消費持續下降,但客戶不一定會停卡。所以同一個觸發條件在 56–73 歲組會圈到較多「消費變少但不會流失」的客戶,精確率就較低。這個推測可以用「56–73 歲組中被加碼回饋圈到但沒有流失的客戶,之後幾季的消費是否持續低檔而帳戶仍然有效」來驗證,交由 M8 列入後續工作。<br/>

36–45 歲組與 46–55 歲組的 AIR 0.724、0.701 低於提醒線,是同一個現象的另一面:AIR 的分母是選取率最高的族群,在加碼回饋名單中是 56–73 歲組(4.21%);36–45 與 46–55 歲組的選取率 3.05%、2.95% 與全體客戶的加碼回饋選取率 3.04%(308 ÷ 10,127)幾乎相同,這兩組並沒有被少圈,是 56–73 歲組被多圈、把分母墊高,才讓這兩組的 AIR 算出來偏低。<br/>

36–45 歲組與 46–55 歲組在加碼回饋名單內的精確率 80.70%、75.41% 反而是四組中最高的(表示這兩組「被圈進名單的客戶,大部分真的會流失」);這兩組也沒有「相對其他組被多漏掉」的問題:低成本區內的漏報率 75.92%、78.15%,與其他兩組(80.28%、81.34%)的差距,信賴區間都包含 0。這四個漏報率在絕對數字上都很高,原因是加碼回饋的觸發條件很嚴,2,050 位低成本區客戶只有 308 人收到,所以每一個族群大部分的流失客戶都沒有收到加碼回饋;這是 M5 設計這個動作時決定的覆蓋程度(加碼回饋是基本推播之外的額外優惠,不是主要的挽留動作),不是公平性問題;公平性檢視看的是各族群被漏掉的比例是否不同,不是漏掉的比例高不高。<br/>

處置:依上述三步,本模組不改規則,把「加碼回饋名單內 56–73 歲組的精確率」列為上線後的監測項(第 10 節 #37),等人數累積到 100 人以上再判讀,並在 M8 引用加碼回饋規則時註明這個觀察。<br/>

(7) 對照名單(聯繫 ≥ 5 次人工回訪)× 性別:不做差別性影響的判定,只記錄為「代理變數存在的證據」。<br/>
依據:女性的 AIR 0.637,男性高頻率聯繫客服的比例明顯高於女性(2.81% 對 1.79%)。這個結果否定事前「只用一個行為條件圈出的名單,族群組成應接近全體」的假設(假設的邏輯見本節事前預期 (3)),並證明 1.3 所說的代理變數問題確實存在:**行為欄位本身與性別有關聯**,所以 **「規則沒有使用性別」不等於「名單與性別無關」**。<br/>
年齡也是相同狀況:56–73 歲組在對照名單的選取率 3.23% 最高,36–45 歲組與 46–55 歲組只有 2.11%、2.01%(AIR 0.653、0.622),顯示高頻率聯繫客服的行為也與年齡有關聯(判斷邏輯:若聯繫客服的次數與年齡無關,每一個年齡組中聯繫 ≥ 5 次的比例都應該接近全體的 2.27%(230 ÷ 10,127);實測四組依序為 2.72%、2.11%、2.01%、3.23%,56–73 歲組是 46–55 歲組的 1.6 倍,代表聯繫次數確實隨年齡不同)。<br/>

不做差別性影響判定的原因是:這份對照名單由客服系統依聯繫次數自動標記產生,客戶聯繫達 5 次就排入客戶經理人工回訪,不經過任何風險或價值的判斷;它的目的是服務補救,不是本模組檢視的對象「挽留資源的分配」(Cell M7-1)。差別性影響這個概念針對的是「由規則依風險分配、有人得到有人沒得到」的資源;人工回訪是對每一位聯繫達 5 次的客戶一律提供的服務,它的族群組成差異反映的是「誰常聯繫客服」,不是「誰被規則優待或排除」。至於回訪服務本身對不同族群的品質是否一致,屬於金管會公平待客原則(友善服務、告知與揭露)的範疇,是客服單位的服務品質議題,不在本模組的範圍。所以本模組只把這份名單當作代理變數存在的證據,不對它判定差別性影響。<br/>


**對決策的意涵:**<br/>
(1) 名單規則是否可用:M5 的現行名單規則可以照原樣使用,本模組不要求修改。<br/>
這個結論的推論脈絡分為以下四步,每一步寫出依據的判定編號與數字:<br/>
**第一步,先確認判斷標準**。<br/>
本模組判斷「名單規則是否可用」的標準只有一個,就是第 1 節 1.5 事前寫定的三條判讀規則:若某一個族群的漏報率明顯高於基準族群(規則 (c):漏報率差距的 95% 信賴區間不包含 0 且在 0 以上),就構成差別性影響,必須修改規則;若各族群的漏報率看不出差異(規則 (a):每一個族群相對基準族群的漏報率差距,信賴區間都包含 0),就判定不構成差別性影響,現行規則可以使用。<br/>
**第二步,性別的檢查結果**。<br/>
三個名單層級的兩性漏報率差距,信賴區間都包含 0,沒有任何一個名單層級符合規則 (c):<br/>
挽留名單女性漏報率 28.49%、男性 27.40%,差距 +1.09 個百分點,信賴區間 −3.31 至 5.49(判定 (2));<br/>
人工聯繫名單以高價值層為母體,女性漏報率 50.00%、男性 45.27%,差距 +4.73 個百分點,信賴區間 −7.02 至 16.47(判定 (4));<br/>
加碼回饋名單以低成本區為母體,女性漏報率 79.74%、男性 75.00%,差距 +4.74 個百分點,信賴區間 −0.58 至 10.06(判定 (5))。<br/>
**第三步,年齡的檢查結果**。<br/>
三個名單層級中,26–35、36–45、56–73 歲組相對基準族群 46–55 歲組的漏報率差距,信賴區間都包含 0,沒有任何一組符合規則 (c):<br/>
挽留名單四組的漏報率為 26.23%、26.57%、29.80%、27.49%(判定 (3));<br/>
人工聯繫名單以高價值層為母體,四組的漏報率為 55.81%、40.57%、49.59%、45.71%,最大的差距是 36–45 歲組的 −9.03 個百分點,信賴區間 −21.89 至 3.84(實測 (4));<br/>
加碼回饋名單以低成本區為母體,四組的漏報率為 80.28%、75.92%、78.15%、81.34%(實測 (4))。<br/>
其中 26–35 歲組與 56–73 歲組在人工聯繫名單內各只有 45 人與 43 人、在加碼回饋名單內各只有 16 人與 56 人,都不足 100 人,依決策日誌 [M2-10] 的小樣本慣例,這四格的數字本模組只描述、不作為判定依據(判定 (6)、待辦 (3))。<br/>
**第四步,回答「信賴區間包含 0 為什麼可以用來判定」**。<br/>
信賴區間包含 0 在統計上的意思是「這一批資料看不出兩個族群的漏報率有差異」,它不能證明兩個族群的漏報率完全相同(1.8)。**本模組仍然可以據此判定「不構成差別性影響、現行規則可以使用」**,理由是:<br/>
**修改一條已經在使用的名單規則,需要有「規則對某一個族群不利」的證據**;本模組事前寫定的**規則 (c) 就是這個證據的門檻**;**三個名單層級、兩性與三個年齡組的所有漏報率差距,都沒有達到這個門檻,所以本模組沒有「要求修改規則的依據」。**
**正因為「看不出差異」不等於「沒有差異」,本模組才同時「把各族群的漏報率列為上線後的監測項」(意涵 (4))**,等資料累積之後重新檢驗;**若日後某一格的差距達到規則 (c) 的門檻,再修改規則**。<br/>

補充說明**各族群選取率不同的原因**:<br/>
因為「選取率不同」是最容易被誤認為差別性影響的現象。本模組確認,三個名單層級中各族群選取率的差異,都來自「同樣命中旗標的客戶中,各族群實際流失的比例不同」,也就是規則在某些族群身上誤圈較多,依 1.5 判讀規則 (a) 與 (b),這是成本問題,不是哪一個族群被少給挽留資源。<br/>
(白話解釋:因為圈選名單的方式是「以旗標命中數來圈選」,但實際情形是:各族群就算命中相同旗標,也未必有相同的實際流失情形。因此,就算命中相同旗標,選取率也不會相同。某些族群可能有符合命中旗標的標準而被圈選進名單,但實際流失不高。舉例:A族群同樣命中三旗標的客戶,流失率高,但B族群同樣命中三旗標的客戶,流失率沒那麼高。也因此,對於B族群的誤圈率就會比較高,這就是成本問題,但沒有「哪個族群是被少給挽留資源」。本案的實例:挽留名單中同樣命中 ≥ 2 個旗標的客戶,女性有 53.24% 真的流失,男性只有 41.24%,男性就是例子裡的 B 族群,規則在男性身上誤圈較多)。<br/>
依據有三處:<br/>
挽留名單中女性的選取率較低(23.31% 對男性 25.73%),對應的是女性的精確率較高(53.24% 對 41.24%,差距的信賴區間 8.10 至 15.91 不包含 0),規則在男性身上誤圈較多(判定 (2));<br/>
挽留名單中 26–35 歲組與 56–73 歲組的選取率較高(28.40%、26.97%),對應的是這兩組的精確率較低(34.48%、42.62%,相對 46–55 歲組 50.47% 的差距,信賴區間都不包含 0),規則在這兩組身上誤圈較多(判定 (3));<br/>
人工聯繫名單中女性的選取率較低(高價值層內 10.19% 對男性 15.99%,女性的 AIR 0.787 低於提醒線),對應的是高價值層內女性的流失率(5.4%)確實低於男性(14.2%),選取率的差異與真實風險同方向(判定 (4))。<br/>


(2) 寫在 docs/07_compliance_fairness.md 第 3 節的決策句,共四句,每一句後面標出其出自本節的哪一個判定:<br/>
第一句:「挽留名單以四個行為旗標與交易金額圈選,未使用性別與年齡,沒有差別待遇。」出自判定 (1)。<br/>
第二句:「三個名單層級在各自的候選範圍內,兩性與各年齡組真的會流失的客戶被名單覆蓋的比例看不出差異(每一個族群相對基準族群的漏報率差距,95% 信賴區間都包含 0),符合第 1 節 1.5 判讀規則 (a),**不構成差別性影響**;各族群選取率的差異由「同樣命中旗標的客戶中,各族群實際流失的比例不同」解釋,屬於成本問題。」出自判定 (2)、(3)、(4)、(5),推論脈絡見意涵 (1)。<br/>
第三句:「性別組與年齡組在三個名單層級的選取率、AIR、精確率、漏報率,列為名單上線後的每季監測項。」原因是意涵 (1) 第四步所說「看不出差異不等於沒有差異」,以及意涵 (4) 所說「族群組成會隨客戶結構改變」;出自判定 (5)、(6) 與意涵 (4)。<br/>
第四句:「每季監測時,優先重算兩個數字:**低成本區內女性流失客戶的加碼回饋覆蓋率(覆蓋率 = 1 − 漏報率)**,以及**加碼回饋名單內 56–73 歲組的精確率**。」
前者出自判定 (5):低成本區內兩性漏報率差距的信賴區間 −0.58 至 10.06,**下限接近 0,是三個名單層級中最接近規則 (c) 門檻的一格,所以要優先確認資料累積後是否越過門檻**(下限 −0.58 只差 0.58 個百分點就不包含 0,代表再累積一些資料,信賴區間就有可能整個落在 0 以上,達到規則 (c) 的門檻;現在的資料還不能這樣判定)。<br/>
後者出自判定 (6):加碼回饋名單內 56–73 歲組的精確率 44.64%,比 46–55 歲組低 30.77 個百分點,但名單內只有 56 人,**依小樣本慣例本次不下結論,要等人數累積到 100 人以上再判讀**。<br/>


(3) 名單規則的精確率在五個「名單層級 × 族群」的格子明顯較低,這五格不是公平性問題,但是**日後修改 M5 名單規則時應優先檢視的對象**。<br/>
挑出這五格的條件有兩個,都是本模組事前寫定的:<br/>
**該格的精確率相對基準族群的差距,95% 信賴區間不包含 0(1.8 的「明顯」定義),而且該格名單內的人數不少於 100 人(決策日誌 [M2-10] 的小樣本慣例)**。<br/>
符合這兩個條件的格子如下:<br/>
挽留名單內的男性(精確率 41.24%,比女性低 12.00 個百分點,信賴區間 8.10 至 15.91,名單內 1,227 人;判定 (2));<br/>
挽留名單內的 26–35 歲組(34.48%,比 46–55 歲組低 15.99 個百分點,信賴區間 −22.57 至 −9.41,名單內 261 人;判定 (3));<br/>
挽留名單內的 56–73 歲組(42.62%,低 7.85 個百分點,信賴區間 −13.87 至 −1.83,名單內 359 人;判定 (3));<br/>
人工聯繫名單內的女性(26.50%,比男性低 22.17 個百分點,信賴區間 −31.11 至 −13.24,名單內 200 人;判定 (4));<br/>
加碼回饋名單內的男性(66.00%,比女性低 12.48 個百分點,信賴區間 2.55 至 22.41,名單內 150 人;判定 (5))。<br/>
加碼回饋名單內的 56–73 歲組精確率 44.64% 雖然也明顯較低,但名單內只有 56 人,不符合第二個條件,只列為監測項(意涵 (2) 第四句)。<br/>
**這五格不是公平性問題的理由見 1.5 判讀規則 (b):精確率低代表規則在這個族群身上誤圈較多,這個族群得到的挽留資源是較多、不是較少**。它們是修改規則時應優先檢視的對象,理由是:**精確率低代表四個行為旗標在這些族群身上區分「會流失」與「不會流失」的能力較弱,若日後要提高名單的精確率,在這些族群上最容易看出新條件有沒有效果**;<br/>
M4 第 7 節的決策意涵曾提出「在命中 ≥ 2 與命中 ≥ 3 之間,可加入循環欠款(本金)低、不活躍月數高作為第三種名單條件的候選」,就可以先在這五格上測試。<br/>
這件事交由 M8 在 Limitations & Future Work 記錄,本模組不修改規則(Cell M7-1「本模組不處理的事項」(5))。<br/>

(4) **上線後的監測**:<br/>
名單規則每季重算時,應同時**重算性別組與年齡組在三個名單層級的選取率、AIR、精確率、漏報率(也就是表 A-2 至 A-7 的四個數字)**。<br/>
每季重算的時點沿用 notebook 05 第 8 節名單總表的規定(監測區客戶每季重算旗標命中數與交易金額分位;這裡的「監測區」是四個策略區之一,不是「監測作業」的意思),族群指標與名單一起重算,不需要另外排程。<br/>
要重算的理由有兩個:<br/>
第一,本模組對檢查 A 的判定「三個名單層級不構成差別性影響」,**只對這一批 10,127 位客戶的資料成立**;客戶結構(各年齡組的人數比例、各族群的實際流失率)會隨時間改變,**同一套規則在新的客戶結構下,各族群的選取率與漏報率可能改變,這一次的判定不能延伸到之後的每一期**。<br/>
第二,本次「有一格的數字接近判讀規則 (c) 的門檻(判定 (5):低成本區內兩性漏報率差距的信賴區間下限 −0.58,接近 0)」,「另有一格因為人數不足沒有判定(判定 (6):加碼回饋名單內 56–73 歲組 56 人)」,**這兩格要等資料累積後才能確認**。<br/>
監測的觸發條件與優先重算項寫在第 10 節 #37。<br/>



**待辦或限制:**<br/>
(1) **檢查 A 只覆蓋性別與年齡兩個特徵**。<br/>
原因有兩個:<br/>
第一,本資料集有記錄的受保護特徵只有性別、年齡、婚姻狀態三欄,種族、居住地、身心障礙等其他國家法規列為受保護特徵的屬性,本資料集沒有欄位,無法檢查;<br/>
第二,這三欄之中,婚姻狀態只在模型層(檢查 B)以「多訓練一版不含婚姻狀態的模型」處理,名單層沒有檢查婚姻狀態的族群組成,理由見 Cell M7-1「檢視的特徵與名單層級」。<br/>
(2) **檢查 A 通過,不代表資料本身沒有歷史偏差**。<br/>
檢查 A 通過的意思只是「名單在各族群的選取率與漏報率,與各族群的真實流失率一致」;但**四個行為旗標本身可能承載既有的性別或年齡差異**(NYDFS 報告所說的 legacy bias,見 1.3),對照名單的結果就是證據:聯繫 ≥ 5 次的客戶占男性的 2.81%、占女性的 1.79%(判定 (7)),代表**高頻率聯繫客服這個行為與性別有關聯**。本模組「無法用這批資料判斷這種關聯是客戶行為的真實差異,還是歷史上服務方式造成的差異,所以只能列為限制」。<br/>
(3) **年齡組與小名單交叉後有四格人數不足 100 人,本模組不對這四格下結論。**<br/>
這四格是:人工聯繫名單內的 26–35 歲組(45 人)與 56–73 歲組(43 人)、加碼回饋名單內的 26–35 歲組(16 人)與 56–73 歲組(56 人)。<br/>
這四格的精確率差距與漏報率差距,本模組只描述數字並列為監測項(意涵 (4)),不作為判定差別性影響或判定優先檢視對象的依據。<br/>
(4) **本檢查的判準是機會平等,若審核單位改用人口比例平等,結論會不同。**<br/>
本模組依 1.6 選擇「機會平等(各族群漏報率相近)」為主要判準。若法遵或風險控管人員要求改以人口比例平等(各族群選取率相同)為判準,本案會有兩格被視為需要處理:<br/>
挽留名單中,女性的選取率較低(AIR 0.906)、46–55 歲組的選取率較低(AIR 0.815)。但 1.6 已說明,在各族群基準率(真實流失率)不同時,不做差別待遇就無法同時滿足兩種判準;<br/>
而且在本案的數字下,要拉平選取率只有兩種做法:(a)多圈「女性」與「46–55 歲組中命中旗標較少的客戶」,名單在這兩個族群的精確率會再下降(這是成本問題);或(b)少圈男性與其他年齡組的客戶(這會提高這些族群流失客戶的漏報率)。<br/>
無論哪一種做法都是依性別、年齡設定不同的圈選門檻,也就是 1.2 的差別待遇,所以本模組不採用。<br/>

In [3]:
# Cell M7-7(code)|第 5 節:檢查 B:模型 B 三個版本(含性別、不含性別、不含性別與婚姻狀態)的名單品質比較(待驗證事項 #27)
# 本 Cell 要回答的問題:若把性別(以及婚姻狀態)從模型 B 的輸入特徵中移除,在同樣的名單人數下,名單的精確率會下降多少、名單會換掉多少人、名單的女性占比會改變多少。
# 做法:以同一組資料、同一種切分、同一套特徵處理,訓練三個「只差在類別變數清單的模型 B 版本」,在同一個測試集上比較;五個 random_state 各做一次,結論以五次的平均為準(與 M4 第 7 節的穩定性檢查相同)。

# 特徵處理完全沿用 notebook 04 Cell M4-13 : 12 個連續變數做標準化(每一個變數減去訓練集的平均、再除以訓練集的標準差;平均與標準差只從訓練集計算,再套用到測試集,避免測試集的資訊滲入訓練), 4 個類別變數轉成虛擬變數(dummy variable,每一個類別一欄、是為 1 否為 0)並去掉基準組那一欄(基準組是比較的基準,例如 gender 以 M 為基準,所以只保留 gender_F 一欄)。
NUMERIC = ["customer_age", "dependent_count", "total_relationship_count", "months_inactive_12_mon",
           "contacts_count_12_mon", "credit_limit", "total_revolving_bal", "avg_utilization_ratio",
           "total_amt_chng_q4_q1", "total_trans_amt", "total_trans_ct", "total_ct_chng_q4_q1"]
CATEGORICAL_FULL = {"gender": "M", "income_category": "Less than $40K",
                    "education_level": "Graduate", "marital_status": "Married"}   # 鍵 = 類別變數欄位,值 = 該變數的基準組(轉虛擬變數時去掉的那一個類別)

# 三個模型版本:三版之間唯一的差別是「類別變數的清單」,其餘設定(12 個連續變數、標準化方式、訓練集與測試集的切分、random_state)完全相同,
# 這樣三版名單精確率的差異才能只歸因於「模型有沒有使用該特徵」,不會混入其他因素。
VERSIONS = {
    "with_gender":          CATEGORICAL_FULL,                                                           # 含性別版:M4 模型 B 原樣,27 個特徵(12 個連續變數 + 15 個虛擬變數)
    "no_gender":            {k: v for k, v in CATEGORICAL_FULL.items() if k != "gender"},                # 不含性別版:從類別變數清單移除 gender,26 個特徵(性別只有一個虛擬變數 gender_F)
    "no_gender_no_marital": {k: v for k, v in CATEGORICAL_FULL.items() if k not in ("gender", "marital_status")},  # 不含性別與婚姻狀態版:再移除 marital_status,23 個特徵(婚姻狀態有四個類別,去掉基準組 Married 後是 3 個虛擬變數,所以少 3 個特徵)
}


def build_design(d: pd.DataFrame, categorical: dict, means=None, stds=None):
    """建立模型的設計矩陣(design matrix,也就是模型輸入的特徵表:每一列一位客戶、每一欄一個特徵)。
    做法與 notebook 04 Cell M4-13 相同,只多一個 categorical 參數,用來切換三個版本的類別變數清單。
    means、stds 參數:處理訓練集時不傳入,函式會從訓練集算出平均與標準差並回傳;處理測試集時把訓練集算出的平均與標準差傳入,使測試集用同一組數字標準化。"""
    X = d[NUMERIC].astype(float).copy()
    if means is None:
        means, stds = X.mean(), X.std(ddof=0)          # 只有在處理訓練集時才會進到這裡計算平均與標準差
    X = (X - means) / stds                             # 標準化:每一個連續變數減去平均、除以標準差
    for col, ref in categorical.items():
        dummies = pd.get_dummies(d[col], prefix=col, dtype=float).drop(columns=[f"{col}_{ref}"])   # 轉成虛擬變數,並去掉基準組那一欄(例如 gender_M)
        X = pd.concat([X, dummies], axis=1)
    return X, means, stds


def top_k(te: pd.DataFrame, prob: pd.Series, k: int) -> pd.DataFrame:
    """從測試集中取出模型預測流失機率最高的 k 位客戶,作為該版模型的名單。
    機率相同時依 clientnum 由小到大排,目的是讓同一組資料每次執行都圈出同一份名單,不受排序演算法內部順序的影響(與 M4 Cell M4-16 相同)。"""
    return te.assign(prob=prob.values).sort_values(["prob", "clientnum"], ascending=[False, True]).head(k)


def run_once(seed: int) -> dict:
    """執行一個切分:以傳入的 random_state 把全體客戶切成訓練集(70%)與測試集(30%),三版模型各自用同一個訓練集訓練,再在同一個測試集上比較四件事:
    (1)固定名單規模下的精確率、(2)平均精確率(AP)、(3)名單中的女性占比、(4)不同版本名單之間的重疊率。
    回傳一個字典,一個切分一列,欄位名稱就是表 B-1、B-2 的欄位名稱。"""
    # stratify=df["is_churned"]:分層抽樣,讓訓練集與測試集的流失率都等於全體的 16.07%,避免某一次切分的測試集流失客戶特別多或特別少
    tr_i, te_i = train_test_split(df.index, test_size=TEST_SIZE, stratify=df["is_churned"], random_state=seed)
    tr, te = df.loc[tr_i], df.loc[te_i]
    # 兩種名單規模 k3、k2 = 這個切分的測試集上,規則命中數 ≥ 3 與 ≥ 2 的人數;模型名單取同樣的人數,兩者才能公平比較精確率(M4 第 7 節的「固定名單規模比精確率」)。
    # random_state 42 時 k3 = 178、k2 = 703,與 M4 相同;其他切分的人數不同(k3 為 162 至 182 人、k2 為 721 至 755 人),本節文字以「178 人名單」「703 人名單」代稱這兩種規模。
    k3, k2 = int((te["hit_count"] >= 3).sum()), int((te["hit_count"] >= 2).sum())
    out = {"seed": seed, "k3": k3, "k2": k2,
           "rule_p@k3": round(100 * te[te["hit_count"] >= 3]["is_churned"].mean(), 1),     # 規則名單(命中 ≥ 3)的精確率 = 名單內真的流失的比例
           "rule_p@k2": round(100 * te[te["hit_count"] >= 2]["is_churned"].mean(), 1),     # 規則名單(命中 ≥ 2)的精確率
           "rule_shareF@k2": round(100 * (te[te["hit_count"] >= 2]["gender"] == "F").mean(), 1)}   # 規則名單(命中 ≥ 2)中女性的占比,作為三版模型名單女性占比的對照基準
    lists_k2 = {}                                      # 暫存三版模型各自 k2 人名單的 clientnum 集合,最後用來算名單之間的重疊率
    for name, cat in VERSIONS.items():
        xb_tr, m, s = build_design(tr, cat)            # 訓練集的特徵表,並取得訓練集的平均與標準差
        xb_te, _, _ = build_design(te, cat, m, s)      # 測試集的特徵表,用訓練集的平均與標準差標準化
        xb_te = xb_te.reindex(columns=xb_tr.columns, fill_value=0.0)   # 保險:模型是用訓練集的欄位訓練的,predict 時測試集的欄位必須與訓練集完全一致;若測試集剛好缺少某一個類別(例如沒有人是 Divorced),轉虛擬變數後就會少一欄而報錯,所以補上該欄並填 0(意思是測試集沒有人屬於這個類別)。本案每一個類別在測試集都有人,這一行實際上不會改變任何數字
        model = sm.Logit(tr["is_churned"], sm.add_constant(xb_tr)).fit(disp=0, maxiter=200)   # 邏輯斯迴歸,加常數項;係數以迭代法求解,disp=0 代表不印出每一步迭代的訊息,maxiter=200 代表最多迭代 200 次
        p = model.predict(sm.add_constant(xb_te))      # 測試集每一位客戶的預測流失機率(M4 所稱的模型 B 機率分數),是 0 到 1 之間的數字(例如 A 客戶 0.8752、B 客戶 0.1589),越大代表模型認為越可能流失;本案只用它由大到小排序取名單,不對客戶宣稱為流失機率(決策日誌 [M4-2])
        t3, t2 = top_k(te, p, k3), top_k(te, p, k2)    # 該版模型的兩份名單
        out[f"{name}_p@k3"] = round(100 * t3["is_churned"].mean(), 1)            # 該版模型 k3 人名單的精確率
        out[f"{name}_p@k2"] = round(100 * t2["is_churned"].mean(), 1)            # 該版模型 k2 人名單的精確率
        out[f"{name}_AP"] = round(average_precision_score(te["is_churned"], p), 3)   # 平均精確率(AP):整條精確率—召回率曲線的總結,用來確認三版模型的整體排序能力是否相近(notebook 04 第 1 節 1.7)
        out[f"{name}_shareF@k2"] = round(100 * (t2["gender"] == "F").mean(), 1)   # 該版模型 k2 人名單中女性的占比
        lists_k2[name] = set(t2["clientnum"])
        if name == "with_gender":
            out["OR_female"] = round(float(np.exp(model.params["gender_F"])), 2)     # 含性別版中 gender_F 係數取 e 的次方 = 女性相對男性的流失勝算比(random_state 42 時應等於 M4 的 2.12,可作為本節與 notebook 04 一致的核對點)
            out["p_female"] = round(float(model.pvalues["gender_F"]), 4)            # gender_F 係數的 p 值;四捨五入到小數第 4 位,實際值小於 0.001 時會顯示 0.0(第 5 節判讀實測 (4) 有說明)
    # 名單重疊率:不含性別版與含性別版各取 k2 人的名單,「兩份名單中相同客戶的人數」 ÷ k2 × 100,得出「名單重疊率」;回答「移除性別之後,名單實際上換掉了多少人」,例如重疊率 92.0% 代表兩份名單有 92% 的客戶相同、換掉了 8% 的客戶,換算人數要乘 k2(k2 = 703 時約 56 人);本案實測重疊率 97.1%,換掉約 20 人
    # overlap_nm 是「不含性別與婚姻狀態版」與「含性別版」的重疊率。
    out["overlap@k2_pct"] = round(100 * len(lists_k2["with_gender"] & lists_k2["no_gender"]) / k2, 1)
    out["overlap_nm@k2_pct"] = round(100 * len(lists_k2["with_gender"] & lists_k2["no_gender_no_marital"]) / k2, 1)
    return out


# 五個切分各執行一次,疊成一張表(一個切分一列),再加一列五個切分的平均(seed 欄標為 mean);判讀以 mean 列為準,不以單一切分為準(決策日誌 [M4-2])
stability_b = pd.DataFrame([run_once(s) for s in SEEDS])
mean_row = stability_b.drop(columns=["seed"]).mean(numeric_only=True).round(2)
mean_row["seed"] = "mean"
stability_b = pd.concat([stability_b, mean_row.to_frame().T], ignore_index=True)

# 表 B-1 只列「精確率相關欄位」,表 B-2 列「其餘欄位」(AP、勝算比、女性占比、重疊率),兩張表的每一欄在第 5 節判讀「各表欄位的意義」逐一說明
cols_prec = ["seed", "k3", "k2", "rule_p@k3", "with_gender_p@k3", "no_gender_p@k3", "no_gender_no_marital_p@k3",
             "rule_p@k2", "with_gender_p@k2", "no_gender_p@k2", "no_gender_no_marital_p@k2"]
cols_other = ["seed", "with_gender_AP", "no_gender_AP", "no_gender_no_marital_AP", "OR_female", "p_female",
              "rule_shareF@k2", "with_gender_shareF@k2", "no_gender_shareF@k2", "no_gender_no_marital_shareF@k2",
              "overlap@k2_pct", "overlap_nm@k2_pct"]
print("表 B-1:固定名單規模的精確率(%):規則 vs 三版模型,五個切分與平均")
display(stability_b[cols_prec])
print("表 B-2:平均精確率(AP)、含性別版的女性勝算比、703 人名單的女性占比(%)、與含性別版名單的重疊率(%)")
display(stability_b[cols_other])

# 表後三行:以 mean 列(五個切分的平均)計算三個精確率差距:不含性別版減含性別版、不含性別與婚姻狀態版減含性別版、不含性別版減規則名單;單位為百分點;第三行括號內的 +7.4 / +8.6 是 notebook 04 第 7 節算出的含性別版減規則名單的增量,供對照
m = stability_b[stability_b["seed"] == "mean"].iloc[0]
print("移除性別特徵的精確率代價(不含性別版的精確率 − 含性別版的精確率;五個切分平均;負值代表移除性別後名單精確率下降):",
      f"178 人名單 {m['no_gender_p@k3'] - m['with_gender_p@k3']:+.1f} 個百分點,",
      f"703 人名單 {m['no_gender_p@k2'] - m['with_gender_p@k2']:+.1f} 個百分點")
print("再移除婚姻狀態的精確率代價(不含性別與婚姻狀態版的精確率 − 含性別版的精確率;五個切分平均):",
      f"178 人名單 {m['no_gender_no_marital_p@k3'] - m['with_gender_p@k3']:+.1f} 個百分點,",
      f"703 人名單 {m['no_gender_no_marital_p@k2'] - m['with_gender_p@k2']:+.1f} 個百分點")
print("不含性別版相對規則名單的精確率增量(不含性別版的精確率 − 規則名單的精確率;五個切分平均;正值代表模型名單比規則名單更準):",
      f"178 人名單 {m['no_gender_p@k3'] - m['rule_p@k3']:+.1f} 個百分點,",
      f"703 人名單 {m['no_gender_p@k2'] - m['rule_p@k2']:+.1f} 個百分點(對照:notebook 04 第 7 節以含性別版算過同一個差距,178 人名單 +7.4、703 人名單 +8.6 個百分點;兩者相比可看出移除性別後,模型相對規則的優勢保留了多少)")

表 B-1:固定名單規模的精確率(%):規則 vs 三版模型,五個切分與平均


,seed,k3,k2,rule_p@k3,with_gender_p@k3,no_gender_p@k3,no_gender_no_marital_p@k3,rule_p@k2,with_gender_p@k2,no_gender_p@k2,no_gender_no_marital_p@k2
0,0,165,731,83.6,91.5,89.7,90.9,48.2,56.2,56.1,56.0
1,1,182,755,81.9,91.2,90.1,90.1,47.0,54.8,54.3,54.3
2,7,162,731,85.2,92.0,91.4,88.9,46.5,55.5,55.4,54.9
3,42,178,703,84.3,89.9,87.6,88.8,46.5,56.3,55.9,55.5
4,99,177,721,81.4,88.7,89.3,85.9,47.7,55.9,56.0,55.6
5,mean,172.8,728.2,83.28,90.66,89.62,88.92,47.18,55.74,55.54,55.26


表 B-2:平均精確率(AP)、含性別版的女性勝算比、703 人名單的女性占比(%)、與含性別版名單的重疊率(%)


,seed,with_gender_AP,no_gender_AP,no_gender_no_marital_AP,OR_female,p_female,rule_shareF@k2,with_gender_shareF@k2,no_gender_shareF@k2,no_gender_no_marital_shareF@k2,overlap@k2_pct,overlap_nm@k2_pct
0,0,0.762,0.759,0.754,2.38,0.0,49.0,55.0,52.4,51.2,97.1,93.4
1,1,0.77,0.771,0.768,2.97,0.0,52.8,56.8,53.5,54.4,96.3,92.8
2,7,0.766,0.76,0.751,2.39,0.0,50.6,55.4,52.4,52.7,97.0,94.0
3,42,0.753,0.745,0.746,2.12,0.0,51.9,59.0,57.5,56.9,98.2,94.2
4,99,0.746,0.743,0.74,2.6,0.0,50.5,56.2,53.5,52.8,96.9,93.3
5,mean,0.76,0.76,0.75,2.49,0.0,50.96,56.48,53.86,53.6,97.1,93.54


移除性別特徵的精確率代價(不含性別版的精確率 − 含性別版的精確率;五個切分平均;負值代表移除性別後名單精確率下降): 178 人名單 -1.0 個百分點, 703 人名單 -0.2 個百分點
再移除婚姻狀態的精確率代價(不含性別與婚姻狀態版的精確率 − 含性別版的精確率;五個切分平均): 178 人名單 -1.7 個百分點, 703 人名單 -0.5 個百分點
不含性別版相對規則名單的精確率增量(不含性別版的精確率 − 規則名單的精確率;五個切分平均;正值代表模型名單比規則名單更準): 178 人名單 +6.3 個百分點, 703 人名單 +8.4 個百分點(對照:notebook 04 第 7 節以含性別版算過同一個差距,178 人名單 +7.4、703 人名單 +8.6 個百分點;兩者相比可看出移除性別後,模型相對規則的優勢保留了多少)


Cell M7-8(markdown)|第 5 節判讀:檢查 B

第 5 節判讀|從模型 B 移除性別特徵的代價(待驗證事項 #27)

**本節結論**:<br/>
**性別不作為名單圈選的依據,也不作為模型 B 的輸入特徵**;模型 B 若日後依決策日誌 [M5-3] 的切換條件用於圈選名單,**一律採用不含性別版的模型**。<br/>

**依據**:<br/>
**1.移除性別要付出的精確率代價小**。<br/>
703 人名單的精確率平均下降 0.2 個百分點,低於第 1 節 1.7 判讀規則 (a)「小於 1 個百分點視為無實質差異」的門檻;178 人名單平均下降 1.0 個百分點,恰好落在這個門檻的邊界上,相當於 178 人中 2 位客戶的差別,本節如實載明「約 1 個百分點」。<br/>
不含性別版與含性別版的 703 人名單有 97.1% 的客戶相同,移除性別只換掉約 20 人。<br/>
**2.這個決策的理由不是「代價小」,而是 1.7 事先寫定的判讀規則**:<br/>
「以性別圈名單」是 1.2 的差別待遇,對客戶、業務單位與主管機關都無法說明,所以**無論代價多少都不以性別為特徵**;檢查 B 的工作是**把代價算出來,讓決策者知道這個決策付出了什麼**。<br/>

**本節另外確認兩件事,後面的章節會用到**:<br/>
1.**含性別版模型確實把更多女性排進名單**(703 人名單的女性占比 56.48%,規則名單只有 50.96%),而且移除性別後,其他特徵仍帶有部分性別資訊。<br/>
推論脈絡:規則名單只依四個行為旗標圈選、完全不使用性別,所以它的女性占比 50.96% 代表「不用性別時,名單裡自然會有多少女性」,是另外兩版比較的起點;含性別版名單的女性占比 56.48%,比起點高 5.52 個百分點,這一段差距來自性別特徵。移除性別之後,如果模型其他的特徵都與性別無關,女性占比應該會回到 50.96% 附近;但實測是 53.86%,仍然比起點高 2.90 個百分點。性別已經不在模型裡,卻仍然多排進女性,代表模型其他的特徵(年齡、扶養人數、收入、教育程度、婚姻狀態與行為變數)之中,有一部分與性別有關聯,模型透過這些特徵間接地把女性排在前面。這就是 1.3 所述的代理變數問題。<br/>
2.不含性別版模型 B 的 703 人名單精確率,在五個切分都比規則名單高 5 個百分點以上(平均高 8.4 個百分點,最小的差距是 random_state 1 的 7.3 個百分點:54.3% 對 47.0%),符合 M4 採信模型 B 的標準,[M5-3]「模型 B 為升級選項」維持,但切換條件中的成本節省比例要由 17.4% 改為 16.8%(算法見意涵 (2));178 人名單規模下有一個切分不符合標準,若日後名單規模改為「命中 ≥ 3」,升級前須重新評估(判定 (5))。<br/>

以下先說明「表 B-1、B-2 與表後三行文字的每一欄」代表什麼,再說明「事前預期」,最後依「實測 / 判定 / 對決策的意涵 / 待辦或限制」逐項判讀。<br/>


**各表欄位的意義**:<br/>
**表 B-1(精確率)**:<br/>
每一列是一個切分(seed 欄是該切分的 random_state),最後一列 mean 是五個切分的平均。<br/>
(1) k3、k2 是該切分的測試集上,規則命中數 ≥ 3 與 ≥ 2 的人數,也就是「本節使用的兩種名單規模」;模型名單取同樣的人數,才能與規則名單公平比較精確率(M4 第 7 節的「固定名單規模比精確率」,定義見 1.10)。random_state 42 時 k3 = 178、k2 = 703,與 M4 相同;其他四個切分的 k3 為 162 至 182 人、k2 為 721 至 755 人。本節的文字與表後三行輸出以「178 人名單」「703 人名單」代稱 k3、k2 這兩種規模。<br/>
(2) rule_p@k3、rule_p@k2 是規則名單(命中 ≥ 3、命中 ≥ 2)的「精確率」,也就是「名單內真的流失的客戶比例」。(規則名單是「命中旗標的人」,但命中旗標不等於會流失,名單裡有人事後真的流失、有人沒有;精確率 = 名單內事後真的流失(is_churned = 1)的人數 ÷ 名單人數,衡量的是「規則圈進來的人準不準」。例如 random_state 42 的 703 人規則名單精確率 46.5%,代表 703 人中約 327 人真的流失)。<br/>
(3) with_gender_p@k3、with_gender_p@k2 是「含性別版模型」取預測流失機率最高的 k3 人、k2 人時,名單的精確率;<br/>
no_gender_、no_gender_no_marital_ 開頭的欄位是「不含性別版、不含性別與婚姻狀態版的同一個數字」。<br/>
同一列內,**三版模型用的是同一個訓練集與同一個測試集**,三版之間的**差異只來自「特徵集不同(Cell M7-7 的 VERSIONS)」**。<br/>

**表 B-2(其他指標)**:<br/>
(1) with_gender_AP、no_gender_AP、no_gender_no_marital_AP 是三版模型的平均精確率(average precision, AP)。精確率與 AP 的分工:精確率回答「在本案實際會用的名單人數(178 人或 703 人)下,名單準不準」,直接對應業務決策;AP 回答「不管名單切在哪一個人數,模型把會流失的人排在前面的整體能力」,它是把每一個可能的名單人數下的精確率平均起來的數字(notebook 04 第 1 節 1.7),不依賴某一個人數。<br/>
兩者都要看的原因是:精確率只能說明「在 178 人與 703 人這兩種名單人數下,三版模型的名單精確率幾乎相同」; AP 才能說明「移除性別或婚姻狀態之後,模型把會流失的客戶排在不會流失的客戶前面的能力,在任何名單人數下都幾乎相同」。所以若日後業務單位把名單人數改成其他數字(例如 400 人),精確率就必須重算,但 AP 的結論仍能適用。<br/>
同時報一個固定門檻下的指標(精確率)與一個不依賴門檻的指標(AP),是「多種模型比較」的常見做法,M4 也是這樣做,本節沿用;**本節的主要依據仍是精確率,AP 是輔助**。<br/>
(2) OR_female 是「含性別版中 gender_F 的勝算比」,也就是女性相對男性的流失勝算比(定義見 1.10);p_female 是這個係數的 p 值(p 值小於 0.05 代表統計顯著,定義見 1.10);四捨五入到小數點後第 4 位,實際值小於 0.001 時會顯示 0.0,表中顯示 0.0 的原因在實測 (4) 說明。<br/>
(3) rule_shareF@k2 是「規則名單(命中 ≥ 2)中女性的占比」;三版的 shareF@k2 是「各版模型 k2 人名單中女性的占比」。這四欄回答「含性別的模型是否把更多女性排進名單」。<br/>
(4) overlap@k2_pct 是「不含性別版」與「含性別版」的 **「k2 人名單中相同客戶的人數」 ÷ k2 × 100**,也就是 1.7 定義的 **「名單重疊率」**;<br/>
overlap_nm@k2_pct 是「不含性別與婚姻狀態版」與「含性別版」名單的**重疊率**。這兩欄回答「移除特徵之後,名單實際上換掉了多少比例的客戶」;換算成人數要乘 k2,例如重疊率 97.1% 代表 2.9% 的客戶被換掉,k2 = 703 時是 703 × 2.9% ≈ 20 人。<br/>

**表後印出三行文字**:<br/>
1.表後三行是「三個精確率差距」,都用表 B-1 的 mean 列(五個切分的平均精確率)相減算出,單位是百分點。<br/>
2.第一行是「模型移除性別」的精確率代價(「不含性別版」的精確率減「含性別版」的精確率,負值代表移除性別後精確率下降);第二行是「模型繼移除性別後再移除婚姻狀態」的精確率代價(「不含性別與婚姻狀態版」的精確率減「含性別版」的精確率);第三行是「不含性別版」相對「規則名單」的精確率增量(「不含性別版」的精確率減「規則名單」的精確率,正值代表「不含性別版模型名單」比「規則名單」更準)。<br/>
括號內是對照用的數字:notebook 04 第 7 節也算過「模型名單精確率減規則名單精確率」,只是當時用的是「含性別版模型」,結果是 178 人名單 +7.4、703 人名單 +8.6 個百分點;與本行的「不含性別版模型」的增量數值「178 人名單 +6.3、703 人名單 +8.4」相比,就能看出移除性別後,模型相對規則的優勢保留了多少。<br/>


**事前預期**:<br/>
本模組在執行 Cell M7-7 之前,先寫下判斷標準與預期方向,目的與第 4 節相同:判定時只能依事先寫定的標準,避免看到數字之後才選一個讓結果好看的解釋。<br/>
判斷標準(第 1 節 1.7 事先寫定):移除性別後,固定名單規模的精確率平均下降小於 1 個百分點,視為無實質差異;下降達到 3 個百分點以上,必須明寫代價;介於兩者之間,如實載明數字。無論落在哪一個區間,決策都是「不以性別作為模型 B 的輸入特徵」。<br/>
**預期的方向與依據**:本模組預期移除性別後,模型名單的變動很小。依據是性別在模型 B 中的效果遠小於行為變數:含性別版模型中性別的勝算比是 2.12(random_state 42;其他條件相同時,女性的流失勝算是男性的 2.12 倍);交易筆數每多一個標準差,流失勝算就變成原本的 0.063 倍(notebook 04 第 6 節),也就是差一個標準差,流失勝算相差約 16 倍。模型名單是把測試集客戶依預測流失機率(模型 B 對每一位客戶算出的 0 到 1 之間的數字,M4 稱為模型 B 機率分數)由高到低排序後取前 k 位;性別對預測流失機率的影響既然遠小於行為變數,移除性別後,每一位客戶的排序只會小幅改變,名單中大部分的客戶會維持不變。<br/>
**由此推出三個方向**:<br/>
(1) 移除性別後,精確率的下降會落在判斷標準的第一個區間(小於 1 個百分點)。<br/>
(2) 不含性別版與含性別版的名單重疊率高,也就是兩份名單中相同客戶的比例高。<br/>
(3) 含性別版名單的女性占比高於不含性別版。原因:含性別版模型對女性客戶算出的預測流失機率,比條件相同的男性客戶高(勝算比 2.12 大於 1),女性客戶因此較容易進入名單;移除性別後這個差異消失,女性占比會下降。<br/>
**實測結果與三個方向一致**(數字見實測 (2)、(5)、(6)):<br/>
(1) 703 人名單的精確率下降 0.2 個百分點,落在第一個區間;178 人名單下降 1.0 個百分點,恰好落在第一與第二區間的邊界,判定 (1) 說明這一點的處理方式。<br/>(2) 不含性別版與含性別版的 703 人名單重疊率 97.1%。<br/>
(3) 含性別版名單的女性占比 56.48%,高於不含性別版的 53.86%。<br/>


**實測:**<br/>
1.含性別版模型的數字與 notebook 04 完全相同,所以本節比較的基準就是 M4 的模型 B。<br/>
核對的項目有三個:<br/>
(1) 含性別版 178 人名單的精確率,random_state 0、1、7、42、99 五個切分,分別為 91.5%、91.2%、92.0%、89.9%、88.7%;<br/>
(2) 含性別版 703 人名單的精確率,random_state 0、1、7、42、99 五個切分,分別為 56.2%、54.8%、55.5%、56.3%、55.9%;<br/>
(3) random_state 42 的女性勝算比 2.12。<br/>
三項都與 notebook 04 Cell M4-17 的穩定性表一致。<br/>

2.**移除性別後,名單精確率的下降很小**:<br/>
703 人名單平均下降 0.2 個百分點,178 人名單平均下降 1.0 個百分點(表 B-1、表後第一行)。<br/>
五個切分平均的精確率如下:<br/>
178 人名單,含性別版 90.66%、不含性別版 89.62%、不含性別與婚姻狀態版 88.92%。<br/>
703 人名單,含性別版 55.74%、不含性別版 55.54%、不含性別與婚姻狀態版 55.26%。<br/>
「不含性別版」減「含性別版」的差距:<br/>
178 人名單在 random_state 0、1、7、42、99 分別為 −1.8、−1.1、−0.6、−2.3、+0.6 個百分點,五個切分平均 −1.0。<br/>
703 人名單分別為 −0.1、−0.5、−0.1、−0.4、+0.1 個百分點,五個切分平均 −0.2。<br/>
「不含性別與婚姻狀態版」減「含性別版」的差距:<br/>
178 人名單,五個切分平均 −1.7 個百分點。<br/>
703 人名單,五個切分平均 −0.5 個百分點(表後第二行敘述文字)。<br/>

與「不含性別版」相比,「再移除婚姻狀態」使「精確率再下降 0.7 個百分點(178 人名單)與 0.3 個百分點(703 人名單)」。<br/>

3.**三版模型的平均精確率(AP)幾乎相同**,代表「移除性別或婚姻狀態」**沒有降低模型把會流失的客戶排在前面的能力**,這個結論不限於 178 人與 703 人這兩種名單規模(表 B-2)。AP 的定義見欄位說明表 B-2 (1)。<br/>
五個切分平均的 AP:含性別版 0.76、不含性別版 0.76、不含性別與婚姻狀態版 0.75;五個切分中,三版 AP 的最大差距出現在 random_state 7,含性別版 0.766 對不含性別與婚姻狀態版 0.751,相差 0.015。<br/>

4.**「含性別版模型」中性別的效果穩定且顯著,與 notebook 04 第 7 節的結論一致(表 B-2)。**<br/> 
女性相對男性的流失勝算比,random_state 0、1、7、42、99 分別為 2.38、2.97、2.39、2.12、2.60,五個切分平均 2.49;五個切分的 p 值都小於 0.001(表中 p_female 欄顯示 0.0,是程式四捨五入到小數第 4 位的結果)。<br/> 
notebook 04 第 7 節寫明「五個切分介於 2.1 至 3.0,皆統計顯著」,本表的精確值是 2.12 至 2.97。<br/>

5.**「含性別版模型」確實把更多女性排進名單;移除性別後女性占比下降,但沒有降回規則名單的水準(表 B-2,703 人名單)。**
名單中**女性的占比**,五個切分平均:<br/>
規則名單(命中 ≥ 2)50.96%、含性別版 56.48%、不含性別版 53.86%、不含性別與婚姻狀態版 53.60%。<br/>
全體客戶的女性占比是 52.9%(表 A-1a)。<br/>
含性別版比規則名單高 5.52 個百分點,703 人名單中約多 39 位女性(703 × 5.52%);不含性別版比規則名單高 2.90 個百分點,約多 20 位女性(703 × 2.90%)。<br/>

6.**移除性別只換掉 703 人名單中約 3% 的客戶;再移除婚姻狀態,換掉約 6% 的客戶(表 B-2)。**<br/>
「不含性別版」與「含性別版」的 703 人名單重疊率,五個切分平均 97.1%,也就是 2.9% 的客戶不同,703 × 2.9% ≈ 20 人;<br/>
「不含性別與婚姻狀態版」與「含性別版」的名單重疊率,五個切分平均 93.54%,也就是 6.46% 的客戶不同,703 × 6.46% ≈ 45 人。<br/>

7.**不含性別版的 703 人名單精確率,在五個切分都比規則名單高 7 個百分點以上;178 人名單有一個切分只高 3.3 個百分點(表 B-1、表後第三行)**<br/>
「不含性別版」減「規則名單」的精確率差距,703 人名單在 random_state 0、1、7、42、99 分別為 +7.9、+7.3、+8.9、+9.4、+8.3 個百分點,五個切分平均 +8.4。178 人名單在 random_state 0、1、7、42、99 分別為 +6.1、+8.2、+6.2、+3.3、+7.9 個百分點,五個切分平均 +6.3,其中 random_state 42 的 +3.3 明顯低於其他四個切分。<br/>
(1) 對照 notebook 04 第 7 節算過的「含性別版減規則名單」的精確率差距:178 人名單平均 +7.4、703 人名單平均 +8.6 個百分點。這兩個數字用本表 B-1 的 mean 列也算得出來(178 人名單:含性別版 90.66% − 規則名單 83.28% = 7.38;703 人名單:55.74% − 47.18% = 8.56),因為本節使用的資料、切分與模型設定與 M4 完全相同;這一點再次確認本表可以直接與 notebook 04 對照。<br/>
(2) 另外一個要留到判定 (5) 使用的對照:M4 第 7 節採信「模型 B 明顯勝過規則」的標準是「五個切分的精確率差距都高出 5 個百分點以上」;含性別版的 178 人名單符合這個標準(random_state 0、1、7、42、99 的差距分別為 +7.9、+9.3、+6.8、+5.6、+7.3 個百分點,最小 +5.6),但本節的不含性別版 178 人名單不符合(random_state 42 只有 +3.3);703 人名單則兩版都符合(不含性別版最小 +7.3)。這個差異對「模型 B 升級選項」的意義在判定 (5) 說明。<br/>



**判定**(判定結果與判定依據):<br/>
(1) **移除性別的精確率代價小,維持「不以性別作為模型 B 輸入特徵」的決策**。<br/>
703 人名單的精確率代價(平均下降 0.2 個百分點)落在第 1 節 1.7 判讀規則的區間 (a)(下降小於 1 個百分點);178 人名單的精確率代價(平均下降 1.0 個百分點)恰好落在區間 (a) 與區間 (c)(下降 1 至 3 個百分點)的邊界,本節如實載明「約 1 個百分點」,不寫成「幾乎不需付出精確率代價」。<br/>
依據:不含性別版的精確率比含性別版平均低 0.2 個百分點(703 人名單)與 1.0 個百分點(178 人名單),五個切分的差距分別介於 −0.5 至 +0.1 與 −2.3 至 +0.6 個百分點(表後第一行、實測 2)。依 1.7,區間 (a) 與區間 (c) 的決策相同,差別在於區間 (c) 要求在結論中寫出下降的百分點數;178 人名單的精確率代價在兩區間的分界上,本節採用區間 (c) 的寫法。<br/>

(2) **性別對名單排序的貢獻小:移除性別之後,703 人名單只換掉約 20 位客戶**。<br/>
依據:不含性別版與含性別版的 703 人名單重疊率 97.1%(實測 6);依 1.7 的定義,重疊率越高,代表性別對名單排序的影響越小。<br/>
性別在含性別版模型中是統計顯著的特徵(女性相對男性的流失勝算比在五個切分為 2.12 至 2.97,p 值都小於 0.001,實測 4),但實測名單仍然幾乎不變,代表名單排序主要由其他特徵決定;這與判定 (1) 精確率代價小的結果一致。<br/>

(3) **含性別版模型確實把更多女性排進名單;移除性別之後,模型其他的特徵仍帶有部分性別資訊,這就是 1.3 所說的代理變數問題**。<br/>
依據:703 人名單的女性占比,規則名單 50.96%(不使用性別,是比較的起點)、含性別版 56.48%、不含性別版 53.86%、不含性別與婚姻狀態版 53.60%,全體客戶的女性占比 52.9%(實測 5)。含性別版比起點高 5.52 個百分點(約 39 位女性),這一段來自性別特徵;移除性別後仍比起點高 2.90 個百分點(約 20 位女性),性別已不在模型裡,卻仍多排進女性,代表模型其他的特徵之中有一部分與性別有關聯,模型透過這些特徵間接地把女性排在前面。再移除婚姻狀態之後只再降 0.26 個百分點(53.86% 減 53.60%),代表婚姻狀態只帶有少量性別資訊;剩下的 2.64 個百分點(2.90 − 0.26)來自不含性別與婚姻狀態版其餘的 23 個特徵(Cell M7-7 的特徵清單:12 個連續變數,加上收入 5 個與教育程度 6 個虛擬變數)之中的一個或多個。本模組沒有逐一檢驗這 23 個特徵與性別的關聯,唯一的實際證據來自檢查 A:聯繫 ≥ 5 次的客戶占男性 2.81%、占女性 1.79%(第 4 節判定 (7)),代表 contacts_count_12_mon 與性別有關聯;其餘特徵是否帶有性別資訊,列入待辦。<br/>
本模組不再往下移除這 23 個特徵,理由有三個:行為變數(交易筆數、循環欠款、往來產品數等)是名單圈選的依據本身,移除它們就沒有名單;年齡在模型 B 中的效果不穩定,Cell M7-1「檢視的特徵與名單層級」已說明模型層不另外訓練不含年齡的版本;收入、教育程度、扶養人數在 1.1 已說明不列為檢查對象。這個發現與檢查 A 對照名單的結果一致(第 4 節判定 (7):聯繫 ≥ 5 次的客戶占男性 2.81%、占女性 1.79%,行為欄位本身與性別有關聯)。<br/>

(4) 婚姻狀態仍保留在模型中,不移除:**模型 B 預設採用不含性別版**;「不含性別與婚姻狀態版」留作備用版本。<br/>
依據:本案名單屬於行銷用途,適用台灣法規;台灣的個人資料保護法、金融消費者保護法與金管會的公平待客原則,都沒有把婚姻狀態列為行銷活動不得考量的特徵(Cell M7-1「檢視的特徵與名單層級」),所以本案沒有排除婚姻狀態的法規依據。美國 ECOA 在授信領域明列婚姻狀態為不得考量的特徵,因此若日後名單用途改為授信、或適用的法規把婚姻狀態列為受保護特徵、或業務單位要求排除婚姻狀態,應改用「不含性別與婚姻狀態版」這個備用版本。<br/>
備用版本的精確率代價:不含性別與婚姻狀態版的精確率比不含性別版再低 0.7 個百分點(178 人名單)與 0.3 個百分點(703 人名單)(表後第二行),703 人名單再換掉約 25 位客戶(45 − 20,實測 6);精確率代價已算出,若日後適用的法規或業務要求改變,就可依此代價評估是否採用備用版本。<br/>

(5)**「不含性別版模型 B 」的 703 人名單「精確率」,在五個切分都比規則名單的精確率高 5 個百分點以上**,符合 M4 採信模型 B 的標準,決策日誌 [M5-3]**「模型 B 為升級選項」維持**;178 人名單規模下,「不含性別版模型 B 」有一個切分不符合標準,**若日後 M5 名單規模改為「命中 ≥ 3」,升級模型前須重新評估**。<br/>
依據:M4 第 7 節採信「模型 B 可作為升級選項」的標準是「五個切分中,模型 B 名單的精確率都比規則名單的精確率高 5 個百分點以上」。703 人名單(對應 M5 挽留名單「命中 ≥ 2」的規模):不含性別版的精確率比規則名單高 +7.9、+7.3、+8.9、+9.4、+8.3 個百分點,最小 +7.3,五個切分都符合標準(實測 7)。178 人名單(對應「命中 ≥ 3」的規模):不含性別版的精確率比規則名單高 +6.1、+8.2、+6.2、+3.3、+7.9 個百分點,**random_state 42 的 +3.3 低於 5,不符合標準**(實測 7)。<br/>
這個結果與 M4 第 7 節待辦 (2)「名單規模改變時,須重算模型與規則的精確率差距」一致,所以本節不改變 [M5-3],只加上名單規模的條件:703 人名單規模下 [M5-3] 的升級選項維持;178 人名單規模下若要升級模型 B,須重新評估。<br/>


**對決策的意涵**:<br/>
(1) 性別不作為名單圈選的依據,也不作為模型 B 的輸入特徵;<br/>
模型 B 若日後用於圈選名單,一律採用不含性別版,精確率代價為 178 人名單約 1 個百分點、703 人名單 0.2 個百分點(判定 (1));決策的理由見本節結論依據 2。<br/>
升級為模型 B 圈選名單的前提是「名單規模」:703 人名單規模下, [M5-3]的升級選項維持;178 人名單規模下,須重新評估(判定 (5))。<br/>

(2) 決策日誌 [M5-3] 的切換條件要重算。<br/>
[M5-3] 的切換條件是「每期投放總成本 × 17.4% > 模型維護成本」,其中 17.4% 是「含性別版在 random_state 42 的 703 人名單算出的」(1 − 46.5 ÷ 56.3 = 17.4%)。這個比例的假設是「名單內每一位客戶的投放成本相同、每一位流失客戶被挽留成功的機率相同」,算法是「1 − 規則名單精確率 ÷ 模型名單精確率」(notebook 05 第 6 節)。<br/>
改用「不含性別版」之後,同一個切分(random_state 42,不含性別版 703 人名單精確率 55.9%,表 B-1)重算為 1 − 46.5 ÷ 55.9 = 16.8%(若以五個切分的平均計算,則為 1 − 47.18 ÷ 55.54 = 15.1%)。本案以與 M5 相同的 random_state 42 為準,**切換條件改為「每期投放總成本 × 16.8% > 模型維護成本」**,這部分同時寫在 `docs/07_compliance_fairness.md` 第 3 節與待驗證事項 #39。<br/>

(3) 表 B-1、B-2 與表後三行,就是 1.7 所說的較少歧視替代方案紀錄(候選特徵集、公平性指標、效果指標、採用或否決的理由四項都已具備);日後主管機關或內部審查要求提出「為什麼不以性別為特徵、代價是多少」的佐證時,可直接提出本節,不需要重做分析。<br/>


**待辦或限制**:<br/>
(1) 模型其餘 23 個特徵之中「哪些帶有性別資訊」,本模組沒有檢驗,列為待辦事項。<br/>
性別特徵已移除,不含性別版 703 人名單的女性占比 53.86% 仍高於規則名單的 50.96%(實測 5),代表其餘 23 個特徵之中至少有一個與性別有關聯(模型透過它間接帶入性別資訊);本模組只對 contacts_count_12_mon 一項有實際證據(第 4 節判定 (7):聯繫 ≥ 5 次的客戶占男性 2.81%、占女性 1.79%)。<br/>
說明:若不知道是哪一個特徵帶有性別資訊,日後名單女性占比改變時,就無法判斷是客戶結構改變還是模型的影響。<br/>
待辦的做法:對 23 個特徵各算一次依性別分組的平均值(連續變數)或類別比例(虛擬變數),找出與性別有關聯的特徵,再決定是否需要處理。<br/>
本模組不再往下移除這些特徵,理由見判定 (3);此待辦列入第 10 節。<br/>

(2) 模型移除性別的「精確率代價」,以五個切分的平均為準,不以任何單一切分為準。<br/>
同一版模型的精確率在五個切分之間不同,例如含性別版 703 人名單的精確率在五個切分介於 54.8% 至 56.3%(表 B-1),原因是每一個切分的測試集是不同的一組客戶。不含性別版減含性別版的差距,703 人名單在五個切分介於 −0.5 至 +0.1 個百分點、178 人名單介於 −2.3 至 +0.6 個百分點(實測 2)。<br/>
說明:單一切分的差距同時包含「移除性別的影響」與「測試集客戶不同的影響」,只看一個切分會把後者誤認為前者;五個切分平均能降低測試集不同造成的影響。<br/>
此做法與決策日誌 [M4-2]「模型評估以五個切分為準」的做法相同。<br/>

(3) 檢查 B 只比較三個特徵集,不是 1.7 定義的「完整」較少歧視替代方案搜尋。<br/>
本節比較的是含性別版(27 個特徵)、不含性別版(26 個特徵)、不含性別與婚姻狀態版(23 個特徵)三個版本;完整的搜尋要對許多候選特徵集逐一比較(1.7 本案的設定第 1 點)。<br/>
說明:本案名單屬於行銷用途,依 1.9 的風險基礎方法,三個版本足以回答「性別與婚姻狀態能不能留在模型裡」;若名單日後改用於授信或定價,屬於高風險用途,須依公平授信的完整流程逐一檢視每一個特徵。<br/>
此限制同時寫在第 9 節方法限制。<br/>

(4) 「不含性別與婚姻狀態版模型」是備用版本,只在三種情況下啟用:(a) 名單用途改為授信、(b) 適用的法規把婚姻狀態列為受保護特徵、(c) 業務單位要求排除婚姻狀態。<br/>
「備用版本」的精確率比「不含性別版」再低 0.7 個百分點(178 人名單)與 0.3 個百分點(703 人名單),703 人名單相對「含性別版」共換掉約 45 位客戶,比「不含性別版的 20 位」再多 25 位(實測 6、判定 (4))。<br/>
說明:以上三種情況任一發生時,模型 B 改用備用版本,並依意涵 (2) 的算法,以備用版本的精確率重算 [M5-3] 的成本節省比例。<br/>
備用版本的精確率代價已算出,啟用此備用模型時不需要重做分析。<br/>
啟用條件列入第 10 節。<br/>

Cell M7-9(markdown)|第 6 節:不採用公平性套件的取捨

## 第 6 節|公平性套件:它們是什麼、實務上用在哪裡、本案為什麼不採用

**本節結論**:<br/>
本模組的所有計算都以 pandas 與 statsmodels 自行完成,沒有使用專門的公平性套件(fairness toolkit,專門用來計算公平性指標、並調整模型以滿足某一種公平性定義的程式庫)。<br/>
不採用的理由有三個:<br/>
(1) 本案需要的指標只有四個,自行計算幾行程式就能完成,而且每一步都透明可核對;<br/>
(2) 本案不做「偏見緩解(bias mitigation,指主動調整資料、模型或分數門檻,以縮小族群之間的指標差距;以下簡稱緩解)」;<br/>
(3) 引入套件對計算結果沒有影響,卻多一個相依(Dependency)套件要維護。<br/>

6.5 寫明在什麼情況下應該改用套件。<br/>
以下依序說明套件的用途、實務價值、優缺點、本案不採用的理由、改用套件的條件,以及本模組的方法在其他情境怎麼套用。<br/>

### 6.1 常見的三個套件與它們做的事

| 套件 | 維護者 | 做什麼 | 特點 |
|---|---|---|---|
| Fairlearn | 由微軟發起,目前由開源社群維護 | 給定模型的預測值、真實結果與族群欄位,一次算出各族群的選取率、偽陽性率、偽陰性率等指標,以及族群之間的差距(`MetricFrame` 類別);另外提供「緩解」的方法:訓練時加入公平性約束(例如 `ExponentiatedGradient`),或訓練後對各族群設定不同的分數門檻(例如 `ThresholdOptimizer`) | 介面與 scikit-learn 一致,最容易接進既有的建模流程 |
| AIF360(AI Fairness 360) | IBM | 指標最多(七十餘種),涵蓋資料前處理(重新加權樣本)、訓練中(加入約束)、後處理(調整門檻)三類緩解方法 | 功能最完整,但名詞多、學習成本較高;第 2 節引用的從業者研究指出,它的名詞對非工程人員不直觀 |
| Aequitas | 芝加哥大學資料科學與公共政策中心 | 偏重「審計報告」:輸入模型的預測結果與族群欄位,產出各族群的指標表,以及每一個指標是否通過門檻的報告(預設門檻就是五分之四法則) | 設計給政策與審計人員使用,只做檢查、不做緩解 |

**三類緩解方法的定義**:<br/>
(1) 前處理(pre-processing)是「在訓練模型之前調整資料」,例如:提高少數族群樣本在訓練時的權重,目的是讓模型在訓練時更重視這個族群的樣本,縮小模型在該族群的錯誤率與其他族群的差距。<br/>
(2) 訓練中(in-processing)是「在模型的目標函數裡加入一條約束」,例如:『兩個族群的漏報率差距不得超過 5 個百分點』(差距指兩族群同一個指標的相減),目的是讓模型在求最佳係數的同時,就把族群差距控制在設定的上限內。<br/>
(3) 後處理(post-processing)是「在模型訓練完成之後,對不同族群使用不同的分數門檻」,把各族群的選取率或偽陰性率拉到相同,目的是不重新訓練模型,直接達成選定的公平性定義。<br/>

三類方法都要先選定一種公平性定義(1.6),因為方法是依據該定義而調整模型。<br/>


### 6.2 實務上誰在用、價值在哪裡

套件在三種情境的價值:<br/>
(1) 要檢查的指標多、族群多,例如「同時檢視性別 × 年齡 × 地區的交叉族群」、十幾個指標,手寫計算容易出錯,套件能一次算完並使用同一套定義;<br/>
(2) 需要「緩解」,也就是:不只檢查、還要主動調整模型讓某一個公平性指標的族群差距落在機構自訂的上限內(例如 AIR 不低於 0.8、或兩族群漏報率差距不超過幾個百分點),而自己實作緩解演算法的成本高,使用套件較符合成本效益;<br/>
(3) 需要對外提出審計報告:套件產出的指標表欄位固定、每一個指標的定義寫在套件的公開文件裡,外部審計人員或主管機關可以直接依文件核對每一個數字的算法,不必逐一確認機構自行設計的表格。<br/>

使用這些套件的典型單位是大型金融機構的模型風險管理團隊、設有負責任 AI(responsible AI,企業內負責 AI 倫理與風險管理的職能)團隊的科技公司,以及接受監理審查時需要附上標準報告的機構。第 2 節引用的從業者研究(Deng 等人,2022)顯示,一般資料團隊的使用率不高:受訪者所屬的組織只有 16% 有明確的公平性流程,是否做公平性測試幾乎完全取決於利害關係人有沒有要求。<br/>


### 6.3 套件的優點與缺點

優點:<br/>
指標定義一致、計算快、附視覺化;其中 Fairlearn 的學習成本不高(AIF360 的學習成本較高,見 6.1)。<br/>
缺點有三個:<br/>
(1) 套件只回答「指標是多少」,不回答「該選哪一個指標、差距多大算有問題、所以該怎麼決策」,而這三件事才是法遵與公平性檢視報告、以及對業務單位說明時的重點。<br/>
(2) 多數緩解方法要求把問題寫成二元分類(本案的流失預測本來就是二元分類,這一點對本案不構成障礙),而且後處理類的方法(對不同族群設定不同門檻)本身就是依受保護特徵差別對待,在許多法規情境(包含授信,以及本案的挽留資源分配)不被允許或不被建議這麼做。<br/>
(3) 名詞不直觀,對非工程人員的解釋成本高。<br/>


### 6.4 本案不採用的三個理由

(1) 本案需要的指標只有四個(選取率、不利影響比率、精確率、漏報率)加上一個「差距的信賴區間」,每一個都是 M2 至 M4 已經寫過的計算,用少量 pandas 程式即可完成(Cell M7-5 的三個函式),而且算法完全透明:審核者(例如:法遵或風險控管人員)不需要理解套件的內部定義,看 Cell M7-5 函式的說明文字就能重算核對。<br/>

(2) 本案不做緩解:本案的名單屬於行銷資源分配用途,依其風險等級不需要主動調整模型;而且 1.6 已說明,在各族群基準率不同時,後處理式的「拉平選取率」等於依性別設定不同的圈選門檻,本身就是 1.2 的差別待遇,而且會提高「選取率被降低的那一個族群」流失客戶的漏報率(1.6 以本案女性 17.36%、男性 14.62% 的流失率說明)。本案的處置是「不以性別為特徵、名單仍以行為旗標圈選、名單的族群組成(各族群的選取率、精確率、漏報率)列為每季監測項,因為客戶結構改變時這些數字會跟著改變」,這些處置都不需要套件就能做到。<br/>

(3) 引入套件只是把幾行 pandas 換成一個函式呼叫,對計算結果沒有影響,對業務單位的理解與說明沒有直接幫助,卻多一個相依套件要維護;決策日誌 [M2-12] 的原則是「引入套件必須有需要使用此套件的原因」,本案沒有這個原因。<br/>


### 6.5 什麼情況下應該改用套件

若日後出現下列任一種情況,應改用 Fairlearn 套件(因為其介面與 scikit-learn 一致,而本案的資料切分與評估已經使用 scikit-learn(模型本身用 statsmodels 訓練,預測值可以直接交給 Fairlearn 計算指標),切換成本最低):<br/>
(1) 指標數量或族群交叉的數量明顯增加。<br/>
(2) 名單用途改為授信或定價,需要依公平授信流程做「完整的」較少歧視替代方案搜尋,並產出標準化報告。<br/>
(3) 業務單位決定要做緩解(例如訓練中加入約束),而不只是檢查。<br/>

屆時若用套件對同一批名單重算,套件算出的各族群選取率、精確率、漏報率應與本模組的表 A-2 至 A-7、表 B-1、B-2 相同,因為兩者的指標定義相同;若數字不同,代表套件使用的母體或指標定義與本模組不同,應先查明差異再採用套件的結果。<br/>


### 6.6 本模組的方法套用到其他情境

以下五個問題是公平性檢視報告交付給法遵、風控與業務單位時常被提出的問題,各以本案的數字回答,說明本模組的方法如何套用到其他情境:<br/>

1.「為什麼不直接把性別刪掉就好」:<br/>
把性別刪掉只處理了差別待遇,沒有處理差別性影響;其他變數可能是性別的代理變數(1.3),所以本案在移除性別之後,仍然檢查名單的實際性別組成(檢查 A,以及表 B-2 的 shareF 欄:不含性別版 703 人名單的女性占比 53.86%,仍高於規則名單的 50.96%,第 5 節判定 (3)),並量化移除性別的精確率代價(檢查 B:178 人名單 −1.0 個百分點、703 人名單 −0.2 個百分點,第 5 節判定 (1))。<br/>

2.「人口比例平等與機會平等,本案選哪一個、為什麼」:<br/>
本案選機會平等(各族群漏報率相近),因為挽留資源的目的是找到會流失的客戶;在基準率不同時(女性流失率 17.36% 對男性 14.62%)強求各族群選取率相同,會系統性地漏掉高風險族群的流失客戶(1.6)。<br/>

3.「如果移除性別之後精確率下降幅度很大怎麼辦」:<br/>
結論仍然是「不以性別為特徵」(1.7 判讀規則 (b)):如實寫出「精確率下降的百分點數,以及對應名單內少了幾位真的會流失的客戶」,並向決策者說明這個精確率損失是用「降低法遵風險」與「名單可向客戶、業務單位、主管機關說明」換來的;精確率的損失是此決策需付出的代價,但不是否決這個決策的理由。<br/>
百分點數與人數的換算如下:<br/>
178 人名單每下降 1 個百分點約少 2 位,下降 3 個百分點約少 5 位(換算說明:在 178 人名單裡,名單內每少 1 位真的會流失的客戶,精確率就下降 1 ÷ 178 ≈ 0.6 個百分點;反過來說,精確率每下降 0.6 個百分點,就是名單內少了 1 位真的會流失的客戶);<br/>
703 人名單每下降 1 個百分點約少 7 位,下降 3 個百分點約少 21 位(1.7 的換算)。<br/>


4.「名單檢查通過,就代表公平嗎」:<br/>
不代表。<br/>
檢查只覆蓋資料集裡有記錄的特徵與本案選定的公平性定義;但行為變數可能承載歷史差異(對照名單中男性 2.81% 對女性 1.79% 就是此論述的證據);族群組成會隨客戶結構改變。所以本案把「性別組與年齡組在三個名單層級的選取率、AIR、精確率、漏報率」列為每季監測項(第 4 節意涵 (4)、第 10 節 #37),並把「行為變數可能承載既有差異」寫進第 9 節方法限制。Google Fairness Indicators 的說明文件對此工具也是相同說法:「各族群的指標相近,不等於這個模型或名單就是公平的,指標本身不能達成公平」(來源見第 2 節)。<br/>

5.「這套檢查換到授信模型還適用嗎」:<br/>
「檢查的方法」仍適用(選取率、依族群切片計算的精確率與漏報率、精確率代價的量化),但嚴格程度要升級:授信屬於高風險用途,須做「完整的」較少歧視替代方案搜尋、逐一檢視每一個特徵的代理關係(代理變數問題)、保留每一個候選方案的紀錄,並考慮獨立第三方審查(1.9 的風險基礎方法)。<br/>

Cell M7-10(markdown)|第 7 至 10 節:M7 結論、法遵對應摘要、方法限制、待驗證事項清單 v8、M8 開工檢核

## 第 7 節|M7 結論:法遵與公平性檢視總表

**本模組結論**:<br/>
本模組回答 Cell M7-1 提出的問題「在考量法規與公平性的前提下,能不能直接使用 M5 設計的挽留名單與 M4 建立的風險模型」。答案有三點:<br/>
(1) 三個名單層級(挽留名單、人工聯繫名單、加碼回饋名單)在性別與年齡上都不構成差別性影響,現行名單規則(使用四個行為旗標與價值層圈選)可以照原樣使用,不需要修改(第 4 節結論)。<br/>
(2) 性別不作為名單圈選的依據,也不作為模型 B 的輸入特徵;模型 B 若日後依決策日誌 [M5-3] 的切換條件用於圈選名單,一律採用「不含性別版的模型」(第 5 節結論);精確率代價為 178 人名單下降約 1 個百分點、703 人名單下降 0.2 個百分點(第 5 節判定 (1));[M5-3] 的升級選項在 703 人名單規模下維持,切換條件中的成本節省比例由 17.4% 改為 16.8%;178 人名單規模下若要升級,須重新評估(第 5 節判定 (5)、意涵 (2))。<br/>
(3) 名單上線後,性別組與年齡組在三個名單層級的選取率、AIR、精確率、漏報率列為每季監測項(第 4 節意涵 (4))。<br/>

下表把「本模組的每一項定案」整理成一列,第三欄指出完整說明的位置。<br/>

| 項目 | 定案 | 完整說明的位置 |
|---|---|---|
| 風險定位 | 本案的挽留名單屬於行銷資源分配用途,不是授信;檢視項目定為「名單的族群組成檢查(檢查 A)」與「從模型 B 移除性別特徵的精確率代價量化(檢查 B)」兩項;若名單用途改變(例如改用於授信或定價),須重新評估檢視項目 | Cell M7-1「風險定位」、第 1 節 1.9 |
| 檢視對象 | 性別(名單層與模型層都檢視)、年齡(只在名單層檢視)、婚姻狀態(只在模型層檢視);三個名單層級(挽留名單 2,476 人、人工聯繫 426 人、加碼回饋 308 人)加一個對照名單(聯繫 ≥ 5 次人工回訪 230 人) | Cell M7-1「檢視的特徵與名單層級」、Cell M7-4 |
| 差別待遇 | 規則式名單沒有使用性別與年齡,沒有差別待遇;模型 B 的含性別版不用於圈選名單;若符合升級條件要改用模型 B 圈選名單,應採用「不含性別版模型 B」 | 第 4 節判定 (1)、第 5 節意涵 (1) |
| 差別性影響(檢查 A) | 三個名單層級在各自的候選範圍內,各族群真的會流失的客戶被名單覆蓋的比例相近(每一個族群相對基準族群的漏報率差距,95% 信賴區間都包含 0),各族群選取率的差異由族群之間實際流失率的差異解釋,不構成差別性影響;<br/>人工聯繫名單中女性的 AIR 0.787 低於提醒線,原因是高價值層的女性較少命中旗標、而且高價值層內女性的流失率(5.4%)低於男性(14.2%);<br/>對照名單顯示行為變數與性別有關聯(聯繫 ≥ 5 次的客戶占全體男性的 2.81%、占全體女性的 1.79%) | 第 4 節判讀 |
| 模型移除性別的代價(檢查 B) | 178 人名單的精確率下降 1.0 個百分點、703 人名單的精確率下降 0.2 個百分點(五個切分平均);不含性別版與含性別版的 703 人名單重疊 97.1%;703 人名單的女性占比:含性別版 56.48%、不含性別版 53.86%、規則名單 50.96% | 第 5 節判讀 |
| 對名單規則的決定 | 性別不作為名單圈選的依據,也不作為模型 B 的輸入特徵;名單仍以四個行為旗標與價值層圈選,現行規則照原樣使用;模型 B 若日後用於圈選名單,採用「不含性別版」,決策日誌 [M5-3] 的切換比例由 17.4% 改為 16.8%;「不含性別與婚姻狀態版」留作備用版本,在「名單用途改為授信、適用的法規把婚姻狀態列為受保護特徵、業務單位要求排除婚姻狀態」三種情況下啟用 | 第 4 節意涵 (1)、第 5 節判定 (4)、意涵 (1)、(2) |
| 上線後的監測 | 名單每季重算時,同時重算性別組與年齡組在三個名單層級的選取率、AIR、精確率、漏報率;「低成本區內女性流失客戶的加碼回饋覆蓋率(覆蓋率 = 1 − 漏報率)」(下限接近 0,是三個名單層級中最接近規則 (c) 門檻的一格,所以要優先確認資料累積後是否越過門檻)列為優先重算項;<br/>「56–73 歲組在加碼回饋名單內的精確率」(本次 44.64%,比基準族群低 30.77 個百分點、信賴區間不包含 0,數字落在規則 (b)「誤圈較多」的範圍;但名單內只有 56 人,依 [M2-10] 不下結論;待人數累積到 100 人以上後判讀)列為優先重算項(第 4 節判定 (6)、意涵 (4));<br/>人工聯繫與加碼回饋名單內的 26–35 歲組與 56–73 歲組四格(45、43、16、56 人)不足 100 人,待人數(樣本數)累積到 100 人以上後再判讀 | 第 4 節意涵 (4)、判定 (5)、(6)、待辦 (3) |
| 不採用公平性套件 | 本案需要的指標少、自行計算較透明、無做「緩解」的需求;需改用套件的條件寫明在 6.5 | 第 6 節 |


**五個結論**:<br/>
1.**名單規則可以照原樣使用**。<br/>
(1) 在各動作的候選範圍內,每一個族群真的會流失的客戶被名單漏掉的比例(漏報率)與基準族群相近(漏報率差距的 95% 信賴區間都包含 0,符合 1.5 判讀規則 (a));<br/>
(2) 各族群選取率的差異,來自「同樣命中旗標的客戶中,各族群實際流失的比例不同」,也就是規則在某些族群身上誤圈較多,這是成本問題,不是哪一個族群被少給挽留資源(第 4 節結論)。<br/>

2.**性別不作為模型的特徵之一,代價小**。<br/>
(1) 從模型移除性別的精確率代價小,不改變「不以性別作為模型 B 輸入特徵」的決策(第 5 節判定 (1));<br/>
(2) 性別對名單排序的貢獻小:移除性別之後,703 人名單只換掉約 20 位客戶(第 5 節判定 (2));**這個決策的理由不是「代價小」**,而是 1.7 事先寫定的判讀規則(第 5 節結論依據 2)。<br/>

3.**公平性判準要事先選定並說明理由**。<br/>
本案以機會平等(各族群漏報率相近)為主要判準,「各族群的選取率差距」與「各族群的 AIR」則一併報告並解釋(第 1 節 1.6 本案的設定);<br/>
在基準率(真實流失率:女性 17.36% 對男性 14.62%)不同的情形下,不做差別待遇就無法同時滿足兩種判準(若在名單人數固定的前提下強求各族群選取率相同,會系統性地漏掉基準率較高族群的流失客戶)(1.6、第 4 節待辦 (4))。<br/>

4.**「規則沒有使用性別」不等於「名單與性別無關」**。<br/>
對照名單證明行為欄位(聯繫 ≥ 5 次)本身與性別、年齡有關聯(男性 2.81% 對女性 1.79%;56–73 歲組 3.23% 對 46–55 歲組 2.01%),代理變數問題確實存在(第 4 節結論、判定 (7));<br/>
含性別版模型確實把更多女性排進名單;移除性別之後,模型其他的特徵仍帶有部分性別資訊,這就是 1.3 所說的代理變數問題(第 5 節判定 (3));本模組沒有檢驗「哪些特徵帶有性別資訊」,列為待辦事項(第 5 節待辦 (1)、第 10 節 #40)。<br/>

5.**檢查的每一步都有業界實例對應,但「檢查通過不等於公平」**。<br/>
本模組的每一個檢查步驟都有監理機關或大型公司的公開做法作為依據(第 2 節);但**各族群的指標相近,不等於這個模型或名單就是公平的**(第 2 節 Google Fairness Indicators 列);<br/>
檢查 A 通過,不代表資料本身沒有歷史偏差(第 4 節待辦 (2));<br/>
檢查結果只對這一批 10,127 位客戶的資料成立,名單上線後須每季重算(客戶結構改變時,各族群的選取率與漏報率可能改變)(第 4 節意涵 (4))。<br/>


## 第 8 節|法遵對應摘要(完整版見 `docs/07_compliance_fairness.md`)

下表把第 2 節依案例整理的法規與指引,改依四個法遵主題重排(個資與目的拘束、公平性檢視、可解釋性、挽留手段的邊界),每一列寫明「本案的對應作法」與「對應的法規或指引」;只寫條號(或指引的原則名稱)與核心要求,不逐條解釋條文;完整版在 `docs/07_compliance_fairness.md`。<br/>

| 法遵主題 | 本案的對應作法 | 對應的法規或指引 |
|---|---|---|
| 個資與目的拘束(目的拘束,purpose limitation,指個人資料只能在蒐集時所定的特定目的範圍內利用) | 既有客戶的挽留行銷屬於信用卡業務蒐集目的範圍內的利用;推播必須提供拒絕機制;<br/>M6 對照組的「不得接觸」名單與分組名單檔案必須有存取權限控制;<br/>交付給業務單位的名單不含 is_churned 欄位與其他不必要的欄位(資料最小化,data minimisation) | 個人資料保護法第 19 條(蒐集須有特定目的)、第 20 條第 1 項(在蒐集目的範圍內利用)、第 20 條第 2 項與第 3 項(當事人的行銷拒絕權,以及首次行銷須提供免費的拒絕方式);金管會 AI 指引「保護隱私及客戶權益」原則、資料最小化 |
| 公平性檢視 | 檢查 A:三個名單層級在性別與年齡上都不構成差別性影響(第 4 節結論);<br/>檢查 B:性別不作為模型 B 的輸入特徵,採用不含性別版(第 5 節結論);<br/>性別組與年齡組的選取率、AIR、精確率、漏報率列為每季監測項(第 4 節意涵 (4)) | 金管會 AI 指引「重視公平性及以人為本的價值觀」原則(宜評估公平性、注意偏見之產生、儘可能避免歧視);美國公平授信的差別待遇 / 差別性影響架構與較少歧視替代方案,作為方法上的參考 |
| 可解釋性 | M4 選用邏輯斯迴歸,而不是樹模型(tree model,例如決策樹)與集成模型(ensemble model,例如隨機森林、梯度提升)(決策日誌 [M4-0]:邏輯斯迴歸的每個係數都能翻譯成「某屬性每增加一單位,流失勝算變幾倍」,樹模型與集成模型沒有係數,做不到這種說明);<br/>名單以規則式矩陣圈選,每一位客戶進名單的原因都可以用四個旗標說明(決策日誌 [M5-0]、[M5-1]);模型 B 的預測流失機率數字只用來排序,不對客戶宣稱為「流失機率」(決策日誌 [M4-2] 的限制) | 金管會 AI 指引「落實透明性與可解釋性」原則(應能清楚說明 AI 系統如何運作及其決策邏輯);NYDFS 報告指出,發卡機構無法向消費者說明決策依據是信任問題的主因 |
| 挽留手段的邊界 | 差別給予優惠的可接受條件:依客戶的行為而不是身分給予優惠、規則可以向客戶說明、未得到優惠的客戶不會因此失去原有的服務或被加價、加碼回饋只對實際達到消費門檻的客戶支付;<br/>零動用旗標對應的動作以「建立持續往來」為目的,不引導客戶動用循環信用(決策日誌 [M5-2]) | 金管會「金融服務業公平待客原則」(商品或服務適合度原則、告知與揭露原則);英國 FCA 禁止保險業忠誠度懲罰,作為「監理機關會檢視挽留定價公平性」的案例 |

本案使用的是公開資料集,沒有真實客戶的同意紀錄與蒐集目的文件。上表「本案的對應作法」這一欄分兩類:<br/>
(1) 在本案資料集內能完成的事(檢查 A 與檢查 B、模型與名單的選法、差別給予優惠的條件)已經做完;<br/>
(2) 屬於真實業務作業的事(客戶同意紀錄、推播的拒絕機制、名單檔案的存取權限控制、交付欄位的限制)是「若為真實業務,名單上線前須確認的事項」,不是已完成的法遵審查(第 2 節個資法列第四欄)。<br/>


## 第 9 節|方法限制

1. **只檢視資料集裡有記錄的特徵**:<br/>
本資料集有記錄的受保護特徵只有性別、年齡、婚姻狀態三欄,種族、居住地、身心障礙等其他國家法規列為受保護特徵的屬性,本資料集沒有欄位,無法檢查;<br/>
婚姻狀態只在模型層(檢查 B)處理,名單層沒有檢查婚姻狀態的族群組成(第 4 節待辦 (1);理由見 Cell M7-1「檢視的特徵與名單層級」)。<br/>

2. **檢查通過,不代表資料本身沒有歷史偏差**:<br/>
四個行為旗標本身可能承載既有的性別或年齡差異(NYDFS 報告所說的 legacy bias,1.3),對照名單的結果就是此論述的證據:聯繫 ≥ 5 次的客戶占全體男性的 2.81%、占全體女性的 1.79%(第 4 節判定 (7)、待辦 (2));<br/>
模型移除性別之後,模型其他的特徵仍帶有部分性別資訊,本模組沒有檢驗是哪些特徵帶有性別資訊(第 5 節判定 (3)、待辦 (1));<br/>
各族群的指標相近,不等於這個模型或名單就是公平的(第 2 節 Google Fairness Indicators 列)。<br/>

3. **公平性判準是一種選擇**:<br/>
本檢查的判準是機會平等,若審核單位改用人口比例平等,結論會不同;<br/>
在各族群基準率(真實流失率)不同時,不做差別待遇就無法同時滿足兩種判準(1.6、第 4 節待辦 (4))。<br/>

4. **門檻都不是法規數字**:<br/>
AIR 的 0.8 只作為「需要在判讀中解釋」的提醒線,不作為法遵門檻(1.4);<br/>
精確率差距 1 個百分點的判準是本模組依 178 人名單中每一位客戶約 0.6 個百分點的換算自訂(1.7);<br/>
小樣本慣例 100 人沿用決策日誌 [M2-10];<br/>
178 人名單的精確率代價 1.0 個百分點恰好落在 1.7 區間 (a) 與 (c) 的邊界,本模組如實載明,採用區間 (c) 的寫法(第 5 節判定 (1))。<br/>

5. **結果只對這一批資料成立**:<br/>
本模組的判定只對這一批 10,127 位客戶的資料成立;客戶結構改變時,各族群的選取率與漏報率可能改變,名單上線後須每季重算(第 4 節意涵 (4))。<br/>

6. **檢查 B 不是「完整的」較少歧視替代方案搜尋**:<br/>
只比較含性別版(27 個特徵)、不含性別版(26 個)、不含性別與婚姻狀態版(23 個)三個特徵集;<br/>
名單若改用於授信或定價,屬於高風險用途,須依公平授信的完整流程逐一檢視每一個特徵,並考慮由獨立第三方審查(第 5 節待辦 (3)、1.9)。<br/>

7. **小樣本格子只描述數字、不下結論**:<br/>
人工聯繫名單內的 26–35 歲組(45 人)與 56–73 歲組(43 人)、加碼回饋名單內的 26–35 歲組(16 人)與 56–73 歲組(56 人),「精確率差距與漏報率差距」只「描述數字並列為監測項」(第 4 節待辦 (3)、第 10 節 #37)。<br/>

8. **法遵對應只寫法規條號與核心要求,不是正式法律意見**:<br/>
本案使用公開資料集,沒有真實客戶的同意紀錄與蒐集目的文件;法遵對應是「若為真實業務,名單上線前須確認的事項」,須由法遵單位依當時有效的法規與內部規範審查(第 2 節個資法列、第 8 節表後說明)。<br/>


## 第 10 節|待驗證事項清單 v8(M7 收尾定版)

**定位**:<br/>
承接 notebook 06 Cell M6-14 的清單 v7,回填本模組承接的 #27、#29,新增本模組產生的 #37 至 #41。<br/>
編號沿用,不重編;v7 其餘項目(本模組未處理的項目)原文照抄。<br/>

**本模組結案的項目**:<br/>

| # | 事項 | 結案結果 | 依據 |
|---|---|---|---|
| 27 | M7 公平性檢視:模型 B 中女性的勝算比 2.12,以性別為特徵的模型會系統性地把女性列為較高風險;<br/>M7 應比較「含性別」與「不含性別」兩版模型的精確率,將「移除性別的代價」量化,並從法遵角度說明是否可以用「性別」作為圈選名單的依據 | ✅ 完成:三個版本的模型 B(含性別、不含性別、不含性別與婚姻狀態)在五個切分、178 人與 703 人兩種名單規模下比較精確率;移除性別的精確率代價為 178 人名單 −1.0 個百分點、703 人名單 −0.2 個百分點,不含性別版與含性別版的名單重疊率 97.1%;703 人名單的女性占比:含性別版 56.48%、不含性別版 53.86%、規則名單 50.96%。<br/>決定:性別不作為名單圈選的依據,也不作為模型 B 的輸入特徵;模型 B 若用於圈選名單,應採用「不含性別版模型」;決策日誌 [M5-3] 的切換比例由 17.4% 改為 16.8%。<br/>另記錄:178 人名單規模下,不含性別版相對規則名單的精確率差距在 random_state 42 只有 +3.3 個百分點,不符合 M4「五個切分都高 5 個百分點以上」的標準;若日後名單規模改為「命中 ≥ 3」,升級模型前須重新評估(第 5 節判定 (5)) | notebook 07 第 5 節;決策日誌 [M7-1] |
| 29 | M7 檢視四個策略區的性別與年齡組成;名單在不同性別或年齡組的比例若明顯不均,須說明是否構成差別待遇 | ✅ 完成:三個名單層級依性別、年齡組計算選取率與 AIR;<br/>挽留名單中女性的 AIR 0.906、46–55 歲組的 AIR 0.815;每一個族群相對基準族群的漏報率差距,信賴區間都包含 0,各族群選取率的差異由實際流失率的差異解釋,**不構成差別性影響**。<br/>人工聯繫名單中女性的 AIR 0.787,以高價值層(人工聯繫的候選範圍)為母體重算漏報率後,兩性流失客戶被名單覆蓋的比例看不出差異(女性漏報率 50.00%、男性 45.27%,差距的信賴區間 −7.02 至 16.47 包含 0),原因是高價值層內的女性較少命中旗標、而且流失率較低(5.4% 對 14.2%)(第 4 節判定 (4))。<br/>加碼回饋名單中 56–73 歲組的精確率偏低(44.64%,名單內 56 人)與低成本區內女性流失客戶的加碼回饋覆蓋率(覆蓋率 = 1 − 漏報率;兩性漏報率差距的信賴區間下限接近 0)列為優先監測項;<br/>族群組成(各族群的選取率、精確率、漏報率)列為每季監測項(因為客戶結構改變時這些數字會跟著改變) | notebook 07 第 4 節;決策日誌 [M7-2] |

**交辦後續模組的項目**<br/>
**v7 既有項目維持原文,本模組新增 #37 至 #41**<br/>
(v7 項目中出現的名詞與縮寫,例如 MDE(minimum detectable effect,實驗設計上能偵測到的最小效果)、H6(M2 的第六個假設)、C2(M2 階段 C 的第二項分析)、mart(資料倉儲中供報表使用的資料層)、_SIM(模擬數字的欄位後綴),完整定義見各該項目「依據」欄所列的 notebook,本表依清單慣例原文照抄、不改寫):<br/>

| # | 事項 | 為什麼要做 | 交辦階段 | 依據 |
|---|---|---|---|---|
| 37 | 名單上線後每季重算性別組與年齡組在三個名單層級的選取率、AIR、精確率、漏報率(人工聯繫以高價值層、加碼回饋以低成本區為母體);<br/>任一名單層級中,某一個族群相對基準族群的漏報率差距的 95% 信賴區間不包含 0 且在 0 以上時(第 1 節 1.5 判讀規則 (c)),重新檢視規則;<br/>某一個族群的 AIR 低於 0.8 時,須在監測報告中依 1.5 判讀規則解釋原因;<br/>加碼回饋名單中 56–73 歲組的精確率(本次 44.64%,名單內 56 人)累積到 100 人以上後重新判讀;<br/>低成本區內女性流失客戶的加碼回饋覆蓋率(覆蓋率 = 1 − 漏報率;本次兩性漏報率差距 +4.74 個百分點,信賴區間 −0.58 至 10.06)為優先重算項 | 族群組成(各族群的選取率、精確率、漏報率)的數字會隨客戶結構改變而變動,一次檢查通過不代表永遠通過;與 notebook 05 第 8 節監測區客戶的每季重算同時執行 | 名單上線後(本專案範圍外;M8 README 載明) | notebook 07 第 4 節意涵 (4)、判定 (5)、(6) |
| 38 | M8 README 的 Compliance & Fairness Considerations 一節由 `docs/07_compliance_fairness.md` 摘要、第 3 節、第 5 節精簡而成(500 至 800 字),不另外重寫;引用時須保留「本資料集無真實客戶同意紀錄,法遵對應為上線前待確認事項」的限制 | 避免 README 把第 8 節的法遵對應(哪一條法規對應本案哪一項作法)寫成已完成的法遵審查 | M8 | notebook 07 第 8 節 |
| 39 | M8 簡報與 README 引用模型 B 時,一律以「不含性別版」的數字為準(178 人與 703 人名單的精確率 89.62% / 55.54%,五個切分平均;切換比例 16.8%),並註明「含性別版模型僅作對照」;M8 的 Limitations & Future Work 記錄「挽留名單內的男性、26–35 歲組與 56–73 歲組,人工聯繫名單內的女性,加碼回饋名單內的男性,這五格的精確率明顯低於基準族群(表示「誤圈較多」),是日後修改名單規則時應優先檢視的對象,不是公平性問題(理由見 notebook 07 第 1 節 1.5 判讀規則 (b):誤圈較多是成本問題,這些族群得到的是較多而不是較少的挽留資源) | 避免簡報引用「含性別版模型」的數字,與 M7 的決定矛盾;精確率明顯較低的五格交由 M8 記錄為日後修改名單規則時應優先檢視的對象 | M8 | notebook 07 第 4 節意涵 (3)、第 5 節意涵 (2) |
| 40 | 「模型 B 不含性別版」其餘的特徵之中,本模組沒有檢驗哪些帶有性別資訊;檢驗做法:對「不含性別與婚姻狀態版的 23 個特徵」各算一次「依性別分組的平均值(連續變數)或類別比例(虛擬變數)」,找出與性別有關聯的特徵,再決定是否需要處理 | 性別從模型移除後,不含性別版 703 人名單的女性占比 53.86% 仍高於規則名單的 50.96%,代表 23 個特徵之中至少有一個與性別有關聯;本模組只對 contacts_count_12_mon 一項有實際證據(聯繫 ≥ 5 次的客戶占男性 2.81%、占女性 1.79%);若不知道是哪一個特徵與性別有關聯(模型透過它間接帶入性別資訊),日後名單女性占比改變時無法判斷是客戶結構改變還是模型的影響 | 模型 B 依 [M5-3] 升級為名單圈選依據之前(本專案範圍外;M8 README 載明) | notebook 07 第 5 節判定 (3)、待辦 (1) |
| 41 | 「不含性別與婚姻狀態版」為模型 B 的備用版本,在下列任一情況啟用:(a) 名單用途改為授信、(b) 適用的法規把婚姻狀態列為受保護特徵、(c) 業務單位要求排除婚姻狀態;啟用時 [M5-3] 的成本節省比例依該版精確率重算(算法同 notebook 07 第 5 節意涵 (2)) | 台灣的個人資料保護法、金融消費者保護法與金管會公平待客原則沒有把婚姻狀態列為行銷活動不得考量的特徵,美國 ECOA 在授信領域則明列;備用版本的精確率代價已算出(比「不含性別版」的精確率再低 0.7 個百分點(178 人名單)與 0.3 個百分點(703 人名單),意同 703 人名單再換掉約 25 位客戶),啟用備用版本時不需要重做分析 | 以上三種情況任一發生時(本專案範圍外;M8 README 載明) | notebook 07 第 5 節判定 (4)、待辦 (4) |
| 33 | 實驗開始前以當期資料(仍為有效客戶者)重算基準率與樣本數;sql/05 [E2] 的 WHERE 條件加上「目前仍為有效客戶」;517 人在歷史基準率下已不足(每組 258 人),但實際需要每組 390 人,重算基準率後,樣本數不足的情形會擴大,補救方式為分批累積新產生的命中 3+ 客戶至每組 390 人,不放寬 MDE、不擴大母體 | 歷史流失率 82.4% 包含已流失客戶,實驗期間的基準率必然較低,相同成功率對應下,降幅較小,因此所需樣本數(人數)增加 | 實驗執行前(本專案範圍外;M8 計畫書載明) | notebook 06 第 1 節 1.6、第 4 節 4.1 與待辦 1、第 7 節風險 3 |
| 34 | 真實實驗結束後,以兩組的實際流失率算出實測成功率 s = (p_c − p_t) ÷ p_c,代回 notebook 05 第 6 節公式 1 與公式 3,使 M5 的三個情境收斂為一組數字。將 s 代回 notebook 05 第 6 節公式 1 與公式 3 後,加上「業務單位提供的每人投放成本」,M5 第 6 節的七個方案能從倍數換算成金額,達成待驗證事項 #28 要求的「使成本效益試算收斂為一組實測數字」;第一個實驗測出成功率 s 後,即可以 s 回推「對搶救區客戶做實驗所需的人數」 | M5 的成本效益試算只能到倍數,成功率是唯一無法從歷史資料估計的參數;搶救區實驗的樣本數計算需要「效果大小(第一個實驗的實測成功率)」作為輸入 | 實驗執行後(本專案範圍外) | notebook 06 第 6 節、notebook 05 第 6 節意涵 (3) |
| 35 | 第二個實驗(調高信用額度)的完整設計需要三項輸入:授信單位提供的調高額度資格規則與幅度、第一個實驗的實測成功率、風險控管單位的核准 | 效果大小無依據、母體資格缺乏對應欄位,因此目前只能設計草案,無法實際執行實驗 | 實驗執行後(本專案範圍外) | notebook 06 第 7 節 7.2、[M6-4] |
| 36 | M8 簡報引用 M6 時須明示「本模組為實驗設計,未執行、無結果」;所有帶 _SIM 的數字都是模擬,不得引用為真實實驗的效果;M5 的動作仍為「假設性方案」 | 避免把模擬結果或設計誤述為真實實驗的驗證結果 | M8 | notebook 06 第 1 節 1.10、第 9 節限制 1 |
| 11(M6 部分) | 「調高信用額度」不得直接列為挽留手段;若要採用,須經實驗驗證;狀態為「待第二個實驗驗證,已設計實驗草案,尚未執行」(草案內容:假設、候選母體、主指標、不可與第一個實驗合併的理由、缺少的輸入) | 本資料集缺乏「調高信用額度的效果大小」與「母體資格」的依據,若要完成完整實驗設計,需要第一個實驗執行後得到的實測成功率,以及授信單位提供的調高額度資格規則(待驗證事項 #35);不結案,因結案需真實實驗結果 | 實驗執行後(本專案範圍外) | notebook 06 第 7 節 7.2、[M6-4] |
| 15(M8 部分) | M8 簡報陳述 H6 時須明示「風險標記,非提前預警」,不得寫成因果 | 快照資料無法區分聯繫與流失決定的先後 | M8 | notebook 03 T1 判讀、[M2-5] |
| 24 | M8 引用 M2 C2 的交互作用結論時,須改寫為「只有淺關係與低活躍存在交互作用;零動用與高聯繫則是各自獨立的主效果」,並註明 C2 的百分點尺度與模型的勝算尺度不同 | 兩種尺度結論不同,避免簡報誤述: C2「三訊號效果皆集中於低活躍子群」的敘述僅在百分點尺度成立,M8 引用須註明尺度。因為在勝算尺度下:只有淺關係成立,零動用無交互作用,高聯繫則方向相反 | M8 | notebook 04 第 5 節判讀 |
| 30 | M8 儀表板的矩陣資料:第 3 節的 12 格(人數、流失率、流失人數、平均交易金額(平均年消費))與第 8 節名單總表為儀表板的資料來源,與 #13 的 mart 匯出一併處理 | 儀表板設計定案後一次匯出 | M8 | notebook 05 第 3、8 節、[M2-13] |
| 31 | k-means 對照的結論(演算法分群區分力 2.7% 至 42.1%,矩陣分群區分力 0% 至 88%)為 M8 敘事「為什麼用規則式分群」的素材;k-means 分群結果不進名單 | 記錄在此,以免 M8 誤將 k-means 分的群當作名單 | M8 | notebook 05 第 7 節判讀 |
| 32 | README 資料說明區載明 total_amt_chng_q4_q1 的定義為「依欄位名稱推斷為 Q4 ÷ Q1,未經驗證」;M8 引用第 5 節「第四季消費不到第一季的一半」等描述時須加註 | 欄位定義風險,決策日誌 [M2-4] 執行紀錄二 | M8 | notebook 05 第 5 節待辦 (1) |
| 13 | mart 層匯出:依 M8 儀表板設計定案後,一次匯出 data/processed/ 的 CSV | 下游需求(儀表板要呈現什麼)未定前匯出無驗收標準;只是延後,不是取消 | M8 | [M2-13] |
| 14 | Excel 樞紐分析重現:以樞紐表重現一張儀表板選用的切片,驗證與 SQL 數字一致,產出 churn_pivot_summary.xlsx,README 技術棧列入 Excel | 展示跨工具重現同一分析並核對數字一致的能力;與 #13 一併完成 | M8 | [M2-14] |

**清單演變紀錄**:<br/>
v7(M6 收尾)→ v8(本表:#27、#29 結案;新增 #37 至 #41)。<br/>

**M8 開工檢核**(進入 M8 之前先核對):<br/>
M8 承接的 M7 事項為 #37(README 載明監測項)、#38(Compliance & Fairness Considerations 一節由 docs/07_compliance_fairness.md 精簡而成)、#39(模型 B 的數字以不含性別版為準,並記錄精確率明顯較低的五格)、#40(模型 B 升級前的「特徵性別關聯檢驗」,README 載明)、#41(備用版本的啟用條件,README 載明);其餘承接項目為 #13、#14、#15、#24、#30、#31、#32、#36。<br/>
M8 開工時,將本表複製到 M8 交付文件的開頭作為承接依據。<br/>

**本模組的重現步驟**:<br/>
(1) 資料庫就緒:依 README 的 Load into PostgreSQL 三個步驟載入資料,再執行 `python -m src.cleaning` 建立 clean.customers。<br/>
(2) 以 Restart & Run All 執行本 notebook。<br/>
(3) 核對 Cell M7-4 的輸出:四個策略區的人數應為 426 / 2,050 / 2,949 / 4,702,三個名單的人數應為 2,476 / 426 / 308,對照名單 230 人。<br/>
(4) 核對 Cell M7-7 表 B-1 中 seed 為 42 那一列的含性別版數字,應與 notebook 04 第 7 節相同:178 人名單的精確率 89.9%、703 人名單的精確率 56.3%,女性勝算比 2.12。<br/>
(5) 本模組不另開 SQL 檔,Cell M7-4 以 print(M7_SQL) 印出查詢全文供核對;若要以純 SQL 重現檢查 A 的名單組成,對查詢中的 m6_base 以 gender(或年齡組)與 zone 分組計數即可(Cell M7-4 註解「客戶層級表的查詢說明 (3)」)。<br/>